# Human or AI? Text Authorship Classification

**Updated route:** format-robust sparse Logistic Regression + frozen GPT-2 statistics.

**Fast continuation:** run Sections **8–10** in order; Section 8 can load the data and reuse the saved 8,000/2,000 split independently. Sections 1–7 retain the earlier course experiments and outputs and need not be rerun to use the new route.

For Colab, select **Runtime → Change runtime type → T4 GPU** (or another available GPU) before Section 8. Data remain in `MyDrive/human_or_ai/data`; new artifacts go to a separate fingerprinted output folder. Local CPU execution is supported, but the measured time limit may stop feature extraction.

**Measured public scores (user-reported):** no-dropout v2 = **0.92319**; feature dropout = **0.92340**. The new language-model experiment has **no measured public score yet**. Neither 0.98 nor a 20-minute total runtime is guaranteed.


## 本地 Jupyter / Google Colab 使用说明

1. 从顶部按顺序运行。在首个代码单元设置 `PROJECT_OVERRIDE` 为项目目录；留空时，本地从当前工作目录向上查找 `data/` 或 `Data/`，Colab 默认使用 `/content/drive/MyDrive/human_or_ai`。Colab 如使用临时上传文件，把 `USE_GOOGLE_DRIVE=False`，将数据放到 `/content/data/`。
2. 项目目录下的 `data/`（或 `Data/`）应包含五个比赛 CSV。输出统一写入该项目的 `outputs/`。切换电脑或 Colab 时，请一并复制原有 `outputs/`，保持子目录结构。
3. 缺少的基础依赖会安装到当前内核。可关闭 `AUTO_INSTALL_DEPENDENCIES` 后自行安装。可选模型步骤需要额外依赖及首次模型下载的网络连接。
4. 自动使用 NVIDIA CUDA，否则使用 CPU（Mac 也可使用 CPU）。大型模型的 CPU 特征提取可能很慢；原有耗时预算保护仍保留，需要长时间计算时请调整对应实验的允许长时间提取开关。
5. 第 8 节及后续的最佳提交恢复依赖历史实验文件，并非仅凭五个 CSV 就能重新生成已确认的 0.98816 结果。缺失时需复制原始项目的 `outputs/`；修改运行环境不会补造历史结果。实验开关仍默认关闭。

本版本清除了旧运行输出，避免将旧环境结果误认为本次运行结果。模型方法、数据划分与评分校验保持原有逻辑。


## 1. Environment Setup and Configuration


### 1.1 Install and Import Dependencies


In [ ]:
# Shared setup for local Jupyter and Google Colab.
import os
import sys
import subprocess
import importlib.util
from pathlib import Path

# Set to your project folder (containing data/ or Data/ and outputs/), or leave None.
PROJECT_OVERRIDE = os.environ.get("HUMAN_AI_PROJECT_DIR") or None
USE_GOOGLE_DRIVE = True  # Colab only; False uses uploaded files under /content.
AUTO_INSTALL_DEPENDENCIES = True

def ensure_packages(packages):
    missing = [package for module, package in packages.items()
               if importlib.util.find_spec(module) is None]
    if missing:
        if not AUTO_INSTALL_DEPENDENCIES:
            raise ImportError("Install in this notebook's Python environment: " + " ".join(missing))
        subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])

ensure_packages({"numpy":"numpy", "pandas":"pandas>=1.4", "matplotlib":"matplotlib>=3.4",
                 "sklearn":"scikit-learn", "scipy":"scipy", "joblib":"joblib",
                 "threadpoolctl":"threadpoolctl", "IPython":"ipython", "jinja2":"jinja2"})
try:
    import google.colab
except ImportError:
    IN_COLAB = False
else:
    IN_COLAB = True

def configure_paths():
    global PROJECT_DIR, DATA_DIR, OUTPUT_DIR
    if PROJECT_OVERRIDE:
        root = Path(PROJECT_OVERRIDE).expanduser().resolve()
    elif IN_COLAB and USE_GOOGLE_DRIVE:
        root = Path("/content/drive/MyDrive/human_or_ai")
    else:
        root = next((p for p in [Path.cwd(), *Path.cwd().parents]
                     if any((p / d / "train.csv").is_file() for d in ["data", "Data"])), Path.cwd())
    if IN_COLAB and str(root).startswith("/content/drive/"):
        from google.colab import drive
        if not Path("/content/drive/MyDrive").is_dir():
            drive.mount("/content/drive")
    PROJECT_DIR = root
    DATA_DIR = next((root / d for d in ["data", "Data"]
                     if (root / d / "train.csv").is_file()), root / "data")
    OUTPUT_DIR = root / "outputs"
    required = ["train.csv", "test.csv", "unlabeled.csv", "train_starter_2000.csv", "sample_submission.csv"]
    missing = [name for name in required if not (DATA_DIR / name).is_file()]
    if missing:
        raise FileNotFoundError(f"Data folder: {DATA_DIR}\nMissing: {', '.join(missing)}. "
                                "Set PROJECT_OVERRIDE above to your project folder; place CSVs in data/ or Data/.")
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    print("Environment:", "Colab" if IN_COLAB else "Local Jupyter")
    print("Project:", PROJECT_DIR, "\nData:", DATA_DIR, "\nOutputs:", OUTPUT_DIR)

def synchronize_device(device):
    import torch
    if torch.device(device).type == "cuda":
        torch.cuda.synchronize(device)

def clear_device_cache():
    import torch
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

import sys
import json
import random
import zipfile
from io import BytesIO
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score

print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("Scikit-learn:", sklearn.__version__)
print("Dependencies imported successfully.")

### 1.2 Configure Data and Output Paths


In [ ]:
configure_paths()
REQUIRED_FILES = ["train.csv", "test.csv", "unlabeled.csv", "train_starter_2000.csv", "sample_submission.csv"]
TRAIN_PATH = DATA_DIR / "train.csv"
TEST_PATH = DATA_DIR / "test.csv"
UNLABELED_PATH = DATA_DIR / "unlabeled.csv"
STARTER_PATH = DATA_DIR / "train_starter_2000.csv"
SAMPLE_SUBMISSION_PATH = DATA_DIR / "sample_submission.csv"


### 1.3 Set Random Seeds and Experiment Parameters


In [ ]:
import sys
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn

# Ensure the output folder exists on both Colab and local machines.
OUTPUT_DIR = Path(OUTPUT_DIR)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Reproducible experiment settings.
SEED = 42
VALIDATION_SIZE = 2000
TRAINING_SIZES = [500, 1000, 2000, 5000, 8000]
DECISION_THRESHOLD = 0.5

random.seed(SEED)
np.random.seed(SEED)

# Also pass random_state=SEED to splits and estimators that support it.
CONFIG = {
    "seed": SEED,
    "validation_size": VALIDATION_SIZE,
    "training_sizes": TRAINING_SIZES,
    "decision_threshold": DECISION_THRESHOLD,
    "metric": "roc_auc",
    "label_mapping": {"human": 0, "ai": 1},
    "data_dir": str(Path(DATA_DIR).resolve()),
    "output_dir": str(OUTPUT_DIR.resolve()),
}

VERSIONS = {
    "python": sys.version.split()[0],
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "scikit_learn": sklearn.__version__,
}

CONFIG_PATH = OUTPUT_DIR / "config.json"
ENVIRONMENT_PATH = OUTPUT_DIR / "environment.json"

CONFIG_PATH.write_text(
    json.dumps(CONFIG, indent=2),
    encoding="utf-8",
)
ENVIRONMENT_PATH.write_text(
    json.dumps(VERSIONS, indent=2),
    encoding="utf-8",
)

print("Experiment configuration:")
print(json.dumps(CONFIG, indent=2))

print("\nPackage versions:")
print(json.dumps(VERSIONS, indent=2))

print("\nSaved files:")
print(CONFIG_PATH.resolve())
print(ENVIRONMENT_PATH.resolve())

## 2. Data Loading and Exploration


### 2.1 Load the Datasets


In [ ]:
import pandas as pd
from IPython.display import display

# Read each CSV once and keep the dataframes for later sections.
datasets = {
    filename: pd.read_csv(DATA_DIR / filename)
    for filename in REQUIRED_FILES
}

train_df = datasets["train.csv"]
test_df = datasets["test.csv"]
unlabeled_df = datasets["unlabeled.csv"]
starter_df = datasets["train_starter_2000.csv"]
sample_submission_df = datasets["sample_submission.csv"]

dataset_summary = pd.DataFrame([
    {
        "Dataset": filename,
        "Rows": df.shape[0],
        "Columns": df.shape[1],
        "Headers": ", ".join(df.columns),
        "File size (MB)": (DATA_DIR / filename).stat().st_size / 1_000_000,
    }
    for filename, df in datasets.items()
])

display(dataset_summary.style.format({
    "Rows": "{:,}",
    "Columns": "{:d}",
    "File size (MB)": "{:.3f}",
}).hide(axis="index"))

print("Row counts exclude the header. 1 MB = 1,000,000 bytes.")
print("train_starter_2000.csv is a subset of train.csv; do not combine them as extra training data.")


### 2.2 Inspect Data Structure and Class Distribution


In [ ]:
# Inspect the original columns without modifying any source dataframe.
expected_columns = {
    "train.csv": ["id", "text", "label"],
    "test.csv": ["id", "text"],
    "unlabeled.csv": ["id", "text"],
    "train_starter_2000.csv": ["id", "text", "label"],
    "sample_submission.csv": ["id", "label"],
}
for filename, expected in expected_columns.items():
    assert datasets[filename].columns.tolist() == expected, filename

schema_summary = pd.DataFrame([
    {"Dataset": filename, "Column": column,
     "Data type": str(df[column].dtype), "Non-null rows": int(df[column].notna().sum())}
    for filename, df in datasets.items() for column in df.columns
])
display(schema_summary)

LABEL_NAMES = {0: "Human", 1: "AI"}
class_rows = []
for filename in ["train.csv", "train_starter_2000.csv"]:
    labels = datasets[filename]["label"]
    assert labels.notna().all() and labels.isin([0, 1]).all(), filename
    for label, count in labels.value_counts().sort_index().items():
        class_rows.append({"Dataset": filename, "Label": label,
                           "Class": LABEL_NAMES[label], "Rows": count,
                           "Share (%)": 100 * count / len(labels)})
class_distribution = pd.DataFrame(class_rows)
display(class_distribution)

fig, ax = plt.subplots(figsize=(6, 3.5))
counts = train_df["label"].value_counts().reindex([0, 1], fill_value=0)
bars = ax.bar(["Human (0)", "AI (1)"], counts, color=["#4070A0", "#DB8B39"])
ax.bar_label(bars, labels=[f"{n:,}" for n in counts], padding=3)
ax.set(title="Training class distribution", ylabel="Number of texts", ylim=(0, counts.max() * 1.15))
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "class_distribution.png", dpi=160, bbox_inches="tight")
plt.show()

schema_summary.to_csv(OUTPUT_DIR / "dataset_schema.csv", index=False)
class_distribution.to_csv(OUTPUT_DIR / "class_distribution.csv", index=False)
print("Human = 0; AI = 1. Test and unlabeled data have no ground-truth labels.")
print("The 0.5 values in sample_submission.csv are placeholders, not class labels.")


### 2.3 Check Missing Values and Duplicate Texts


In [ ]:
# Normalization is only for duplicate checks; original text stays unchanged.
def normalize_for_audit(text):
    return text.astype("string").str.strip().str.replace(r"\s+", " ", regex=True).str.lower()

missing_summary = pd.DataFrame([
    {"Dataset": name, "Column": column, "Missing": int(df[column].isna().sum())}
    for name, df in datasets.items() for column in df.columns
])
audit_rows = []
for name, df in datasets.items():
    row = {"Dataset": name, "Missing cells": int(df.isna().sum().sum()),
           "Duplicate rows": int(df.duplicated().sum()),
           "Duplicate IDs": int(df["id"].duplicated().sum()),
           "Blank texts": pd.NA, "Duplicate texts": pd.NA,
           "Normalized duplicate texts": pd.NA}
    if "text" in df:
        text = df["text"].dropna()
        row.update({"Blank texts": int(text.str.strip().eq("").sum()),
                    "Duplicate texts": int(text.duplicated().sum()),
                    "Normalized duplicate texts": int(normalize_for_audit(text).duplicated().sum())})
    audit_rows.append(row)
quality_summary = pd.DataFrame(audit_rows)
display(missing_summary)
display(quality_summary.fillna("N/A"))
print("Duplicate counts are extra occurrences after the first; N/A means there is no text column.")

pairs = [("train.csv", "test.csv"), ("train.csv", "unlabeled.csv"),
         ("test.csv", "unlabeled.csv"), ("train.csv", "train_starter_2000.csv")]
overlap_rows = []
for left, right in pairs:
    a, b = datasets[left], datasets[right]
    overlap_rows.append({"Left dataset": left, "Right dataset": right,
        "Shared IDs": len(set(a["id"]) & set(b["id"])),
        "Shared normalized texts": len(set(normalize_for_audit(a["text"]).dropna()) &
                                       set(normalize_for_audit(b["text"]).dropna()))})
overlap_summary = pd.DataFrame(overlap_rows)
display(overlap_summary)

assert quality_summary["Missing cells"].eq(0).all(), "Review missing data before modeling."
assert quality_summary["Duplicate IDs"].eq(0).all(), "Review duplicate IDs before splitting."
assert quality_summary["Blank texts"].dropna().eq(0).all(), "Review blank text before modeling."
assert sample_submission_df["id"].equals(test_df["id"]), "Submission IDs/order differ from test."
starter_match = starter_df.merge(train_df, on=["id", "text", "label"], validate="one_to_one")
assert len(starter_match) == len(starter_df), "Starter data is not an exact training subset."

missing_summary.to_csv(OUTPUT_DIR / "missing_values.csv", index=False)
quality_summary.to_csv(OUTPUT_DIR / "data_quality.csv", index=False)
overlap_summary.to_csv(OUTPUT_DIR / "dataset_overlap.csv", index=False)
print("Data integrity checks passed. No rows were removed or changed.")
print("Starter overlap is expected: all 2,000 starter rows already belong to train.csv.")
print("This audit detects exact and case/whitespace-normalized matches, not semantic near-duplicates.")


### 2.4 Explore Text Length Distributions


In [ ]:
# Words = whitespace-separated tokens; characters include spaces/punctuation.
# Keep these descriptive features separate from the original model inputs.
length_frames = []
for name, df in datasets.items():
    if "text" not in df:
        continue
    lengths = pd.DataFrame({"Dataset": name, "id": df["id"],
        "words": df["text"].str.count(r"\S+"),
        "characters": df["text"].str.len()})
    lengths["Class"] = df["label"].map(LABEL_NAMES) if "label" in df else "Unknown"
    length_frames.append(lengths)
length_data = pd.concat(length_frames, ignore_index=True)

length_summary = length_data.groupby("Dataset", sort=False).agg(
    Rows=("id", "size"), Mean_words=("words", "mean"),
    Median_words=("words", "median"), Min_words=("words", "min"),
    P90_words=("words", lambda x: x.quantile(0.90)),
    P95_words=("words", lambda x: x.quantile(0.95)), Max_words=("words", "max"),
    Mean_characters=("characters", "mean"), Median_characters=("characters", "median"),
    Min_characters=("characters", "min"), Max_characters=("characters", "max")
).reset_index()
display(length_summary.round(1))
train_lengths = length_data[length_data["Dataset"].eq("train.csv")]
class_length_summary = train_lengths.groupby("Class").agg(
    Rows=("id", "size"), Mean_words=("words", "mean"), Median_words=("words", "median"),
    Mean_characters=("characters", "mean"), Median_characters=("characters", "median")
).reset_index()
display(class_length_summary.round(1))

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for feature, ax in zip(["words", "characters"], axes[:2]):
    bins = np.histogram_bin_edges(train_lengths[feature], bins=35)
    for label, color in [("Human", "#4070A0"), ("AI", "#DB8B39")]:
        values = train_lengths.loc[train_lengths["Class"].eq(label), feature]
        ax.hist(values, bins=bins, density=True, alpha=0.5, label=label, color=color)
    ax.set(title=f"Training text length: {feature}", xlabel=feature.capitalize(), ylabel="Density")
    ax.legend()
for name, color in [("train.csv", "#4070A0"), ("test.csv", "#DB8B39"), ("unlabeled.csv", "#469A74")]:
    values = np.sort(length_data.loc[length_data["Dataset"].eq(name), "words"].to_numpy())
    axes[2].step(values, np.arange(1, len(values) + 1) / len(values),
                 where="post", label=name, color=color)
axes[2].set(title="Word counts across datasets", xlabel="Words", ylabel="Cumulative proportion")
axes[2].legend()
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "text_length_distributions.png", dpi=160, bbox_inches="tight")
plt.show()
length_summary.to_csv(OUTPUT_DIR / "text_length_summary.csv", index=False)
class_length_summary.to_csv(OUTPUT_DIR / "class_length_summary.csv", index=False)
print("sample_submission.csv has no text, so it is excluded from length analysis.")
print("The starter subset is reported in the table but excluded from the cross-dataset plot.")
print("These are descriptive summaries. Similar lengths alone do not establish similar writing styles.")
print("Any learned preprocessing or later length-bin cutoffs must use only the training split.")


### 2.5 Create a Fixed Stratified Training–Validation Split


In [ ]:
import hashlib

# Freeze a single stratified split for every later experiment.
assert len(train_df) == 10000 and VALIDATION_SIZE == 2000
assert train_df["id"].is_unique and train_df["id"].notna().all()
SPLIT_PATH = OUTPUT_DIR / "split_assignments.csv"
SPLIT_META_PATH = OUTPUT_DIR / "split_metadata.json"
expected_metadata = {
    "seed": SEED, "validation_size": VALIDATION_SIZE,
    "train_csv_sha256": hashlib.sha256(TRAIN_PATH.read_bytes()).hexdigest(),
}
assert SPLIT_PATH.exists() == SPLIT_META_PATH.exists(), "Incomplete saved split; review output files."
reuse_split = SPLIT_PATH.exists()
if reuse_split:
    split_metadata = json.loads(SPLIT_META_PATH.read_text())
    assert all(split_metadata.get(k) == v for k, v in expected_metadata.items()), (
        "Data or split settings changed. Keep the existing split intact and review the mismatch."
    )
    split_assignments = pd.read_csv(SPLIT_PATH, dtype={"id": train_df["id"].dtype})
else:
    train_ids, val_ids = train_test_split(
        train_df["id"].to_numpy(), test_size=VALIDATION_SIZE,
        stratify=train_df["label"].to_numpy(), random_state=SEED,
    )
    split_assignments = pd.concat([
        pd.DataFrame({"id": train_ids, "split": "train"}),
        pd.DataFrame({"id": val_ids, "split": "validation"}),
    ], ignore_index=True)

assert split_assignments.columns.tolist() == ["id", "split"]
assert split_assignments["id"].is_unique and len(split_assignments) == len(train_df)
assert set(split_assignments["id"]) == set(train_df["id"])
assert set(split_assignments["split"]) == {"train", "validation"}
train_ids = split_assignments.loc[split_assignments["split"].eq("train"), "id"].to_numpy()
val_ids = split_assignments.loc[split_assignments["split"].eq("validation"), "id"].to_numpy()
indexed_train = train_df.set_index("id", drop=False)
train_split = indexed_train.loc[train_ids].reset_index(drop=True).copy()
val_split = indexed_train.loc[val_ids].reset_index(drop=True).copy()

assert len(train_split) == 8000 and len(val_split) == 2000
assert not (set(train_ids) & set(val_ids))
assert train_split["label"].value_counts().to_dict() == {0: 4000, 1: 4000}
assert val_split["label"].value_counts().to_dict() == {0: 1000, 1: 1000}
shared_texts = set(normalize_for_audit(train_split["text"])) & set(normalize_for_audit(val_split["text"]))
assert not shared_texts, "Duplicate text crosses the split; review before training."
assert max(TRAINING_SIZES) <= len(train_split)

if not reuse_split:
    split_assignments.to_csv(SPLIT_PATH, index=False)
    SPLIT_META_PATH.write_text(json.dumps({**expected_metadata,
        "sklearn_version": sklearn.__version__, "method": "stratified train_test_split"}, indent=2))

X_train, y_train = train_split["text"].copy(), train_split["label"].copy()
X_val, y_val = val_split["text"].copy(), val_split["label"].copy()
split_summary = pd.DataFrame([
    {"Split": name, "Rows": len(df), "Human (0)": int(df["label"].eq(0).sum()),
     "AI (1)": int(df["label"].eq(1).sum()), "AI share (%)": 100 * df["label"].mean()}
    for name, df in [("Train", train_split), ("Validation", val_split)]
])
display(split_summary)
split_summary.to_csv(OUTPUT_DIR / "split_summary.csv", index=False)
print("Reused the saved split." if reuse_split else "Created and saved the fixed split.")
print("ID overlap: 0; normalized-text overlap: 0.")
print("Split assignments:", SPLIT_PATH)
print("Fit vectorizers and other learned preprocessing on X_train only.")
print("Keep X_val/y_val fixed for all experiments; do not train or pseudo-label with validation rows.")
print("Test/unlabeled rows and the starter subset were not added to this split.")


## 3. Baseline Model


### 3.1 Text Preprocessing


In [ ]:
from time import perf_counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    roc_auc_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_curve,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

# Validate the fixed split before training.
for name, texts, labels in [
    ("Training", X_train, y_train),
    ("Validation", X_val, y_val),
]:
    assert len(texts) == len(labels), f"{name}: length mismatch."
    assert texts.notna().all(), f"{name}: missing text."
    assert texts.map(lambda value: isinstance(value, str)).all()
    assert texts.str.strip().ne("").all(), f"{name}: blank text."
    assert labels.isin([0, 1]).all(), f"{name}: invalid labels."

assert set(train_split["id"]).isdisjoint(set(val_split["id"]))

# Keep the original text. TF-IDF handles lowercasing and tokenization;
# this baseline does not remove stop words or apply stemming.
print(f"Training samples: {len(X_train):,}")
print(f"Validation samples: {len(X_val):,}")
print("Text validation completed; original texts preserved.")

### 3.2 TF-IDF Feature Extraction


In [ ]:
TFIDF_CONFIG = {
    "lowercase": True,
    "strip_accents": None,
    "stop_words": None,
    "ngram_range": (1, 2),     # Use individual words and two-word phrases.
    "min_df": 2,              # Ignore terms appearing in only one training text.
    "max_df": 1.0,
    "max_features": 50_000,   # Limit vocabulary size to control memory usage.
    "sublinear_tf": True,     # Reduce the influence of repeated terms.
    "norm": "l2",
}

baseline_vectorizer = TfidfVectorizer(
    **TFIDF_CONFIG,
    dtype=np.float32,
)

# Learn the vocabulary and IDF weights from training data only.
start = perf_counter()
X_train_tfidf = baseline_vectorizer.fit_transform(X_train)
tfidf_fit_seconds = perf_counter() - start

# Reuse the fitted vectorizer to avoid validation-data leakage.
start = perf_counter()
X_val_tfidf = baseline_vectorizer.transform(X_val)
tfidf_val_transform_seconds = perf_counter() - start

feature_summary = pd.DataFrame([
    {
        "Split": name,
        "Rows": matrix.shape[0],
        "Features": matrix.shape[1],
        "Nonzero values": matrix.nnz,
        "Density (%)": (
            100 * matrix.nnz / (matrix.shape[0] * matrix.shape[1])
        ),
    }
    for name, matrix in [
        ("Train", X_train_tfidf),
        ("Validation", X_val_tfidf),
    ]
])

display(feature_summary.round(4))
print(f"Training TF-IDF fit/transform: {tfidf_fit_seconds:.2f} seconds")
print(f"Validation transform: {tfidf_val_transform_seconds:.2f} seconds")

### 3.3 Train a Logistic Regression Baseline


In [ ]:
LR_CONFIG = {
    "C": 1.0,                 # Inverse regularization strength.
    "solver": "liblinear",
    "max_iter": 1000,
    "random_state": SEED,
}

baseline_classifier = LogisticRegression(**LR_CONFIG)

start = perf_counter()
baseline_classifier.fit(X_train_tfidf, y_train)
classifier_fit_seconds = perf_counter() - start

# Include feature extraction when reporting total training time.
baseline_fit_seconds = (
    tfidf_fit_seconds + classifier_fit_seconds
)

# Package the already-fitted components for prediction from raw text.
# Constructing this pipeline does not fit either component again.
baseline_model = Pipeline([
    ("tfidf", baseline_vectorizer),
    ("classifier", baseline_classifier),
])

print("Baseline training completed.")
print(f"Classifier fit time: {classifier_fit_seconds:.2f} seconds")
print(f"Total training time including TF-IDF: {baseline_fit_seconds:.2f} seconds")
print("Classes:", baseline_model.classes_)
print("Solver iterations:", baseline_classifier.n_iter_)

if baseline_classifier.n_iter_.max() >= LR_CONFIG["max_iter"]:
    print(
        "Warning: iteration limit reached; "
        "check convergence before comparing results."
    )

### 3.4 Define Reusable Evaluation Functions


In [ ]:
def evaluate_binary_classifier(
    model,
    texts,
    labels,
    threshold=0.5,
):
    """Evaluate a fitted binary text classifier using AI probabilities."""
    if not 0 <= threshold <= 1:
        raise ValueError("threshold must be between 0 and 1.")

    labels = np.asarray(labels)
    classes = np.asarray(model.classes_)

    if len(classes) != 2 or set(classes.tolist()) != {0, 1}:
        raise ValueError("Expected binary classes 0 (human) and 1 (AI).")

    if set(np.unique(labels).tolist()) != {0, 1}:
        raise ValueError("Evaluation labels must contain both classes.")

    # Locate the AI class explicitly instead of assuming column order.
    ai_column = int(np.flatnonzero(classes == 1)[0])

    # Pipeline prediction time includes text vectorization.
    start = perf_counter()
    probabilities = model.predict_proba(texts)[:, ai_column]
    prediction_seconds = perf_counter() - start

    if len(probabilities) != len(labels):
        raise ValueError("Prediction and label lengths differ.")

    if (
        not np.isfinite(probabilities).all()
        or ((probabilities < 0) | (probabilities > 1)).any()
    ):
        raise ValueError("Invalid predicted probabilities.")

    predictions = (probabilities >= threshold).astype(int)

    # AUROC uses continuous probabilities; other metrics use the threshold.
    metrics = {
        "AUROC": roc_auc_score(labels, probabilities),
        "Accuracy": accuracy_score(labels, predictions),
        "Precision": precision_score(
            labels, predictions, zero_division=0
        ),
        "Recall": recall_score(
            labels, predictions, zero_division=0
        ),
        "F1": f1_score(
            labels, predictions, zero_division=0
        ),
        "Prediction seconds": prediction_seconds,
    }

    return metrics, probabilities, predictions


print("Reusable evaluation function is ready.")

### 3.5 Evaluate and Record Baseline Performance


In [ ]:
import json
import joblib

# Evaluate the same fitted model on training and held-out validation data.
train_metrics, train_prob_ai, train_predictions = (
    evaluate_binary_classifier(
        baseline_model,
        X_train,
        y_train,
        threshold=DECISION_THRESHOLD,
    )
)

val_metrics, val_prob_ai, val_predictions = (
    evaluate_binary_classifier(
        baseline_model,
        X_val,
        y_val,
        threshold=DECISION_THRESHOLD,
    )
)

baseline_results = pd.DataFrame([
    {
        "Model": "TF-IDF + Logistic Regression",
        "Split": "Train",
        **train_metrics,
    },
    {
        "Model": "TF-IDF + Logistic Regression",
        "Split": "Validation",
        **val_metrics,
    },
])

display(baseline_results.round(4))

# A large positive gap may indicate overfitting.
auc_gap = train_metrics["AUROC"] - val_metrics["AUROC"]
print(f"Training AUROC:   {train_metrics['AUROC']:.4f}")
print(f"Validation AUROC: {val_metrics['AUROC']:.4f}")
print(f"Train minus validation AUROC: {auc_gap:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

# Compare ranking performance across all classification thresholds.
for name, labels, probabilities, metrics in [
    ("Train", y_train, train_prob_ai, train_metrics),
    ("Validation", y_val, val_prob_ai, val_metrics),
]:
    fpr, tpr, _ = roc_curve(labels, probabilities)
    axes[0].plot(
        fpr,
        tpr,
        label=f"{name} (AUROC = {metrics['AUROC']:.4f})",
    )

axes[0].plot([0, 1], [0, 1], "--", color="gray")
axes[0].set(
    title="Baseline ROC curves",
    xlabel="False positive rate",
    ylabel="True positive rate",
)
axes[0].legend(loc="lower right")
axes[0].grid(alpha=0.2)

# Show classification errors at the fixed decision threshold.
cm = confusion_matrix(y_val, val_predictions, labels=[0, 1])
ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Human (0)", "AI (1)"],
).plot(
    ax=axes[1],
    cmap="Blues",
    colorbar=False,
    values_format="d",
)
axes[1].set_title(
    f"Validation confusion matrix (threshold = {DECISION_THRESHOLD})"
)

fig.tight_layout()

BASELINE_DIR = OUTPUT_DIR / "baseline"
BASELINE_DIR.mkdir(parents=True, exist_ok=True)

fig.savefig(
    BASELINE_DIR / "evaluation.png",
    dpi=160,
    bbox_inches="tight",
)
plt.show()

# Save metrics and feature statistics for later model comparisons.
baseline_results.to_csv(
    BASELINE_DIR / "metrics.csv",
    index=False,
)
feature_summary.to_csv(
    BASELINE_DIR / "feature_summary.csv",
    index=False,
)

# Preserve IDs so predictions can be joined with texts for error analysis.
validation_predictions = pd.DataFrame({
    "id": val_split["id"].to_numpy(),
    "label": np.asarray(y_val),
    "prob_ai": val_prob_ai,
    "pred_label": val_predictions,
})
validation_predictions.to_csv(
    BASELINE_DIR / "validation_predictions.csv",
    index=False,
)

# Save the complete fitted pipeline, including its vocabulary and IDF weights.
joblib.dump(
    baseline_model,
    BASELINE_DIR / "model.joblib",
)

# Record settings, timings, and split provenance for reproducibility.
baseline_metadata = {
    "model": "TF-IDF + Logistic Regression",
    "seed": SEED,
    "threshold": DECISION_THRESHOLD,
    "training_rows": len(X_train),
    "validation_rows": len(X_val),
    "features": X_train_tfidf.shape[1],
    "tfidf_config": TFIDF_CONFIG,
    "logistic_regression_config": LR_CONFIG,
    "tfidf_fit_seconds": tfidf_fit_seconds,
    "classifier_fit_seconds": classifier_fit_seconds,
    "total_fit_seconds": baseline_fit_seconds,
    "validation_transform_seconds": tfidf_val_transform_seconds,
    "train_minus_validation_auroc": auc_gap,
    "split_metadata": json.loads(SPLIT_META_PATH.read_text()),
    "versions": VERSIONS,
}

(BASELINE_DIR / "metadata.json").write_text(
    json.dumps(baseline_metadata, indent=2),
    encoding="utf-8",
)

print(f"Baseline results saved to: {BASELINE_DIR.resolve()}")

## 4. Model Comparison and Hyperparameter Tuning


### 4.1 Define Candidate Models


In [ ]:
import json
import joblib
from time import perf_counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.metrics import (
    roc_auc_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_curve,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

# Keep this experiment separate from earlier coarse-to-fine searches.
COMPARISON_DIR = OUTPUT_DIR / "model_comparison_5fold_5values"
COMPARISON_DIR.mkdir(parents=True, exist_ok=True)

# All models use the same TF-IDF settings as the baseline.
COMPARISON_TFIDF_CONFIG = TFIDF_CONFIG.copy()

MODEL_SPECS = {
    "Logistic Regression": {
        "file_tag": "logistic_regression",
        "classifier": LogisticRegression(
            solver="liblinear",
            max_iter=5000,
            random_state=SEED,
        ),
        "param_grid": {
            "classifier__C": [0.01, 0.1, 1.0, 10.0, 100.0],
        },
    },
    "Multinomial Naive Bayes": {
        "file_tag": "multinomial_nb",
        "classifier": MultinomialNB(),
        "param_grid": {
            "classifier__alpha": [0.001, 0.01, 0.1, 1.0, 10.0],
        },
    },
    "Linear SVM": {
        "file_tag": "linear_svm",
        "classifier": LinearSVC(
            dual=True,
            max_iter=20_000,
            random_state=SEED,
        ),
        "param_grid": {
            "classifier__C": [0.01, 0.1, 1.0, 10.0, 100.0],
        },
    },
}


def make_candidate_pipeline(classifier):
    """Create a fresh pipeline for leakage-free cross-validation."""
    return Pipeline([
        (
            "tfidf",
            TfidfVectorizer(
                **COMPARISON_TFIDF_CONFIG,
                dtype=np.float32,
            ),
        ),
        ("classifier", clone(classifier)),
    ])


for name, spec in MODEL_SPECS.items():
    print(f"{name}: {spec['param_grid']}")

### 4.2 Establish a Fair Comparison Protocol


In [ ]:
CV_FOLDS = 5
N_JOBS = 2

# Preserve the fixed 8,000/2,000 split from Section 2.
assert len(X_train) == len(y_train) == 8000
assert len(X_val) == len(y_val) == 2000
assert set(train_split["id"]).isdisjoint(set(val_split["id"]))

# Verify that texts, labels, and IDs have matching row order.
assert np.array_equal(
    np.asarray(X_train), train_split["text"].to_numpy()
)
assert np.array_equal(
    np.asarray(y_train), train_split["label"].to_numpy()
)
assert np.array_equal(
    np.asarray(X_val), val_split["text"].to_numpy()
)
assert np.array_equal(
    np.asarray(y_val), val_split["label"].to_numpy()
)

cv = StratifiedKFold(
    n_splits=CV_FOLDS,
    shuffle=True,
    random_state=SEED,
)

# Reuse exactly the same folds for every model and parameter value.
cv_splits = list(cv.split(X_train, y_train))

fold_ids = np.full(len(X_train), -1, dtype=int)
holdout_counts = np.zeros(len(X_train), dtype=int)
fold_rows = []

for fold, (fit_indices, holdout_indices) in enumerate(cv_splits, start=1):
    fold_ids[holdout_indices] = fold
    holdout_counts[holdout_indices] += 1

    assert len(fit_indices) == 6400
    assert len(holdout_indices) == 1600
    assert np.intersect1d(fit_indices, holdout_indices).size == 0

    fold_rows.append({
        "Fold": fold,
        "Training rows": len(fit_indices),
        "Training AI share (%)": (
            100 * np.asarray(y_train)[fit_indices].mean()
        ),
        "CV holdout rows": len(holdout_indices),
        "CV holdout AI share (%)": (
            100 * np.asarray(y_train)[holdout_indices].mean()
        ),
    })

assert (holdout_counts == 1).all()

cv_fold_assignments = pd.DataFrame({
    "id": train_split["id"].to_numpy(),
    "cv_holdout_fold": fold_ids,
})
cv_fold_assignments.to_csv(
    COMPARISON_DIR / "cv_fold_assignments.csv",
    index=False,
)

display(pd.DataFrame(fold_rows))

print("Selection metric: mean 5-fold CV AUROC.")
print("Each fold: 6,400 training rows and 1,600 CV holdout rows.")
print("The fixed 2,000-row validation set is excluded from CV.")
print("Total: 75 CV fits plus 3 full-training refits.")

### 4.3 Tune Logistic Regression, Linear SVM, and Multinomial Naive Bayes


In [ ]:
searches = {}
search_rows = []

for name, spec in MODEL_SPECS.items():
    print(f"\nTuning {name}...", flush=True)

    search = GridSearchCV(
        estimator=make_candidate_pipeline(spec["classifier"]),
        param_grid=spec["param_grid"],
        scoring="roc_auc",
        cv=cv_splits,
        n_jobs=N_JOBS,
        pre_dispatch=N_JOBS,
        refit=True,
        return_train_score=False,
        error_score="raise",
        verbose=1,
    )

    # Use raw text so each fold learns its own vocabulary and IDF weights.
    start = perf_counter()
    search.fit(X_train, y_train)
    search_seconds = perf_counter() - start

    searches[name] = search
    best_index = search.best_index_

    search_rows.append({
        "Model": name,
        "CV AUROC": float(search.best_score_),
        "CV AUROC std": float(
            search.cv_results_["std_test_score"][best_index]
        ),
        "Best parameters": json.dumps(search.best_params_),
        "Search seconds": search_seconds,
        "Refit seconds": float(search.refit_time_),
    })

    # Save all candidate results and the best fitted pipeline.
    pd.DataFrame(search.cv_results_).to_csv(
        COMPARISON_DIR / f"{spec['file_tag']}_cv_results.csv",
        index=False,
    )
    joblib.dump(
        search.best_estimator_,
        COMPARISON_DIR / f"{spec['file_tag']}_model.joblib",
    )

    print("Best parameters:", search.best_params_)
    print(f"Best CV AUROC: {search.best_score_:.4f}")
    print(f"Search time: {search_seconds:.2f} seconds")

cv_summary = (
    pd.DataFrame(search_rows)
    .sort_values("CV AUROC", ascending=False, kind="stable")
    .reset_index(drop=True)
)

cv_summary.to_csv(
    COMPARISON_DIR / "cv_summary.csv",
    index=False,
)
display(cv_summary.round(4))

# Visualize the five candidate values for each model.
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

for ax, (name, search) in zip(axes, searches.items()):
    results = pd.DataFrame(search.cv_results_)
    param_name = next(iter(MODEL_SPECS[name]["param_grid"]))

    results["parameter_value"] = (
        results[f"param_{param_name}"].astype(float)
    )
    results = results.sort_values("parameter_value")

    x = results["parameter_value"].to_numpy()
    mean_auc = results["mean_test_score"].to_numpy()
    std_auc = results["std_test_score"].to_numpy()

    ax.plot(
        x, mean_auc,
        marker="o",
        color="#4070A0",
        label="Mean CV AUROC",
    )
    ax.fill_between(
        x,
        np.clip(mean_auc - std_auc, 0, 1),
        np.clip(mean_auc + std_auc, 0, 1),
        color="#4070A0",
        alpha=0.18,
        label="±1 fold standard deviation",
    )
    ax.scatter(
        float(search.best_params_[param_name]),
        search.best_score_,
        marker="*",
        s=200,
        color="#DB8B39",
        zorder=5,
        label="Selected value",
    )

    ax.set_xscale("log")
    ax.set_xticks(x)
    ax.set_xticklabels([f"{value:g}" for value in x])
    ax.set(
        title=name,
        xlabel=param_name.replace("classifier__", ""),
        ylabel="CV AUROC",
    )
    ax.grid(alpha=0.2)
    ax.legend(fontsize=8)

fig.suptitle("Five candidate values with five-fold CV")
fig.tight_layout()
fig.savefig(
    COMPARISON_DIR / "hyperparameter_comparison.png",
    dpi=160,
    bbox_inches="tight",
)
plt.show()

print("Shaded bands show fold variation, not confidence intervals.")

### 4.4 Compare Training AUROC, Validation AUROC, and Runtime


In [ ]:
def evaluate_candidate(model, texts, labels):
    """Evaluate classifiers using either AI probabilities or SVM scores."""
    labels = np.asarray(labels)
    classes = np.asarray(model.classes_)

    assert np.array_equal(classes, [0, 1])
    assert set(np.unique(labels).tolist()) == {0, 1}

    start = perf_counter()

    if hasattr(model, "predict_proba"):
        ai_column = int(np.flatnonzero(classes == 1)[0])
        scores = model.predict_proba(texts)[:, ai_column]
        threshold = DECISION_THRESHOLD
        score_type = "AI probability"
    else:
        # For binary LinearSVC, positive scores indicate class 1.
        scores = model.decision_function(texts)
        threshold = 0.0
        score_type = "Decision score"

    predictions = (scores >= threshold).astype(int)
    prediction_seconds = perf_counter() - start

    assert len(scores) == len(labels)
    assert np.isfinite(scores).all()

    # AUROC uses continuous scores rather than thresholded predictions.
    metrics = {
        "AUROC": roc_auc_score(labels, scores),
        "Accuracy": accuracy_score(labels, predictions),
        "Precision": precision_score(
            labels, predictions, zero_division=0
        ),
        "Recall": recall_score(
            labels, predictions, zero_division=0
        ),
        "F1": f1_score(
            labels, predictions, zero_division=0
        ),
        "Prediction seconds": prediction_seconds,
        "Score type": score_type,
        "Threshold": threshold,
    }

    return metrics, scores, predictions


comparison_rows = []
validation_outputs = {}

for name, search in searches.items():
    model = search.best_estimator_

    train_metrics, _, _ = evaluate_candidate(
        model, X_train, y_train
    )
    val_metrics, scores, predictions = evaluate_candidate(
        model, X_val, y_val
    )

    # Reuse these outputs for the selected-model diagnostic plots.
    validation_outputs[name] = {
        "metrics": val_metrics,
        "scores": scores,
        "predictions": predictions,
    }

    comparison_rows.append({
        "Model": name,
        "Train AUROC": train_metrics["AUROC"],
        "Validation AUROC": val_metrics["AUROC"],
        "AUROC gap": train_metrics["AUROC"] - val_metrics["AUROC"],
        "Validation accuracy": val_metrics["Accuracy"],
        "Validation precision": val_metrics["Precision"],
        "Validation recall": val_metrics["Recall"],
        "Validation F1": val_metrics["F1"],
        "Validation prediction seconds": val_metrics["Prediction seconds"],
        "Score type": val_metrics["Score type"],
        "Threshold": val_metrics["Threshold"],
    })

    # SVM decision scores are not probabilities.
    pd.DataFrame({
        "id": val_split["id"].to_numpy(),
        "label": np.asarray(y_val),
        "score": scores,
        "score_type": val_metrics["Score type"],
        "pred_label": predictions,
    }).to_csv(
        COMPARISON_DIR
        / f"{MODEL_SPECS[name]['file_tag']}_validation_predictions.csv",
        index=False,
    )

comparison_results = (
    cv_summary.merge(
        pd.DataFrame(comparison_rows),
        on="Model",
        validate="one_to_one",
    )
    .sort_values("CV AUROC", ascending=False, kind="stable")
    .reset_index(drop=True)
)

display(comparison_results[[
    "Model", "CV AUROC", "CV AUROC std",
    "Train AUROC", "Validation AUROC", "AUROC gap",
]].round(4))

display(comparison_results[[
    "Model", "Validation accuracy", "Validation precision",
    "Validation recall", "Validation F1",
]].round(4))

display(comparison_results[[
    "Model", "Search seconds", "Refit seconds",
    "Validation prediction seconds",
]].round(3))

comparison_results.to_csv(
    COMPARISON_DIR / "comparison_results.csv",
    index=False,
)

# Compare predictive performance and computational cost.
plot_results = comparison_results.set_index("Model")
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

plot_results[[
    "Train AUROC", "CV AUROC", "Validation AUROC",
]].plot.bar(
    ax=axes[0, 0],
    color=["#8BA6BE", "#469A74", "#DB8B39"],
    rot=15,
)
axes[0, 0].set(
    title="AUROC comparison",
    xlabel="",
    ylabel="AUROC",
    ylim=(0, 1.08),
)
axes[0, 0].legend(fontsize=9, loc="lower right")

plot_results[[
    "Validation accuracy", "Validation F1",
]].plot.bar(
    ax=axes[0, 1],
    color=["#4070A0", "#DB8B39"],
    rot=15,
)
axes[0, 1].set(
    title="Validation classification performance",
    xlabel="",
    ylabel="Score",
    ylim=(0, 1.08),
)
axes[0, 1].legend(
    ["Accuracy", "F1"], fontsize=9, loc="lower right"
)

plot_results["Search seconds"].plot.bar(
    ax=axes[1, 0], color="#4070A0", rot=15,
)
axes[1, 0].set(
    title="Total five-fold parameter-search time",
    xlabel="",
    ylabel="Seconds",
)

plot_results["Refit seconds"].plot.bar(
    ax=axes[1, 1], color="#469A74", rot=15,
)
axes[1, 1].set(
    title="Selected pipeline refit time on 8,000 rows",
    xlabel="",
    ylabel="Seconds",
)

for ax in axes.ravel():
    ax.grid(axis="y", alpha=0.2)
    ax.set_axisbelow(True)

for ax in axes[0]:
    for container in ax.containers:
        ax.bar_label(container, fmt="%.3f", padding=3, fontsize=8)

for ax in axes[1]:
    ax.margins(y=0.2)
    for container in ax.containers:
        ax.bar_label(container, fmt="%.1f", padding=3, fontsize=9)

fig.tight_layout()
fig.savefig(
    COMPARISON_DIR / "model_performance_and_runtime.png",
    dpi=160,
    bbox_inches="tight",
)
plt.show()

print("Search time includes all CV fits and the final refit.")
print("Refit time includes TF-IDF and classifier fitting on 8,000 rows.")
print("Prediction time includes TF-IDF transformation.")
print("CV AUROC averages models trained on 6,400 rows per fold.")

### 4.5 Select the Best Model Configuration


In [ ]:
# Select by training-set CV AUROC, not held-out validation performance.
best_model_name = cv_summary.iloc[0]["Model"]
best_search = searches[best_model_name]
best_model = best_search.best_estimator_

selected_result = comparison_results.loc[
    comparison_results["Model"].eq(best_model_name)
].iloc[0]

selected_output = validation_outputs[best_model_name]
selected_metrics = selected_output["metrics"]
selected_scores = selected_output["scores"]
selected_predictions = selected_output["predictions"]

true_labels = np.asarray(y_val)
score_type = selected_metrics["Score type"]
threshold = selected_metrics["Threshold"]

print("Selected model:", best_model_name)
print("Best parameters:", best_search.best_params_)
print(f"CV AUROC: {selected_result['CV AUROC']:.4f}")
print(f"Training AUROC: {selected_result['Train AUROC']:.4f}")
print(f"Validation AUROC: {selected_result['Validation AUROC']:.4f}")
print(f"AUROC gap: {selected_result['AUROC gap']:.4f}")

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

# ROC measures ranking quality across decision thresholds.
fpr, tpr, _ = roc_curve(true_labels, selected_scores)
axes[0].plot(
    fpr, tpr,
    color="#4070A0",
    label=f"AUROC = {selected_metrics['AUROC']:.4f}",
)
axes[0].plot(
    [0, 1], [0, 1],
    "--", color="gray", label="Chance",
)
axes[0].set(
    title="Validation ROC curve",
    xlabel="False positive rate",
    ylabel="True positive rate",
)
axes[0].legend(loc="lower right")
axes[0].grid(alpha=0.2)

# Rows represent true labels; columns represent predicted labels.
cm = confusion_matrix(
    true_labels, selected_predictions, labels=[0, 1]
)
ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Human", "AI"],
).plot(
    ax=axes[1],
    cmap="Blues",
    colorbar=False,
    values_format="d",
)
axes[1].set_title(f"Confusion matrix (threshold = {threshold:g})")

# Shared bins make the two score distributions directly comparable.
bins = np.histogram_bin_edges(selected_scores, bins=35)

for label, name, color in [
    (0, "True Human", "#4070A0"),
    (1, "True AI", "#DB8B39"),
]:
    axes[2].hist(
        selected_scores[true_labels == label],
        bins=bins,
        density=True,
        alpha=0.5,
        color=color,
        label=name,
    )

axes[2].axvline(
    threshold,
    color="black",
    linestyle="--",
    label=f"Threshold = {threshold:g}",
)
axes[2].set(
    title="Validation score distributions",
    xlabel=score_type,
    ylabel="Density",
)
axes[2].legend(fontsize=9)

fig.suptitle(f"Selected by CV AUROC: {best_model_name}")
fig.tight_layout()
fig.savefig(
    COMPARISON_DIR / "selected_model_diagnostics.png",
    dpi=160,
    bbox_inches="tight",
)
plt.show()

# Preserve the fitted pipeline and experiment settings.
joblib.dump(
    best_model,
    COMPARISON_DIR / "best_model.joblib",
)

selection_metadata = {
    "selected_model": best_model_name,
    "selection_rule": "Highest mean training-set CV AUROC",
    "best_parameters": best_search.best_params_,
    "cv_folds": CV_FOLDS,
    "seed": SEED,
    "training_rows": len(X_train),
    "validation_rows": len(X_val),
    "cv_auroc": float(selected_result["CV AUROC"]),
    "cv_auroc_std": float(selected_result["CV AUROC std"]),
    "training_auroc": float(selected_result["Train AUROC"]),
    "validation_auroc": float(selected_result["Validation AUROC"]),
    "score_type": score_type,
    "classification_threshold": float(threshold),
    "tfidf_config": COMPARISON_TFIDF_CONFIG,
    "searched_parameters": {
        name: spec["param_grid"]
        for name, spec in MODEL_SPECS.items()
    },
    "split_metadata": json.loads(SPLIT_META_PATH.read_text()),
    "versions": VERSIONS,
}

(COMPARISON_DIR / "best_model_metadata.json").write_text(
    json.dumps(selection_metadata, indent=2),
    encoding="utf-8",
)

print(f"\nResults saved to: {COMPARISON_DIR.resolve()}")

if not hasattr(best_model, "predict_proba"):
    print(
        "The selected SVM produces decision scores, not probabilities. "
        "Probability calibration is needed before a probability submission."
    )

## 5. Effect of Training Data Size


### 5.1 Define the Controlled Experiment


In [ ]:
import json
from pathlib import Path
from time import perf_counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.base import clone

SIZE_DIR = Path(OUTPUT_DIR) / "training_size_experiment"
SIZE_DIR.mkdir(parents=True, exist_ok=True)

SIZE_VALUES = [500, 1000, 2000, 5000, 8000]

size_train_df = train_split.reset_index(drop=True).copy()
size_val_df = val_split.reset_index(drop=True).copy()

assert len(size_train_df) == 8000 and len(size_val_df) == 2000
assert size_train_df["id"].is_unique and size_val_df["id"].is_unique
assert set(size_train_df["id"]).isdisjoint(size_val_df["id"])
assert size_train_df["label"].value_counts().to_dict() == {
    0: 4000, 1: 4000
}

# Keep hyperparameters, but remove fitted vocabulary and coefficients.
size_model_template = clone(best_model)

size_metadata = {
    "model": best_model_name,
    "selected_parameters": best_search.best_params_,
    "training_sizes": SIZE_VALUES,
    "sampling_seed": SEED,
    "sampling_method": (
        "Nested stratified subsets, preserving original row order"
    ),
    "validation_rows": len(size_val_df),
    "selection_scope": (
        "Configuration selected using the 8000-row training "
        "partition in Section 4"
    ),
    "tfidf_parameters": (
        size_model_template.named_steps["tfidf"].get_params()
    ),
    "classifier_parameters": (
        size_model_template.named_steps["classifier"].get_params()
    ),
    "split_metadata": json.loads(SPLIT_META_PATH.read_text()),
    "versions": VERSIONS,
}

(SIZE_DIR / "experiment_metadata.json").write_text(
    json.dumps(size_metadata, indent=2, default=str),
    encoding="utf-8",
)

print("Fixed model:", best_model_name)
print("Fixed selected parameters:", best_search.best_params_)
print("Training sizes:", SIZE_VALUES)
print("The same 2,000 validation rows are used throughout.")
print("Five fresh pipeline fits; no additional hyperparameter search.")

### 5.2 Construct Training Subsets (Stratified Sampling)


In [ ]:
rng = np.random.default_rng(SEED)
size_labels = size_train_df["label"].to_numpy()

# Shuffle each class once to construct nested training subsets.
class_order = {
    label: rng.permutation(np.flatnonzero(size_labels == label))
    for label in [0, 1]
}

size_subset_indices = {}
subset_rows = []
assignment_frames = []
previous_indices = set()

for n in SIZE_VALUES:
    indices = np.sort(np.concatenate([
        class_order[0][:n // 2],
        class_order[1][:n // 2],
    ]))

    subset = size_train_df.iloc[indices]

    assert len(indices) == n and len(np.unique(indices)) == n
    assert previous_indices.issubset(set(indices))
    assert subset["label"].value_counts().to_dict() == {
        0: n // 2, 1: n // 2
    }

    size_subset_indices[n] = indices
    previous_indices = set(indices)

    subset_rows.append({
        "Training rows": n,
        "Human": int(subset["label"].eq(0).sum()),
        "AI": int(subset["label"].eq(1).sum()),
        "AI share (%)": 100 * subset["label"].mean(),
    })

    assignment_frames.append(
        subset[["id", "label"]].assign(training_rows=n)
    )

# Preserve the original training order at the full-data endpoint.
assert np.array_equal(
    size_subset_indices[8000], np.arange(8000)
)

subset_summary = pd.DataFrame(subset_rows)
display(subset_summary)

subset_summary.to_csv(
    SIZE_DIR / "subset_summary.csv",
    index=False,
)

pd.concat(assignment_frames, ignore_index=True).to_csv(
    SIZE_DIR / "subset_assignments.csv",
    index=False,
)

### 5.3 Train and Evaluate Each Subset


In [ ]:
learning_rows = []
size_prediction_frames = []

for n in SIZE_VALUES:
    subset = size_train_df.iloc[size_subset_indices[n]]
    size_model = clone(size_model_template)

    # Learn TF-IDF and classifier parameters from this subset only.
    start = perf_counter()
    size_model.fit(subset["text"], subset["label"])
    fit_seconds = perf_counter() - start

    # Section 4.4 supports both probabilities and SVM decision scores.
    train_metrics, _, _ = evaluate_candidate(
        size_model,
        subset["text"],
        subset["label"],
    )

    val_metrics, val_scores, val_predictions = evaluate_candidate(
        size_model,
        size_val_df["text"],
        size_val_df["label"],
    )

    learning_rows.append({
        "Training rows": n,
        "Training AUROC": train_metrics["AUROC"],
        "Validation AUROC": val_metrics["AUROC"],
        "AUROC gap": (
            train_metrics["AUROC"] - val_metrics["AUROC"]
        ),
        "Training seconds": fit_seconds,
        "Features": len(
            size_model.named_steps["tfidf"].vocabulary_
        ),
    })

    size_prediction_frames.append(pd.DataFrame({
        "training_rows": n,
        "id": size_val_df["id"].to_numpy(),
        "label": size_val_df["label"].to_numpy(),
        "score": val_scores,
        "score_type": val_metrics["Score type"],
        "threshold": val_metrics["Threshold"],
        "pred_label": val_predictions,
    }))

    print(
        f"n={n:>5,} | "
        f"Train AUROC={train_metrics['AUROC']:.4f} | "
        f"Validation AUROC={val_metrics['AUROC']:.4f} | "
        f"Fit time={fit_seconds:.2f}s",
        flush=True,
    )

learning_results = (
    pd.DataFrame(learning_rows)
    .sort_values("Training rows")
    .reset_index(drop=True)
)

learning_results.to_csv(
    SIZE_DIR / "learning_curve_results.csv",
    index=False,
)

pd.concat(size_prediction_frames, ignore_index=True).to_csv(
    SIZE_DIR / "validation_predictions.csv",
    index=False,
)

display(learning_results.round(4))

print(
    "Training time includes TF-IDF and classifier fitting, "
    "excluding evaluation."
)

### 5.4 Plot Training and Validation Learning Curves


In [ ]:
x = learning_results["Training rows"].to_numpy()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(
    x,
    learning_results["Training AUROC"].to_numpy(),
    marker="o",
    color="#4070A0",
    label="Training AUROC",
)

axes[0].plot(
    x,
    learning_results["Validation AUROC"].to_numpy(),
    marker="o",
    color="#DB8B39",
    label="Validation AUROC",
)

axes[0].axhline(
    0.5,
    color="gray",
    linestyle="--",
    label="Chance",
)

minimum_auc = learning_results[
    ["Training AUROC", "Validation AUROC"]
].min().min()

axes[0].set(
    title="Learning curves",
    xlabel="Number of training examples",
    ylabel="AUROC",
    ylim=(max(0, min(0.48, minimum_auc - 0.02)), 1.02),
)
axes[0].legend(loc="lower right")

axes[1].plot(
    x,
    learning_results["Training seconds"].to_numpy(),
    marker="o",
    color="#469A74",
)

axes[1].set(
    title="Pipeline training time",
    xlabel="Number of training examples",
    ylabel="Seconds",
)
axes[1].set_ylim(bottom=0)

for ax in axes:
    ax.set_xticks(x)
    ax.set_xticklabels(
        [f"{n:,}" for n in x],
        rotation=35,
    )
    ax.grid(alpha=0.2)

fig.suptitle(
    f"Effect of training data size: {best_model_name}"
)
fig.tight_layout()

fig.savefig(
    SIZE_DIR / "learning_curves_and_runtime.png",
    dpi=160,
    bbox_inches="tight",
)
plt.show()

### 5.5 Interpret Overfitting, Underfitting, and Data Efficiency


In [ ]:
size_analysis = learning_results.copy()

size_analysis["Added rows"] = (
    size_analysis["Training rows"].diff()
)

size_analysis["Validation AUROC change"] = (
    size_analysis["Validation AUROC"].diff()
)

# Normalize gains because the increases in training size are unequal.
size_analysis["AUROC gain per 1,000 added rows"] = (
    1000
    * size_analysis["Validation AUROC change"]
    / size_analysis["Added rows"]
)

display(size_analysis[[
    "Training rows",
    "Training AUROC",
    "Validation AUROC",
    "AUROC gap",
    "Added rows",
    "Validation AUROC change",
    "AUROC gain per 1,000 added rows",
]].round(4))

size_analysis.to_csv(
    SIZE_DIR / "data_efficiency_analysis.csv",
    index=False,
)

first = learning_results.iloc[0]
last = learning_results.iloc[-1]
previous = learning_results.iloc[-2]

overall_gain = (
    last["Validation AUROC"] - first["Validation AUROC"]
)
last_gain = (
    last["Validation AUROC"] - previous["Validation AUROC"]
)

summary = (
    f"Model: {best_model_name}\n"
    f"Validation AUROC, 500 to 8,000 rows: "
    f"{first['Validation AUROC']:.4f} -> "
    f"{last['Validation AUROC']:.4f}\n"
    f"Overall validation AUROC change: {overall_gain:+.4f}\n"
    f"Validation AUROC change, 5,000 to 8,000 rows: "
    f"{last_gain:+.4f}\n"
    f"Training-validation gap, 500 to 8,000 rows: "
    f"{first['AUROC gap']:.4f} -> "
    f"{last['AUROC gap']:.4f}\n\n"

    # "Interpretation guide:\n"
    # "- A high training score with a lower validation score "
    # "is consistent with overfitting.\n"
    # "- Low scores on both splits may suggest underfitting "
    # "or limited text features.\n"
    # "- Rising validation AUROC suggests additional labeled "
    # "data helps this configuration.\n"
    # "- A flattening validation curve suggests diminishing "
    # "returns over the tested range.\n\n"

    # "Scope: these results use one nested sampling sequence "
    # "and one fixed validation set. Small differences may "
    # "reflect sampling variation. The configuration was selected "
    # "using all 8,000 training rows in Section 4, so the "
    # "small-subset scores do not represent an entire "
    # "model-selection process performed with only that subset. "
    # "The curves alone do not establish the cause of a "
    # "performance gap."
)

print(summary)

(SIZE_DIR / "interpretation_notes.txt").write_text(
    summary,
    encoding="utf-8",
)

print("\nResults saved to:", SIZE_DIR.resolve())

## 6. Learning from Unlabeled Data


### 6.1 Define the Semi-Supervised Learning Method


In [ ]:
import json
import hashlib
import joblib
from pathlib import Path
from time import perf_counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.base import clone
from sklearn.metrics import roc_curve

SSL_DIR = Path(OUTPUT_DIR) / "semi_supervised_single_round"
SSL_DIR.mkdir(parents=True, exist_ok=True)

PSEUDO_THRESHOLD = 0.95
assert 0.5 < PSEUDO_THRESHOLD < 1.0

ssl_train = (
    train_split[["id", "text", "label"]]
    .reset_index(drop=True).copy()
)
ssl_val = (
    val_split[["id", "text", "label"]]
    .reset_index(drop=True).copy()
)
ssl_unlabeled = (
    unlabeled_df[["id", "text"]]
    .reset_index(drop=True).copy()
)

assert (
    len(ssl_train), len(ssl_val), len(ssl_unlabeled)
) == (8000, 2000, 3000)

for frame in [ssl_train, ssl_val, ssl_unlabeled]:
    assert not frame.isna().any().any()
    assert frame["id"].is_unique
    assert frame["text"].map(
        lambda text: isinstance(text, str)
    ).all()
    assert frame["text"].str.strip().ne("").all()

# Reuse Section 2's normalization only for overlap checks.
for left, right in [
    (ssl_train, ssl_val),
    (ssl_train, ssl_unlabeled),
    (ssl_val, ssl_unlabeled),
]:
    assert set(left["id"]).isdisjoint(right["id"])
    assert set(
        normalize_for_audit(left["text"])
    ).isdisjoint(
        normalize_for_audit(right["text"])
    ), "Review overlapping texts before this experiment."

ssl_template = clone(best_model)

if not hasattr(ssl_template, "predict_proba"):
    raise TypeError(
        "This experiment requires predict_proba, as provided "
        "by the selected logistic regression."
    )


def save_ssl_figure(fig, stem):
    """Save both a preview image and a vector PDF."""
    fig.tight_layout()
    fig.savefig(
        SSL_DIR / f"{stem}.png",
        dpi=200,
        bbox_inches="tight",
    )
    fig.savefig(
        SSL_DIR / f"{stem}.pdf",
        bbox_inches="tight",
    )


print("Model:", best_model_name)
print("Fixed parameters:", best_search.best_params_)
print("Pseudo-label confidence threshold:", PSEUDO_THRESHOLD)
print(
    "One round; equal sample weights; "
    "no additional parameter search."
)

### 6.2 Generate and Select Pseudo-Labels


In [ ]:
# Refit a clean teacher on the 8,000 true labels only.
ssl_teacher = clone(ssl_template)

start = perf_counter()
ssl_teacher.fit(ssl_train["text"], ssl_train["label"])
teacher_fit_seconds = perf_counter() - start

assert np.array_equal(ssl_teacher.classes_, [0, 1])

start = perf_counter()

ai_column = int(
    np.flatnonzero(ssl_teacher.classes_ == 1)[0]
)
unlabeled_prob = ssl_teacher.predict_proba(
    ssl_unlabeled["text"]
)[:, ai_column]

assert np.isfinite(unlabeled_prob).all()
assert (
    (unlabeled_prob >= 0) & (unlabeled_prob <= 1)
).all()

pseudo_labels = (unlabeled_prob >= 0.5).astype(int)
pseudo_confidence = np.maximum(
    unlabeled_prob, 1 - unlabeled_prob
)
selected_mask = pseudo_confidence >= PSEUDO_THRESHOLD

pseudo_seconds = perf_counter() - start

pseudo_audit = pd.DataFrame({
    "id": ssl_unlabeled["id"].to_numpy(),
    "prob_ai": unlabeled_prob,
    "pseudo_label": pseudo_labels,
    "confidence": pseudo_confidence,
    "selected": selected_mask,
})
pseudo_audit.to_csv(
    SSL_DIR / "pseudo_label_audit.csv",
    index=False,
)

n_pseudo = int(selected_mask.sum())
n_human = int(
    (selected_mask & (pseudo_labels == 0)).sum()
)
n_ai = int(
    (selected_mask & (pseudo_labels == 1)).sum()
)

selection_summary = pd.DataFrame({
    "Group": [
        "Selected Human",
        "Selected AI",
        "Not selected",
    ],
    "Rows": [
        n_human,
        n_ai,
        len(ssl_unlabeled) - n_pseudo,
    ],
})

display(selection_summary)
selection_summary.to_csv(
    SSL_DIR / "selection_summary.csv",
    index=False,
)

print(
    f"Selected: {n_pseudo:,}/{len(ssl_unlabeled):,} "
    f"({100 * selected_mask.mean():.1f}%)"
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

axes[0].hist(
    unlabeled_prob,
    bins=np.linspace(0, 1, 31),
    color="#839DB4",
    edgecolor="white",
)

for cutoff in [
    1 - PSEUDO_THRESHOLD,
    PSEUDO_THRESHOLD,
]:
    axes[0].axvline(
        cutoff,
        color="#333333",
        linestyle="--",
        linewidth=1,
    )

axes[0].axvspan(
    0, 1 - PSEUDO_THRESHOLD,
    color="#4070A0", alpha=0.15,
)
axes[0].axvspan(
    PSEUDO_THRESHOLD, 1,
    color="#DB8B39", alpha=0.15,
)
axes[0].set(
    title="Teacher probabilities on unlabeled text",
    xlabel="Predicted P(AI)",
    ylabel="Count",
    xlim=(0, 1),
)

bars = axes[1].bar(
    selection_summary["Group"].tolist(),
    selection_summary["Rows"].to_numpy(),
    color=["#4070A0", "#DB8B39", "#B5BCC2"],
)
axes[1].bar_label(bars, padding=3)
axes[1].margins(y=0.15)
axes[1].set(
    title="Pseudo-label selection",
    ylabel="Count",
)

for ax in axes:
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

save_ssl_figure(fig, "pseudo_label_selection")
plt.show()

### 6.3 Retrain with Labeled and Pseudo-Labeled Data


In [ ]:
pseudo_train = ssl_unlabeled.loc[
    selected_mask, ["id", "text"]
].copy()
pseudo_train["label"] = pseudo_labels[selected_mask]

ssl_augmented = pd.concat([
    ssl_train.assign(label_source="true"),
    pseudo_train.assign(label_source="pseudo"),
], ignore_index=True)

assert len(ssl_augmented) == 8000 + n_pseudo
assert ssl_augmented["id"].is_unique
assert set(ssl_augmented["id"]).isdisjoint(ssl_val["id"])

ssl_augmented[[
    "id", "label", "label_source"
]].to_csv(
    SSL_DIR / "training_assignments.csv",
    index=False,
)

# Relearn vocabulary, IDF, and classifier on augmented data.
ssl_student = clone(ssl_template)

start = perf_counter()
ssl_student.fit(
    ssl_augmented["text"],
    ssl_augmented["label"],
)
student_fit_seconds = perf_counter() - start

joblib.dump(
    ssl_teacher,
    SSL_DIR / "supervised_model.joblib",
)
joblib.dump(
    ssl_student,
    SSL_DIR / "pseudo_label_model.joblib",
)

print(f"Student training rows: {len(ssl_augmented):,}")
print(
    f"Student pipeline fit time: "
    f"{student_fit_seconds:.2f}s"
)

if n_pseudo == 0:
    print(
        "No samples passed the threshold; "
        "the student is a supervised refit."
    )

### 6.4 Compare Against the Supervised Baseline


In [ ]:
ssl_rows = []
ssl_predictions = {}

ssl_specs = [
    (
        "Supervised only",
        ssl_teacher,
        0,
        teacher_fit_seconds,
        teacher_fit_seconds,
    ),
    (
        "With pseudo-labels",
        ssl_student,
        n_pseudo,
        student_fit_seconds,
        teacher_fit_seconds + pseudo_seconds + student_fit_seconds,
    ),
]

for method, model, added_rows, fit_time, workflow_time in ssl_specs:
    # Use the same TRUE training labels for both models.
    train_metrics, _, _ = evaluate_candidate(
        model,
        ssl_train["text"],
        ssl_train["label"],
    )
    val_metrics, scores, predictions = evaluate_candidate(
        model,
        ssl_val["text"],
        ssl_val["label"],
    )

    ssl_predictions[method] = scores

    ssl_rows.append({
        "Method": method,
        "True labeled rows": len(ssl_train),
        "Pseudo-labeled rows": added_rows,
        "Training AUROC": train_metrics["AUROC"],
        "Validation AUROC": val_metrics["AUROC"],
        "Validation F1": val_metrics["F1"],
        "Fit seconds": fit_time,
        "Workflow seconds": workflow_time,
    })

    file_tag = (
        "supervised"
        if method == "Supervised only"
        else "pseudo_label"
    )

    pd.DataFrame({
        "id": ssl_val["id"].to_numpy(),
        "label": ssl_val["label"].to_numpy(),
        "prob_ai": scores,
        "pred_label": predictions,
    }).to_csv(
        SSL_DIR / f"{file_tag}_validation_predictions.csv",
        index=False,
    )

ssl_results = pd.DataFrame(ssl_rows)
ssl_results.to_csv(
    SSL_DIR / "comparison_results.csv",
    index=False,
)

display(ssl_results.round({
    "Training AUROC": 6,
    "Validation AUROC": 6,
    "Validation F1": 4,
    "Fit seconds": 2,
    "Workflow seconds": 2,
}))

ssl_auc_before, ssl_auc_after = (
    ssl_results["Validation AUROC"].to_numpy()
)
ssl_auc_change = float(ssl_auc_after - ssl_auc_before)

print(f"Validation AUROC change: {ssl_auc_change:+.6f}")
print(
    "Workflow time includes fitting and pseudo-label generation, "
    "excluding evaluation and file saving."
)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
colors = ["#4070A0", "#DB8B39"]

for color, method in zip(colors, ssl_results["Method"]):
    fpr, tpr, _ = roc_curve(
        ssl_val["label"],
        ssl_predictions[method],
    )

    auc = float(ssl_results.loc[
        ssl_results["Method"].eq(method),
        "Validation AUROC",
    ].iloc[0])

    axes[0].plot(
        fpr, tpr,
        color=color,
        label=f"{method}: {auc:.6f}",
    )

axes[0].plot(
    [0, 1], [0, 1],
    "--", color="gray", linewidth=1,
)
axes[0].set(
    title="Fixed-validation ROC curves",
    xlabel="False positive rate",
    ylabel="True positive rate",
)
axes[0].legend(loc="lower right", fontsize=9)

# Explicitly label the zoomed axis when showing small differences.
auc_values = ssl_results["Validation AUROC"].to_numpy()

axes[1].scatter(
    auc_values, [0, 1],
    c=colors, s=75,
)

for i, value in enumerate(auc_values):
    axes[1].annotate(
        f"{value:.6f}",
        (value, i),
        xytext=(0, 10),
        textcoords="offset points",
        ha="center",
    )

axes[1].set_yticks([0, 1])
axes[1].set_yticklabels(
    ssl_results["Method"].tolist()
)

padding = max(
    0.005,
    float(np.ptp(auc_values)) * 0.8,
)

axes[1].set(
    xlim=(
        max(0, auc_values.min() - padding),
        min(1, auc_values.max() + padding),
    ),
    ylim=(-0.5, 1.5),
    xlabel="Validation AUROC",
    title=(
        "AUROC comparison (zoomed axis)\n"
        f"Change = {ssl_auc_change:+.6f}"
    ),
)

for ax in axes:
    ax.grid(alpha=0.18)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

save_ssl_figure(fig, "validation_comparison")
plt.show()

### 6.5 Discuss Pseudo-Label Noise and Validation Leakage Prevention


In [ ]:
if n_pseudo == 0:
    outcome = (
        "No pseudo-labels were selected; this run did not "
        "add unlabeled training data."
    )
elif ssl_auc_change > 0:
    outcome = (
        "Pseudo-labeling improved AUROC on this fixed "
        "validation set."
    )
elif ssl_auc_change < 0:
    outcome = (
        "Pseudo-labeling reduced AUROC on this fixed "
        "validation set."
    )
else:
    outcome = "Pseudo-labeling left validation AUROC unchanged."

ssl_summary = (
    f"Selected {n_pseudo:,} of {len(ssl_unlabeled):,} "
    f"unlabeled examples using confidence >= "
    f"{PSEUDO_THRESHOLD:.2f}: "
    f"{n_human:,} pseudo-Human and {n_ai:,} pseudo-AI.\n"
    f"Validation AUROC: {ssl_auc_before:.6f} -> "
    f"{ssl_auc_after:.6f} "
    f"(change {ssl_auc_change:+.6f}). {outcome}\n\n"

    # "Pseudo-label accuracy cannot be measured because "
    # "the unlabeled examples have no ground-truth labels. "
    # "The teacher's probabilities are not explicitly calibrated, "
    # "so confidence 0.95 does not guarantee 95% correctness. "
    # "Incorrect predictions may reinforce existing errors; "
    # "selected samples may also be unrepresentative. "

    # "Both classifiers and TF-IDF use the same hyperparameter "
    # "settings, but the student learns its vocabulary and IDF "
    # "from the augmented data. This comparison measures the "
    # "combined effect of pseudo-labeling and refitting the "
    # "representation. "

    # "Validation texts and labels are excluded from fitting, "
    # "and the threshold was fixed before evaluation. "
    # "One run on one validation split does not establish "
    # "statistical significance or guarantee an improvement "
    # "on the hidden test set."
)

print(ssl_summary)

(SSL_DIR / "experiment_summary.txt").write_text(
    ssl_summary,
    encoding="utf-8",
)

ssl_metadata = {
    "method": "Single-round confidence-based pseudo-labeling",
    "model": best_model_name,
    "selected_parameters": best_search.best_params_,
    "pseudo_threshold": PSEUDO_THRESHOLD,
    "classification_threshold": DECISION_THRESHOLD,
    "sample_weight": 1.0,
    "seed": SEED,
    "selected_human": n_human,
    "selected_ai": n_ai,
    "validation_auroc_change": ssl_auc_change,
    "tfidf_parameters": (
        ssl_template.named_steps["tfidf"].get_params()
    ),
    "classifier_parameters": (
        ssl_template.named_steps["classifier"].get_params()
    ),
    "split_metadata": json.loads(
        SPLIT_META_PATH.read_text()
    ),
    "unlabeled_sha256": hashlib.sha256(
        Path(UNLABELED_PATH).read_bytes()
    ).hexdigest(),
    "versions": VERSIONS,
}

(SSL_DIR / "experiment_metadata.json").write_text(
    json.dumps(ssl_metadata, indent=2, default=str),
    encoding="utf-8",
)

print("\nResults saved to:", SSL_DIR.resolve())

## 7. Generalization and Error Analysis


### 7.1 Analyze the Training–Validation Performance Gap


In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.base import clone
from sklearn.metrics import (
    roc_auc_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

ERROR_DIR = Path(OUTPUT_DIR) / "generalization_error_analysis"
ERROR_DIR.mkdir(parents=True, exist_ok=True)

analysis_train = (
    train_split[["id", "text", "label"]]
    .reset_index(drop=True).copy()
)
analysis_val = (
    val_split[["id", "text", "label"]]
    .reset_index(drop=True).copy()
)

assert len(analysis_train) == 8000
assert len(analysis_val) == 2000
assert set(analysis_train["id"]).isdisjoint(analysis_val["id"])

# Refit the selected configuration on true training labels only.
analysis_model = clone(best_model)
analysis_model.fit(
    analysis_train["text"],
    analysis_train["label"],
)

if not hasattr(analysis_model, "predict_proba"):
    raise TypeError(
        "This analysis requires a probability classifier, "
        "such as the selected logistic regression."
    )

analysis_train_metrics, _, _ = evaluate_candidate(
    analysis_model,
    analysis_train["text"],
    analysis_train["label"],
)
analysis_val_metrics, analysis_prob, _ = evaluate_candidate(
    analysis_model,
    analysis_val["text"],
    analysis_val["label"],
)

analysis_val["prob_ai"] = analysis_prob
analysis_val["words"] = analysis_val["text"].str.count(r"\S+")

gap_results = pd.DataFrame([
    {
        "Split": "Training",
        "Rows": len(analysis_train),
        "AUROC": analysis_train_metrics["AUROC"],
    },
    {
        "Split": "Validation",
        "Rows": len(analysis_val),
        "AUROC": analysis_val_metrics["AUROC"],
    },
])

analysis_auc_gap = (
    analysis_train_metrics["AUROC"]
    - analysis_val_metrics["AUROC"]
)

display(gap_results.round(6))
print(f"Training-validation AUROC gap: {analysis_auc_gap:.6f}")

gap_results.to_csv(
    ERROR_DIR / "generalization_gap.csv",
    index=False,
)


def save_analysis_figure(fig, name):
    fig.tight_layout()
    fig.savefig(
        ERROR_DIR / f"{name}.png",
        dpi=200,
        bbox_inches="tight",
    )
    fig.savefig(
        ERROR_DIR / f"{name}.pdf",
        bbox_inches="tight",
    )

### 7.2 Evaluate Performance Across Text Length Groups


In [ ]:
LENGTH_GROUPS = [
    "Short (<=150)",
    "Medium (151-250)",
    "Long (>250)",
]

analysis_val["length_group"] = pd.cut(
    analysis_val["words"],
    bins=[-np.inf, 150, 250, np.inf],
    labels=LENGTH_GROUPS,
)

BOOTSTRAP_REPEATS = 1000


def stratified_auc_interval(labels, scores, seed):
    """Pointwise percentile interval conditional on the fitted model."""
    labels = np.asarray(labels)
    scores = np.asarray(scores)

    if len(np.unique(labels)) < 2:
        return np.nan, np.nan, np.nan

    rng = np.random.default_rng(seed)
    class_indices = [
        np.flatnonzero(labels == label)
        for label in [0, 1]
    ]

    bootstrap_auc = np.empty(BOOTSTRAP_REPEATS)

    for i in range(BOOTSTRAP_REPEATS):
        indices = np.concatenate([
            rng.choice(idx, size=len(idx), replace=True)
            for idx in class_indices
        ])
        bootstrap_auc[i] = roc_auc_score(
            labels[indices],
            scores[indices],
        )

    low, high = np.quantile(
        bootstrap_auc, [0.025, 0.975]
    )

    return roc_auc_score(labels, scores), low, high


length_rows = []

for i, group in enumerate(LENGTH_GROUPS):
    subset = analysis_val.loc[
        analysis_val["length_group"].eq(group)
    ]

    auc, low, high = stratified_auc_interval(
        subset["label"],
        subset["prob_ai"],
        SEED + i,
    )

    length_rows.append({
        "Length group": group,
        "Rows": len(subset),
        "Human": int(subset["label"].eq(0).sum()),
        "AI": int(subset["label"].eq(1).sum()),
        "Validation AUROC": auc,
        "CI low": low,
        "CI high": high,
    })

length_results = pd.DataFrame(length_rows)

assert length_results["Rows"].sum() == len(analysis_val)

display(length_results.round(4))
length_results.to_csv(
    ERROR_DIR / "length_group_results.csv",
    index=False,
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
x = np.arange(len(LENGTH_GROUPS))

axes[0].bar(
    x,
    length_results["Human"].to_numpy(),
    label="Human",
    color="#4070A0",
)
axes[0].bar(
    x,
    length_results["AI"].to_numpy(),
    bottom=length_results["Human"].to_numpy(),
    label="AI",
    color="#DB8B39",
)
axes[0].set(
    title="Validation group sizes",
    ylabel="Number of examples",
)
axes[0].legend()

for i, row in length_results.iterrows():
    if pd.notna(row["Validation AUROC"]):
        axes[1].vlines(
            i,
            row["CI low"],
            row["CI high"],
            color="#4070A0",
            linewidth=2,
        )
        axes[1].scatter(
            i,
            row["Validation AUROC"],
            color="#4070A0",
            zorder=3,
        )
        axes[1].annotate(
            f"{row['Validation AUROC']:.4f}",
            (i, row["CI high"]),
            xytext=(0, 7),
            textcoords="offset points",
            ha="center",
        )

axes[1].axhline(
    analysis_val_metrics["AUROC"],
    color="#777777",
    linestyle="--",
    label="Overall AUROC",
)
axes[1].set(
    title="AUROC with pointwise 95% bootstrap intervals",
    ylabel="Validation AUROC",
)
axes[1].margins(x=0.2, y=0.25)
axes[1].legend(loc="lower right", fontsize=8)

for ax in axes:
    ax.set_xticks(x)
    ax.set_xticklabels(LENGTH_GROUPS)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

save_analysis_figure(fig, "length_group_analysis")
plt.show()

print(
    "Intervals reflect resampling within this validation set, "
    "not model-training variability."
)
print(
    "Group differences alone do not establish that text length "
    "causes the performance difference."
)

### 7.3 Define the Classification Threshold


In [ ]:
ERROR_THRESHOLD = 0.5

analysis_val["pred_label"] = (
    analysis_val["prob_ai"] >= ERROR_THRESHOLD
).astype(int)

analysis_val["error_type"] = np.select(
    [
        analysis_val["label"].eq(0)
        & analysis_val["pred_label"].eq(1),

        analysis_val["label"].eq(1)
        & analysis_val["pred_label"].eq(0),
    ],
    ["FP", "FN"],
    default="Correct",
)

cm = confusion_matrix(
    analysis_val["label"],
    analysis_val["pred_label"],
    labels=[0, 1],
)

tn, fp, fn, tp = cm.ravel()

threshold_results = pd.DataFrame([{
    "Threshold": ERROR_THRESHOLD,
    "TN": tn,
    "FP": fp,
    "FN": fn,
    "TP": tp,
    "False positive rate": fp / (tn + fp),
    "False negative rate": fn / (tp + fn),
}])

display(threshold_results.round(4))

threshold_results.to_csv(
    ERROR_DIR / "threshold_results.csv",
    index=False,
)
analysis_val.drop(columns="text").to_csv(
    ERROR_DIR / "validation_diagnostics.csv",
    index=False,
)

fig, ax = plt.subplots(figsize=(5, 4.5))

ConfusionMatrixDisplay(
    cm,
    display_labels=["Human", "AI"],
).plot(
    ax=ax,
    cmap="Blues",
    colorbar=False,
    values_format="d",
)

ax.set_title(
    f"Validation confusion matrix "
    f"(threshold = {ERROR_THRESHOLD:g})"
)

save_analysis_figure(fig, "validation_confusion_matrix")
plt.show()

print(
    "Review policy: five highest-P(AI) false positives "
    "and five lowest-P(AI) false negatives."
)
print(
    "These are confident mistakes, not a representative "
    "random sample of all errors."
)

### 7.4 Analyze Five False Positives


In [ ]:
def sample_error_cases(error_type, n=5, seed=SEED):
    """Randomly sample actual errors from the fixed validation set."""
    if error_type == "FP":
        mask = (
            analysis_val["label"].eq(0)
            & analysis_val["prob_ai"].ge(ERROR_THRESHOLD)
        )
    elif error_type == "FN":
        mask = (
            analysis_val["label"].eq(1)
            & analysis_val["prob_ai"].lt(ERROR_THRESHOLD)
        )
    else:
        raise ValueError("error_type must be 'FP' or 'FN'.")

    # Stable ordering makes sampling reproducible.
    candidates = analysis_val.loc[mask].sort_values("id")

    if len(candidates) < n:
        raise ValueError(
            f"Only {len(candidates)} {error_type} cases are "
            f"available; cannot sample {n} unique cases."
        )

    sampled = candidates.sample(
        n=n,
        replace=False,
        random_state=seed,
    ).copy()

    sampled["error_type"] = error_type
    sampled["pred_label"] = (
        sampled["prob_ai"] >= ERROR_THRESHOLD
    ).astype(int)

    sampled = sampled[[
        "id",
        "error_type",
        "label",
        "pred_label",
        "prob_ai",
        "words",
        "text",
    ]].reset_index(drop=True)

    print(
        f"{error_type}: sampled {n} of "
        f"{len(candidates)} cases; seed={seed}"
    )

    display(
        sampled.drop(columns="text").round(6)
    )

    # Show the actual text for manual inspection.
    for case in sampled.itertuples(index=False):
        print(
            f"\nID: {case.id} | {case.error_type} | "
            f"True label: {case.label} | "
            f"Predicted label: {case.pred_label} | "
            f"P(AI): {case.prob_ai:.6f}"
        )
        print(f"Words: {case.words}")
        print("Text:")
        print(case.text)

    sampled.to_csv(
        ERROR_DIR / f"{error_type.lower()}_review.csv",
        index=False,
    )

    return sampled


fp_review = sample_error_cases(
    error_type="FP",
    n=5,
    seed=SEED,
)

### 7.5 Analyze Five False Negatives


In [ ]:
fn_review = sample_error_cases(
    error_type="FN",
    n=5,
    seed=SEED,
)

error_review = pd.concat(
    [fp_review, fn_review],
    ignore_index=True,
)

assert len(error_review) == 10
assert error_review["id"].is_unique

error_review.to_csv(
    ERROR_DIR / "ten_error_cases.csv",
    index=False,
)

### 7.6 Identify Recurring Failure Patterns and Potential Remedies


In [ ]:
length_error_rows = []

for group in LENGTH_GROUPS:
    subset = analysis_val.loc[
        analysis_val["length_group"].eq(group)
    ]

    human_mask = subset["label"].eq(0)
    ai_mask = subset["label"].eq(1)
    predicted_ai = subset["prob_ai"].ge(ERROR_THRESHOLD)

    human_count = int(human_mask.sum())
    ai_count = int(ai_mask.sum())

    fp_count = int((human_mask & predicted_ai).sum())
    fn_count = int((ai_mask & ~predicted_ai).sum())

    length_error_rows.append({
        "Length group": group,
        "Rows": len(subset),
        "Human": human_count,
        "AI": ai_count,
        "False positives": fp_count,
        "False negatives": fn_count,
        "False positive rate": (
            fp_count / human_count
            if human_count else np.nan
        ),
        "False negative rate": (
            fn_count / ai_count
            if ai_count else np.nan
        ),
    })

length_error_results = pd.DataFrame(length_error_rows)

display(length_error_results.round(4))

length_error_results.to_csv(
    ERROR_DIR / "length_group_error_rates.csv",
    index=False,
)

sampling_metadata = {
    "selection_method": (
        "Random sampling without replacement within "
        "each error type, after sorting by ID"
    ),
    "seed": SEED,
    "threshold": ERROR_THRESHOLD,
    "false_positive_ids": fp_review["id"].tolist(),
    "false_negative_ids": fn_review["id"].tolist(),
}

(ERROR_DIR / "error_sampling_metadata.json").write_text(
    json.dumps(sampling_metadata, indent=2),
    encoding="utf-8",
)

print(
    "Total false positives:",
    int(length_error_results["False positives"].sum()),
)
print(
    "Total false negatives:",
    int(length_error_results["False negatives"].sum()),
)
print("Results saved to:", ERROR_DIR.resolve())

## 8. Setup, Artifact Registry, and Frozen Feature Caches

**Confirmed public AUROC: 0.98816 (user-reported).**
Best prediction = **50% of the 0.98677 ensemble + 50% embedding/statistics joint tree**.
The new tree uses the existing 122 features plus the first 16 components of a fold-trained
32-component PCA on row-normalized 768-D GPT-2 embeddings.

Normal restart: run Sections 8–10 with all training switches False. This restores
saved comparisons and exports the exact scored CSV. No model download, language-model
extraction, PCA fit or tree training runs by default. Sections 1–7 are preserved.


### 8.1 Portable Paths and Explicit Execution Modes


In [ ]:
from pathlib import Path
import hashlib
import json
import sys
import numpy as np
import pandas as pd
from IPython.display import display

# Normal use: leave these False. No GPT-2 extraction or model search is needed.
REBUILD_COMPONENTS = False  # Rebuild/restore the older sparse, GPT-2 and raw branches.
RUN_JOINT_SEARCH = False    # Repeat the cached 5-C, 5-fold joint-statistics experiment.
REFIT_JOINT_MODEL = False   # Refit the joint classifier on all 10,000 labels.
RUN_TREE_RESEARCH = False   # 20 fixed-iteration tree CV fits; requires saved GPT-2 features.
RUN_WEIGHT_COMPARISON = False  # Reuse OOF predictions; no training.
RUN_AVERAGING = False       # Optional model-average comparison, possibly followed by refits.
RUN_PYTHIA_RESEARCH = False  # Optional extraction, CV, validation and final refit.
RUN_THREE_BRANCH_SEARCH = False  # Prediction-only historical comparison.
RUN_ALL_FEATURES_RESEARCH = False  # Optional 20 tree CV fits and candidate export.
RUN_EMBEDDING_RESEARCH = False  # Optional PCA/tree CV, validation and final refit.
RUN_DIAGNOSTICS = False  # Optional actual-error analysis for the current 0.98816 ensemble.
DOWNLOAD_BEST = False      # Set True to download the verified best CSV in Colab.
configure_paths()
print('Data:', DATA_DIR)
print('Saved experiments:', OUTPUT_DIR)
print('Modes:', dict(rebuild=REBUILD_COMPONENTS, search=RUN_JOINT_SEARCH,
                     refit=REFIT_JOINT_MODEL, download=DOWNLOAD_BEST))

print('Additional modes:',dict(embedding_research=RUN_EMBEDDING_RESEARCH,diagnostics=RUN_DIAGNOSTICS))


### 8.2 Verify the Scored File and Prediction Lineage

`best` means the exact user-confirmed 0.98816 CSV. Historical `experimental` and
`selected` names and CV decisions remain unchanged. This candidate improved four folds
and fixed validation, but did not meet the old 0.0005 conservative mean-CV gate.


In [ ]:
BASELINE_DIR=OUTPUT_DIR/'frozen_lm_lr/8b1f60e51c7f454a'
EMBEDDING_DIR=OUTPUT_DIR/'gpt2_embedding_lr/8d87d6e871124fda'
RAW_BASELINE_DIR=OUTPUT_DIR/'raw_gpt2_followup/9259247aa4f3e359'
CONFIRMED_JOINT_DIR=OUTPUT_DIR/'joint_gpt2_statistics/2d669162b704d735'
TREE_SOURCE_DIR=OUTPUT_DIR/'structure_catboost/404ce0b7b71319b1'
WEIGHT_SOURCE_DIR=OUTPUT_DIR/'tree_weight_extension_097988/2e838e8ef8a0d53c'
PYTHIA_FEATURE_DIR=OUTPUT_DIR/'second_lm/c496bdce6ff4eb8a'
PYTHIA_MODEL_DIR=PYTHIA_FEATURE_DIR/'classifiers/1963be0721ebc96f'
ALL_FEATURES_DIR=OUTPUT_DIR/'all_features_catboost/b28c916d42a794e2'
LEGACY_BEST_DIR=OUTPUT_DIR/'best_096875'
LEGACY_BEST_SUBMISSION_PATH=LEGACY_BEST_DIR/'submission_best_096875.csv'
BEST_DIR=OUTPUT_DIR/'best_098677'
BEST_SOURCE_PATH=ALL_FEATURES_DIR/'submission_all_features_experimental.csv'

def read_json(path):return json.loads(Path(path).read_text(encoding='utf-8'))
def sha256_file(path):return hashlib.sha256(Path(path).read_bytes()).hexdigest()

required=[BASELINE_DIR/'config.json',ALL_FEATURES_DIR/'config.json',
          ALL_FEATURES_DIR/'selection.json',ALL_FEATURES_DIR/'submission_manifest.json',BEST_SOURCE_PATH]
missing=[str(p) for p in required if not p.is_file()]
if missing:
    raise FileNotFoundError('Saved experiment artifacts missing. Copy the outputs folders from Drive '                            'to the local project or check PROJECT_OVERRIDE.\n'+'\n'.join(missing))
base_config=read_json(BASELINE_DIR/'config.json')
for filename,key in [('train.csv','train_sha256'),('test.csv','test_sha256')]:
    assert sha256_file(DATA_DIR/filename)==base_config[key],f'Changed data: {filename}'
record=read_json(ALL_FEATURES_DIR/'submission_manifest.json')
selection=read_json(ALL_FEATURES_DIR/'selection.json')
assert selection['spec']==dict(depth=6,l2_leaf_reg=10.) and selection['experimental_weight']==.5
assert sha256_file(BEST_SOURCE_PATH)==record['sha256']
previous=PYTHIA_MODEL_DIR/'submission_pythia_experimental.csv'
previous_record=read_json(PYTHIA_MODEL_DIR/'submission_manifest.json')
assert sha256_file(previous)==previous_record['sha256']
assert sha256_file(previous)==read_json(ALL_FEATURES_DIR/'config.json')['control_sha']
COMPONENT_BASELINE=dict(sha256=read_json(LEGACY_BEST_DIR/'manifest.json')['sha256'],public_auc=.96875)
CONFIRMED_BEST=dict(public_auc=.98677,score_source='user-reported Kaggle public leaderboard',
    source_file=str(BEST_SOURCE_PATH),sha256=record['sha256'],original_variant='experimental',
    previous_public_auc=.98431,previous_weight=.5,all_feature_tree_weight=.5,
    tree_parameters=selection['spec'],iterations=600,learning_rate=.04,
    historical_cv_gate=selection['passes_cv_gate'],feature_count=122)



PREVIOUS_BEST=CONFIRMED_BEST.copy()
PREVIOUS_BEST_SOURCE_PATH=BEST_SOURCE_PATH
EMBEDDING_JOINT_DIR=OUTPUT_DIR/'embedding_statistics_joint/172ad2e61031c035'
BEST_DIR=OUTPUT_DIR/'best_098816'
BEST_SOURCE_PATH=EMBEDDING_JOINT_DIR/'submission_embedding_statistics_experimental.csv'
embedding_record=read_json(EMBEDDING_JOINT_DIR/'submission_manifest.json')
embedding_selection=read_json(EMBEDDING_JOINT_DIR/'selection.json')
embedding_config=read_json(EMBEDDING_JOINT_DIR/'config.json')
assert embedding_selection['spec']==dict(pca_components=16,depth=6)
assert embedding_selection['experimental_weight']==.5
assert embedding_config['control_sha']==PREVIOUS_BEST['sha256']
assert sha256_file(BEST_SOURCE_PATH)==embedding_record['sha256']
CONFIRMED_BEST=dict(public_auc=.98816,score_source='user-reported Kaggle public leaderboard',
    source_file=str(BEST_SOURCE_PATH),sha256=embedding_record['sha256'],original_variant='experimental',
    previous_public_auc=.98677,previous_weight=.5,embedding_joint_weight=.5,
    statistical_feature_count=122,pca_components_fitted=32,pca_components_used=16,
    embedding_normalization='row_l2',pca_fit='training_rows_only',
    tree_parameters=dict(depth=6,l2_leaf_reg=10.),iterations=600,learning_rate=.04,
    historical_cv_gate=embedding_selection['passes_cv_gate'])
display(pd.DataFrame([CONFIRMED_BEST]))
print('Verified BEST source:',BEST_SOURCE_PATH)


### 8.3 Architecture and Rebuild Guide

| Branch | Inputs | Classifier |
|---|---|---|
| Historical text/statistics ensemble | Sparse text, GPT-2 statistics and embeddings | Logistic regression ensemble |
| Structural tree | GPT-2 statistics, layout and repetition | CatBoost |
| Dual-language-model tree | GPT-2 and Pythia probability/rank statistics | CatBoost |
| All-feature tree | 122 statistical and structural features | CatBoost |
| Embedding/statistics tree | Same 122 features + 16 PCA components | CatBoost |

Current best: **50% previous 0.98677 ensemble + 50% embedding/statistics tree**.
Expanded main-branch weights: **8% historical 0.96875 ensemble, 12% structural tree,
5% dual-LM tree, 25% all-feature tree, 50% embedding/statistics tree**.

Only missing language-model features require GPU inference. Cached-vector PCA and
classification run on CPU. Optional rebuild cells below are disabled by default.


### 8.4 Optional Component Rebuild: Dependencies, Data Split, and Helpers


In [ ]:
if REBUILD_COMPONENTS:
    # Original component step (78)
    import importlib.util
    from importlib.metadata import version, PackageNotFoundError
    import subprocess
    import sys

    required = {"transformers": "4.57.1"}
    for package, wanted in required.items():
        try:
            installed = version(package)
        except PackageNotFoundError:
            installed = None
        if installed != wanted:
            if package in sys.modules:
                raise RuntimeError(f"Restart this runtime, then run Section 8 first. Required {package}=={wanted}.")
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", f"{package}=={wanted}"])
    for package in ["torch", "tokenizers", "safetensors"]:
        if importlib.util.find_spec(package) is None:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

    import importlib.util
    import subprocess
    import sys
    if importlib.util.find_spec("tokenizers") is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "tokenizers"])
    """Training-only five-fold research on sparse text and observable style features.

    Run as a script locally. Import functions in Colab after making this file available.
    No test-set labels, external labeled data, or pretrained encoders are used.
    """
    import argparse
    from collections import Counter
    import gc
    import hashlib
    from importlib.metadata import version
    import json
    from pathlib import Path
    import re
    from time import perf_counter
    import warnings

    import joblib
    import numpy as np
    import pandas as pd
    from scipy.sparse import csr_matrix, hstack
    from sklearn.exceptions import ConvergenceWarning
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import accuracy_score, log_loss, roc_auc_score
    from sklearn.model_selection import StratifiedKFold
    from sklearn.preprocessing import StandardScaler, normalize
    from threadpoolctl import threadpool_limits

    SEED = 42
    C_VALUES = [0.1, 1.0, 10.0, 30.0, 100.0]
    WORD = dict(lowercase=True, strip_accents=None, stop_words=None,
                ngram_range=(1, 2), min_df=2, max_df=1.0, max_features=50_000,
                sublinear_tf=True, norm="l2", dtype=np.float32)
    CHAR = dict(analyzer="char", ngram_range=(3, 5), lowercase=False,
                min_df=2, max_df=1.0, max_features=200_000,
                sublinear_tf=True, norm="l2", dtype=np.float32)
    PUNCTUATION = [",", ".", ";", ":", "!", "?", "-", "—", "–", "'", '"',
                   "’", "“", "”", "(", ")", "[", "]", "\n", "\t"]
    STYLE_NAMES = ["log_characters", "log_words", "word_length_mean", "word_length_sd",
                   "word_length_p90", "type_token_ratio", "hapax_ratio", "repeated_bigram_ratio",
                   "repeated_trigram_ratio", "log_sentence_segments", "segment_words_mean",
                   "segment_words_sd", "segment_words_cv", "uppercase_ratio", "digit_ratio",
                   "whitespace_ratio", "non_ascii_ratio"] + [f"frequency_{p!r}" for p in PUNCTUATION]


    """Five-fold target-like development of format-invariant sparse detectors.

    Target text supplies only source-membership diagnostics; no target authorship labels
    are accessed. Folds remain balanced by training authorship labels. These stress
    scores do not estimate an unbiased leaderboard score.
    """
    import argparse
    from collections import defaultdict
    import gc
    import hashlib
    from importlib.metadata import version
    import json
    from pathlib import Path
    from time import perf_counter
    import unicodedata

    import joblib
    import numpy as np
    import pandas as pd
    from scipy.sparse import csr_matrix, hstack
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.metrics import accuracy_score, log_loss, roc_auc_score
    from sklearn.preprocessing import normalize, StandardScaler
    from tokenizers import Tokenizer, models, pre_tokenizers, trainers


    VERSION='format_invariant_sparse_v1'
    CS=[0.1,1.0,10.0,30.0,100.0]
    BPE_CONFIG=dict(vocab_size=4000,min_frequency=2)
    TRANSLATION=str.maketrans({'’':"'",'‘':"'",'“':'"','”':'"','—':'-','–':'-','…':'...'})


    """Controlled follow-up to the user-reported 0.90551 format-robust submission.

    Select a limited addition to the incumbent using the same five source-likeness
    folds. No held-out authorship labels enter any fitted feature operation.
    """
    import argparse
    import gc
    import hashlib
    from importlib.metadata import version
    import json
    from pathlib import Path
    import shutil
    from time import perf_counter
    import warnings

    import joblib
    import numpy as np
    import pandas as pd
    from scipy.sparse import csr_matrix,hstack
    from sklearn.exceptions import ConvergenceWarning
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import roc_auc_score,log_loss,accuracy_score
    from sklearn.preprocessing import normalize,StandardScaler,PolynomialFeatures
    from threadpoolctl import threadpool_limits
    from tokenizers import Tokenizer,models,pre_tokenizers,trainers


    EXPERIMENT_VERSION='format_robust_increment_v2'
    CS=[10.0,30.0,100.0,300.0,1000.0]
    INCUMBENT=[dict(family='clean_nbstyle_0.3',C=100.0),dict(family='bpe_nb',C=100.0)]
    FAMILIES=['clean_nbstyle_0.3','bpe_nb','clean_poly_0.1','clean_weighted_0.5',
              'bpe_weighted_0.5','bpe_long_4000','bpe_long_8000']


    """Paired, five-fold feature-dropout experiment for the existing sparse LR route.

    One clean view and one fixed corrupted view share each original row's weight.
    This is Monte Carlo input augmentation, not per-iteration neural-network dropout.
    """
    import argparse
    import gc
    import hashlib
    from importlib.metadata import version
    import json
    from pathlib import Path
    from time import perf_counter

    import joblib
    import numpy as np
    import pandas as pd
    from scipy.sparse import vstack
    from sklearn.metrics import roc_auc_score, log_loss, accuracy_score


    DROPOUT_VERSION = 'sparse_lr_dropout_v1'
    DROP_RATES = [0.0, 0.02, 0.05, 0.10]
    STRESS_RATES = [0.0, 0.05, 0.10, 0.20]
    DROP_FAMILIES = ['clean_nbstyle_0.3', 'clean_poly_0.1', 'bpe_nb']
    DROP_C = 100.0
    DROP_SEED = 42
    BLENDS = {
        'public_route': {'clean_nbstyle_0.3': 0.5, 'bpe_nb': 0.5},
        'v2_route': {'clean_nbstyle_0.3': 0.25, 'clean_poly_0.1': 0.5, 'bpe_nb': 0.25},
    }
    MIN_MEAN_GAIN = 0.0005  # A practical adoption gate, not a significance threshold.

    import torch
    from transformers import AutoConfig
    from IPython.display import display

    S8_MODEL_NAME = "openai-community/gpt2"
    S8_CONTEXT, S8_STRIDE, S8_BATCH_SIZE = 512, 256, 4
    S8_C_VALUES = [10.0]
    S8_BLEND_WEIGHTS = [0.0, 0.4]
    S8_FEATURE_BUDGET_MINUTES = 12.0  # Extraction only; download/CV/refit are extra.
    S8_ALLOW_LONG_EXTRACTION = False
    S8_SAVE_PRETRAINED_WEIGHTS = True  # Save the unchanged weights with the final models.
    DROP_RATES = [0.10]  # Frozen choice; do not rerun the old four-rate search.
    S8_SPARSE_WEIGHTS = BLENDS["v2_route"].copy()
    S8_DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Feature-extraction device:", S8_DEVICE)
    if S8_DEVICE.type == "cuda":
        print(torch.cuda.get_device_name(0))
    else:
        print("CPU is supported. On Colab, select a GPU before the benchmark for practical speed.")
    print("Neural fine-tuning fits: 0; fixed sparse CV fits: 15; statistical LR fits: 5; embedding LR fits: 5.")

    from sklearn.pipeline import make_pipeline
    from sklearn.preprocessing import Normalizer
    FORCE_BASELINE_REBUILD = False
    FORCE_EMBEDDING_REBUILD = False
    REFIT_FINAL_MODELS = False
    DOWNLOAD_SUBMISSION = False
    EMBEDDING_C = 0.1
    FINAL_EMBEDDING_WEIGHT = 0.2

    # Original component step (80)
    import matplotlib.pyplot as plt
    from IPython.display import display
    from sklearn.model_selection import train_test_split

    configure_paths()
    started = perf_counter()
    train_df = pd.read_csv(DATA_DIR / "train.csv")
    assert len(train_df) == 10000 and train_df.id.is_unique and train_df.id.notna().all()
    assert set(train_df.label.unique()) == {0, 1} and train_df.text.map(lambda x: isinstance(x, str)).all()
    data_sha = hashlib.sha256((DATA_DIR / "train.csv").read_bytes()).hexdigest()
    split_path, meta_path = OUTPUT_DIR / "split_assignments.csv", OUTPUT_DIR / "split_metadata.json"
    expected = dict(seed=SEED, validation_size=2000, train_csv_sha256=data_sha)
    assert split_path.exists() == meta_path.exists(), "Incomplete split files; review them before proceeding."
    if split_path.is_file():
        metadata = json.loads(meta_path.read_text())
        assert all(metadata.get(k) == v for k, v in expected.items()), "Saved split does not match these data/settings."
        assignments = pd.read_csv(split_path, dtype={"id": train_df.id.dtype})
    else:
        fit_ids, val_ids = train_test_split(train_df.id.to_numpy(), test_size=2000,
                                          stratify=train_df.label.to_numpy(), random_state=SEED)
        assignments = pd.concat([pd.DataFrame({"id": fit_ids, "split": "train"}),
                                 pd.DataFrame({"id": val_ids, "split": "validation"})], ignore_index=True)
        assignments.to_csv(split_path, index=False)
        meta_path.write_text(json.dumps(expected, indent=2), encoding="utf-8")
    assert assignments.id.is_unique and set(assignments.id) == set(train_df.id)
    assert set(assignments.split) == {"train", "validation"}
    indexed = train_df.set_index("id", drop=False)
    train_split = indexed.loc[assignments.loc[assignments.split.eq("train"), "id"]].reset_index(drop=True)
    val_split = indexed.loc[assignments.loc[assignments.split.eq("validation"), "id"]].reset_index(drop=True)
    assert len(train_split) == 8000 and len(val_split) == 2000
    assert set(train_split.id).isdisjoint(val_split.id)
    normalize_text = lambda s: " ".join(s.lower().split())
    assert set(train_split.text.map(normalize_text)).isdisjoint(val_split.text.map(normalize_text))
    print("Data:", DATA_DIR)
    print("Training / validation:", len(train_split), "/", len(val_split))

    test_df = pd.read_csv(DATA_DIR / "test.csv")
    sample_submission_df = pd.read_csv(DATA_DIR / "sample_submission.csv")
    assert test_df.id.is_unique and sample_submission_df.id.is_unique
    assert test_df.text.map(lambda value: isinstance(value, str) and bool(value.strip())).all()
    assert len(test_df) == len(sample_submission_df) == 3000
    assert set(test_df.id) == set(sample_submission_df.id)
    assert sample_submission_df.columns.tolist() == ["id", "label"]
    print("Test rows:", len(test_df), "| All authorship labels are hidden.")

    # Original component step (82)
    def style_features(texts):
        rows = []
        for text in texts:
            n = max(len(text), 1)
            words = re.findall(r"\b[\w']+\b", text.lower())
            counts = Counter(words)
            lengths = np.array([len(w) for w in words] or [0], dtype=float)
            segments = [s for s in re.split(r"[.!?]+", text) if s.strip()]
            segment_lengths = np.array([len(re.findall(r"\b[\w']+\b", s)) for s in segments] or [0], dtype=float)
            repeats = []
            for k in (2, 3):
                grams = list(zip(*(words[i:] for i in range(k))))
                repeats.append(1 - len(set(grams)) / len(grams) if grams else 0.0)
            rows.append([
                np.log1p(len(text)), np.log1p(len(words)), lengths.mean(), lengths.std(),
                np.percentile(lengths, 90), len(counts) / max(len(words), 1),
                sum(v == 1 for v in counts.values()) / max(len(words), 1), *repeats,
                np.log1p(len(segments)), segment_lengths.mean(), segment_lengths.std(),
                segment_lengths.std() / max(segment_lengths.mean(), 1.0),
                sum(c.isupper() for c in text) / n, sum(c.isdigit() for c in text) / n,
                sum(c.isspace() for c in text) / n, sum(ord(c) > 127 for c in text) / n,
                *[text.count(c) / n for c in PUNCTUATION],
            ])
        result = np.asarray(rows, dtype=np.float32)
        assert result.shape == (len(texts), len(STYLE_NAMES)) and np.isfinite(result).all()
        return result


    def canonical(value):
        return json.dumps(value, sort_keys=True, default=str)


    def key(spec):
        return hashlib.sha256(canonical(spec).encode()).hexdigest()[:16]


    def check_prob(p, n):
        p = np.asarray(p, dtype=float)
        assert p.shape == (n,) and np.isfinite(p).all() and ((p >= 0) & (p <= 1)).all()
        return p


    def binary_matrix(x):
        result = x.copy()
        result.data.fill(1.0)
        return result


    def nb_ratio(binary_x, y, alpha=1.0):
        positive = np.asarray(binary_x[y == 1].sum(axis=0)).ravel().astype(float) + alpha
        negative = np.asarray(binary_x[y == 0].sum(axis=0)).ravel().astype(float) + alpha
        return np.log((positive / positive.sum()) / (negative / negative.sum())).astype(np.float32)

    def clean_text(text):
        text=unicodedata.normalize('NFKC',text).translate(TRANSLATION).casefold()
        text=''.join(c for c in text if unicodedata.category(c)!='Cf')
        return ' '.join(text.split())


    def encode(tokenizer,texts):
        return [' '.join('t'+str(t) for t in e.ids) for e in tokenizer.encode_batch(list(texts))]


    def make_folds(fit,source):
        """Contiguous target-likeness quintiles per class, not random folds."""
        table=fit[['id','label']].copy()
        source=source.set_index('id')
        table['target_likeness']=(table.id.map(source.style_nonlinear)+table.id.map(source.word_text))/2
        assert table.target_likeness.notna().all()
        table['fold']=-1
        for label in [0,1]:
            ordered=table[table.label.eq(label)].sort_values(['target_likeness','id']).index.to_numpy()
            for fold,indices in enumerate(np.array_split(ordered,5),1):table.loc[indices,'fold']=fold
        folds=[(np.flatnonzero(table.fold.to_numpy()!=f),np.flatnonzero(table.fold.to_numpy()==f)) for f in range(1,6)]
        return table,folds

    def row_weights(y,likeness,power=.5):
        # Tempered, clipped density-ratio proxy. Source probabilities need not be
        # calibrated density ratios; this is an empirical covariate-shift experiment.
        p=np.clip(np.asarray(likeness,dtype=float),.01,.99)
        weights=np.clip((p/(1-p)*(10000/3000))**power,.25,4.0)
        for label in [0,1]:weights[y==label]/=weights[y==label].mean()
        assert np.isfinite(weights).all() and np.all(weights>0)
        return weights


    def train_classifier(x,y,c,weights=None):
        model=LogisticRegression(C=c,solver='liblinear',max_iter=5000,tol=1e-4,random_state=42)
        with threadpool_limits(limits=2),warnings.catch_warnings():
            warnings.simplefilter('error',ConvergenceWarning)
            model.fit(x,y,sample_weight=weights)
        assert np.array_equal(model.classes_,[0,1])
        return model


    def style_poly(bundle,raw_style):
        z=np.clip(bundle['scaler'].transform(raw_style),-8,8)
        x=bundle['poly'].transform(z)
        return np.clip(bundle['poly_scaler'].transform(x),-8,8).astype(np.float32)


    def matrices(fit_text,hold_text,y,likeness,families):
        clean_fit=[clean_text(t) for t in fit_text];clean_hold=[clean_text(t) for t in hold_text]
        weights=row_weights(y,likeness)
        if any(f.startswith('clean_') for f in families):
            word=TfidfVectorizer(**WORD);char=TfidfVectorizer(**CHAR)
            wf=word.fit_transform(clean_fit);wh=word.transform(clean_hold)
            cf=char.fit_transform(clean_fit);ch=char.transform(clean_hold)
            xf=hstack([wf*.5,cf],format='csr');xh=hstack([wh*.5,ch],format='csr')
            bf=binary_matrix(xf);bh=binary_matrix(xh)
            ratio=nb_ratio(bf,y);weighted_ratio=nb_ratio(bf.multiply(weights[:,None]).tocsr(),y)
            nf=normalize(bf.multiply(ratio)).tocsr();nh=normalize(bh.multiply(ratio)).tocsr()
            raw_sf=style_features(clean_fit);raw_sh=style_features(clean_hold)
            scaler=StandardScaler().fit(raw_sf);sf=scaler.transform(raw_sf);sh=scaler.transform(raw_sh)
            base=dict(word=word,char=char,ratio=ratio,scaler=scaler,preprocessing=VERSION)
            for family in families:
                if family=='clean_nbstyle_0.3':
                    a=hstack([nf,csr_matrix(sf*.3)],format='csr');b=hstack([nh,csr_matrix(sh*.3)],format='csr');bundle=base;sw=None
                elif family=='clean_weighted_0.5':
                    a=normalize(bf.multiply(weighted_ratio)).tocsr();b=normalize(bh.multiply(weighted_ratio)).tocsr()
                    a=hstack([a,csr_matrix(sf*.3)],format='csr');b=hstack([b,csr_matrix(sh*.3)],format='csr')
                    bundle={**base,'ratio':weighted_ratio};sw=weights
                elif family=='clean_poly_0.1':
                    poly=PolynomialFeatures(degree=2,include_bias=False)
                    expanded=poly.fit_transform(np.clip(sf,-8,8));ps=StandardScaler().fit(expanded)
                    bundle={**base,'poly':poly,'poly_scaler':ps}
                    a=hstack([nf,csr_matrix(style_poly(bundle,raw_sf)*.1)],format='csr')
                    b=hstack([nh,csr_matrix(style_poly(bundle,raw_sh)*.1)],format='csr');sw=None
                else:continue
                yield family,a,b,bundle,sw
            del word,char,wf,wh,cf,ch,xf,xh,bf,bh,nf,nh,raw_sf,raw_sh,sf,sh,base,bundle,a,b
            gc.collect()
        requested=[f for f in families if f.startswith('bpe_')]
        for vocab in [4000,8000]:
            group=[f for f in requested if (8000 if f=='bpe_long_8000' else 4000)==vocab]
            if not group:continue
            tokenizer=Tokenizer(models.BPE(unk_token='[UNK]'));tokenizer.pre_tokenizer=pre_tokenizers.ByteLevel(add_prefix_space=True)
            trainer=trainers.BpeTrainer(vocab_size=vocab,min_frequency=2,special_tokens=['[UNK]'],
                      initial_alphabet=pre_tokenizers.ByteLevel.alphabet(),show_progress=False)
            tokenizer.train_from_iterator(clean_fit,trainer=trainer)
            encoded_fit=encode(tokenizer,clean_fit);encoded_hold=encode(tokenizer,clean_hold)
            for gram in [(1,3),(2,5)]:
                gram_group=[f for f in group if ((2,5) if f.startswith('bpe_long_') else (1,3))==gram]
                if not gram_group:continue
                tfidf=TfidfVectorizer(ngram_range=gram,min_df=2,max_features=250000 if gram==(1,3) else 300000,
                            lowercase=False,token_pattern=r'(?u)\bt\d+\b',sublinear_tf=True,dtype=np.float32)
                xf=tfidf.fit_transform(encoded_fit);xh=tfidf.transform(encoded_hold)
                bf=binary_matrix(xf);bh=binary_matrix(xh)
                ratio=nb_ratio(bf,y)
                base=dict(tokenizer_json=tokenizer.to_str(),vectorizer=tfidf,ratio=ratio,preprocessing=VERSION)
                for family in gram_group:
                    sw=weights if family=='bpe_weighted_0.5' else None
                    used_ratio=nb_ratio(bf.multiply(weights[:,None]).tocsr(),y) if sw is not None else ratio
                    a=normalize(bf.multiply(used_ratio)).tocsr();b=normalize(bh.multiply(used_ratio)).tocsr()
                    yield family,a,b,{**base,'ratio':used_ratio},sw
                del tfidf,xf,xh,bf,bh,a,b,base
                gc.collect()

    def dropout_spec(family, rate):
        return dict(family=family, C=DROP_C, dropout=float(rate))


    def dropout_seed(fold, family, offset=0):
        return DROP_SEED + 1000 * int(fold) + 17 * DROP_FAMILIES.index(family) + offset


    def feature_dropout(x, text_columns, rate, seed):
        """Independent entry dropout on the text block; keep all style columns.

        Inverted scaling preserves each feature's expected value. Do not renormalize
        rows afterward. The same seed gives nested masks for different rates.
        """
        if not 0 <= rate < 1:
            raise ValueError('Dropout rate must be in [0, 1).')
        if not 0 < text_columns <= x.shape[1]:
            raise ValueError('Invalid text feature boundary.')
        result = x.copy().tocsr()
        if rate == 0:
            return result
        eligible = result.indices < text_columns
        positions = np.flatnonzero(eligible)
        rng = np.random.default_rng(seed)
        keep = rng.random(len(positions), dtype=np.float32) >= rate
        result.data[positions] *= keep / (1.0 - rate)
        result.eliminate_zeros()
        assert np.isfinite(result.data).all()
        return result


    def fit_dropout(x, y, text_columns, rate, seed):
        started = perf_counter()
        if rate == 0:
            model = train_classifier(x, y, DROP_C)
            augmented_loss = log_loss(y, model.predict_proba(x)[:, 1])
        else:
            corrupted = feature_dropout(x, text_columns, rate, seed)
            augmented = vstack([x, corrupted], format='csr')
            targets = np.concatenate([y, y])
            # Sum of weights stays equal to the number of original training rows.
            weights = np.full(len(targets), 0.5)
            model = train_classifier(augmented, targets, DROP_C, weights)
            augmented_loss = log_loss(targets, model.predict_proba(augmented)[:, 1], sample_weight=weights)
            del augmented, corrupted, targets, weights
        diagnostics = dict(
            training_seconds=perf_counter() - started,
            clean_train_log_loss=float(log_loss(y, model.predict_proba(x)[:, 1])),
            augmented_train_log_loss=float(augmented_loss),
            solver_iterations=int(np.max(model.n_iter_)),
            original_training_rows=len(y),
            effective_sample_weight=float(len(y)),
        )
        return model, diagnostics


    def feature_matrix(bundle, texts):
        """Uncorrupted inference features; identical to the preceding v2 route."""
        clean = [clean_text(t) for t in texts]
        family = bundle['spec']['family']
        if family == 'bpe_nb':
            tokenizer = Tokenizer.from_str(bundle['tokenizer_json'])
            x = bundle['vectorizer'].transform(encode(tokenizer, clean))
            return normalize(binary_matrix(x).multiply(bundle['ratio'])).tocsr()
        x = hstack([bundle['word'].transform(clean) * .5,
                      bundle['char'].transform(clean)], format='csr')
        x = normalize(binary_matrix(x).multiply(bundle['ratio'])).tocsr()
        raw_style = style_features(clean)
        if family == 'clean_poly_0.1':
            s = style_poly(bundle, raw_style) * .1
        else:
            s = bundle['scaler'].transform(raw_style) * .3
        return hstack([x, csr_matrix(s)], format='csr')


    def predict_dropout(bundle, texts):
        x = feature_matrix(bundle, texts)
        return check_prob(bundle['classifier'].predict_proba(x)[:, 1], len(texts))


    def search_dropout(fit, fold_table, folds, out, max_folds=None):
        folder = out / 'cv'
        folder.mkdir(exist_ok=True)
        specs = [dropout_spec(f, p) for f in DROP_FAMILIES for p in DROP_RATES]
        y = fit.label.to_numpy()
        rows = []
        oof = {key(s): np.full(len(fit), np.nan) for s in specs}
        for fold, (idx, hold) in enumerate(folds, 1):
            if max_folds is not None and fold > max_folds:
                break
            needed = []
            for spec in specs:
                path = folder / f'{key(spec)}_{fold}.npz'
                if path.is_file():
                    with np.load(path) as saved:
                        assert np.array_equal(saved['fit'], idx) and np.array_equal(saved['hold'], hold)
                        p = check_prob(saved['p'], len(hold))
                        diagnostics = json.loads(saved['diagnostics'].item())
                    oof[key(spec)][hold] = p
                    rows.append(dict(**spec, fold=fold, auc=roc_auc_score(y[hold], p),
                                     holdout_log_loss=log_loss(y[hold], p), **diagnostics))
                else:
                    needed.append(spec)
            if needed:
                families = [f for f in DROP_FAMILIES if any(s['family'] == f for s in needed)]
                feature_started = perf_counter()
                for family, xf, xh, bundle, _ in matrices(
                        fit.iloc[idx].text, fit.iloc[hold].text, y[idx],
                        fold_table.target_likeness.to_numpy()[idx], families):
                    print(f'Features ready: fold={fold}/5, family={family}, shape={xf.shape}, '
                          f'elapsed={perf_counter()-feature_started:.1f}s', flush=True)
                    columns = len(bundle['ratio'])
                    for spec in [s for s in needed if s['family'] == family]:
                        model, diagnostics = fit_dropout(xf, y[idx], columns, spec['dropout'], dropout_seed(fold, family))
                        p = check_prob(model.predict_proba(xh)[:, 1], len(hold))
                        path = folder / f'{key(spec)}_{fold}.npz'
                        np.savez_compressed(path, fit=idx, hold=hold, p=p,
                                            diagnostics=json.dumps(diagnostics))
                        oof[key(spec)][hold] = p
                        row = dict(**spec, fold=fold, auc=roc_auc_score(y[hold], p),
                                   holdout_log_loss=log_loss(y[hold], p), **diagnostics)
                        rows.append(row)
                        pd.DataFrame(rows).to_csv(out / 'model_fold_results.csv', index=False)
                        print(f"Completed fold={fold}/5, {family}, dropout={spec['dropout']:.0%}, "
                              f"AUC={row['auc']:.6f}, loss={row['holdout_log_loss']:.5f}, "
                              f"fit={diagnostics['training_seconds']:.1f}s", flush=True)
                        del model
                    del xf, xh, bundle
                    gc.collect()
            pd.DataFrame(rows).to_csv(out / 'model_fold_results.csv', index=False)
        if max_folds is None:
            for p in oof.values():
                check_prob(p, len(fit))
            fit[['id', 'label']].assign(**oof).to_csv(out / 'oof.csv', index=False)
        return pd.DataFrame(rows), oof


    def fit_dropout_bundles(frame, likeness, rates, out):
        folder = out / f'fitted_{len(frame)}'
        folder.mkdir(exist_ok=True)
        frame_hash = hashlib.sha256(frame.to_json(orient='split', force_ascii=False).encode()).hexdigest()
        bundles = {}
        missing = []
        for family in DROP_FAMILIES:
            for rate in sorted(set(rates)):
                spec = dropout_spec(family, rate)
                path = folder / (key(spec) + '.joblib')
                if path.is_file():
                    bundle = joblib.load(path)
                    assert bundle['training_hash'] == frame_hash and bundle['spec'] == spec
                    bundles[key(spec)] = bundle
                else:
                    missing.append(spec)
        if missing:
            families = [f for f in DROP_FAMILIES if any(s['family'] == f for s in missing)]
            for family, x, _, features, _ in matrices(frame.text, frame.text.iloc[:1],
                                                        frame.label.to_numpy(), likeness, families):
                for spec in [s for s in missing if s['family'] == family]:
                    model, diagnostics = fit_dropout(x, frame.label.to_numpy(), len(features['ratio']),
                                                    spec['dropout'], dropout_seed(len(frame), family))
                    bundle = dict(**features, classifier=model, spec=spec, training_hash=frame_hash,
                                  training_rows=len(frame), diagnostics=diagnostics, dropout_version=DROPOUT_VERSION)
                    # Verify persisted inference applies exactly the fitted representation.
                    direct = model.predict_proba(x[:16])[:, 1]
                    np.testing.assert_allclose(direct, predict_dropout(bundle, frame.text.iloc[:16]), atol=1e-12, rtol=0)
                    path = folder / (key(spec) + '.joblib')
                    temporary = path.with_suffix('.tmp')
                    joblib.dump(bundle, temporary, compress=3)
                    temporary.replace(path)
                    bundles[key(spec)] = bundle
                    print('Refit', len(frame), spec, diagnostics, flush=True)
                del x, features
                gc.collect()
        return bundles

    """Frozen GPT-2 statistics; labels never enter feature extraction."""
    import gc
    import hashlib
    import json
    from pathlib import Path
    from time import perf_counter

    import numpy as np
    import torch

    LM_FEATURE_VERSION = "gpt2_normalized_fulltext_statistics_v1"
    LM_FEATURE_NAMES = [
        "nll_mean", "nll_std", "nll_q10", "nll_q25", "nll_median", "nll_q75", "nll_q90",
        "nll_above_5", "nll_above_10", "nll_adjacent_change", "nll_top_decile_mean",
        "entropy_mean", "entropy_std", "surprisal_gap_mean", "surprisal_gap_std",
        "standardized_gap_mean", "standardized_gap_std", "conditional_discrepancy",
        "quarter_1_nll", "quarter_2_nll", "quarter_3_nll", "quarter_4_nll",
        "last_minus_first_nll", "quarter_nll_std",
    ]


    def lm_windows(token_ids, context=512, stride=256):
        """Yield (window, first target offset); each target is scored exactly once."""
        if not 1 <= stride < context:
            raise ValueError("Require 1 <= stride < context.")
        previous_end = 1  # The explicit EOS context token is never a prediction target.
        for begin in range(0, len(token_ids) - 1, stride):
            end = min(begin + context, len(token_ids))
            if end <= previous_end:
                break
            first_target = max(1, previous_end - begin)
            yield token_ids[begin:end], first_target
            previous_end = end
            if end == len(token_ids):
                break


    def lm_token_statistics(logits, targets, chunk_size=128):
        """Float32 reductions, with vocabulary work chunked to limit peak memory."""
        pieces = []
        for start in range(0, len(targets), chunk_size):
            logp = torch.log_softmax(logits[start:start + chunk_size].float(), dim=-1)
            p = logp.exp()
            observed = -logp.gather(1, targets[start:start + chunk_size, None]).squeeze(1)
            entropy = -(p * logp).sum(-1)
            # Center before squaring to avoid cancellation for near-uniform distributions.
            variance = (p * (logp + entropy[:, None]).square()).sum(-1).clamp_min(1e-8)
            pieces.append(torch.stack([observed, entropy, variance], -1).cpu().numpy())
        return np.concatenate(pieces, axis=0)


    def lm_summarize(values):
        nll, entropy, variance = np.asarray(values, dtype=np.float64).T
        gap = nll - entropy
        z = gap / np.sqrt(variance)
        quarters = [float(x.mean()) if len(x) else float(nll.mean()) for x in np.array_split(nll, 4)]
        q = np.quantile(nll, [.1, .25, .5, .75, .9])
        result = [nll.mean(), nll.std(), *q,
                  np.mean(nll > 5), np.mean(nll > 10),
                  np.abs(np.diff(nll)).mean() if len(nll) > 1 else 0.,
                  np.sort(nll)[-max(1, int(np.ceil(len(nll) * .1))):].mean(),
                  entropy.mean(), entropy.std(), gap.mean(), gap.std(), z.mean(), z.std(),
                  -gap.sum() / np.sqrt(variance.sum()), *quarters,
                  quarters[-1] - quarters[0], np.std(quarters)]
        result = np.asarray(result, dtype=np.float32)
        assert len(result) == len(LM_FEATURE_NAMES) and np.isfinite(result).all()
        return result


    def lm_extract(texts, tokenizer, model, device, context=512, stride=256, batch_size=4):
        """No truncation: sliding windows cover every normalized text token."""
        cleaned = [clean_text(t) for t in texts]
        if any(not t for t in cleaned):
            raise ValueError("Empty normalized text cannot be scored.")
        encoded = tokenizer(cleaned, add_special_tokens=False, truncation=False)["input_ids"]
        sequences = [[tokenizer.eos_token_id] + ids for ids in encoded]
        windows = [(row, ids, first) for row, seq in enumerate(sequences)
                   for ids, first in lm_windows(seq, context, stride)]
        # Sorting reduces padding; per-document results are restored to token order below.
        ordered = sorted(enumerate(windows), key=lambda item: len(item[1][1]))
        outputs = [None] * len(windows)
        model.eval()
        with torch.inference_mode():
            for begin in range(0, len(ordered), batch_size):
                batch = ordered[begin:begin + batch_size]
                length = max(len(w[1]) for _, w in batch)
                ids = torch.full((len(batch), length), tokenizer.eos_token_id, dtype=torch.long, device=device)
                mask = torch.zeros_like(ids)
                for j, (_, (_, seq, _)) in enumerate(batch):
                    ids[j, :len(seq)] = torch.tensor(seq, dtype=torch.long, device=device)
                    mask[j, :len(seq)] = 1
                logits = model(input_ids=ids, attention_mask=mask, use_cache=False).logits
                for j, (window_number, (_, seq, first)) in enumerate(batch):
                    # Logit t predicts token t+1. Right padding and overlap are excluded.
                    outputs[window_number] = lm_token_statistics(
                        logits[j, first - 1:len(seq) - 1], ids[j, first:len(seq)])
                del logits, ids, mask
        values = [[] for _ in sequences]
        for (row, _, _), stats in zip(windows, outputs):
            values[row].append(stats)
        result = []
        for seq, chunks in zip(sequences, values):
            stats = np.concatenate(chunks)
            assert len(stats) == len(seq) - 1, "Missing or repeated target tokens."
            result.append(lm_summarize(stats))
        return np.stack(result), np.asarray([len(s) - 1 for s in sequences])


    def lm_atomic_npz(path, **values):
        temporary = Path(str(path) + ".tmp")
        with temporary.open("wb") as handle:
            np.savez_compressed(handle, **values)
        temporary.replace(path)


    def lm_cached_extract(frame, name, folder, tokenizer, model, device, config, shard_size=128):
        """Fingerprint data + model + numerical settings; resume completed shards."""
        fingerprint = hashlib.sha256(json.dumps(config, sort_keys=True).encode()
                                     + frame[["id", "text"]].to_json(orient="split").encode()).hexdigest()
        cache_dir = Path(folder) / f"{name}_{fingerprint[:16]}"
        cache_dir.mkdir(parents=True, exist_ok=True)
        result, token_counts = [], []
        start_time = perf_counter()
        for start in range(0, len(frame), shard_size):
            stop = min(start + shard_size, len(frame))
            path = cache_dir / f"{start:06d}_{stop:06d}.npz"
            if path.is_file():
                with np.load(path, allow_pickle=False) as saved:
                    assert saved["fingerprint"].item() == fingerprint
                    assert np.array_equal(saved["ids"], frame.id.iloc[start:stop].astype(str).to_numpy())
                    x, count = saved["features"].copy(), saved["token_count"].copy()
            else:
                x, count = lm_extract(frame.text.iloc[start:stop].tolist(), tokenizer, model, device,
                                      context=config["context"], stride=config["stride"],
                                      batch_size=config["batch_size"])
                lm_atomic_npz(path, fingerprint=fingerprint,
                              ids=frame.id.iloc[start:stop].astype(str).to_numpy(dtype=str),
                              features=x, token_count=count)
            assert x.shape == (stop - start, len(LM_FEATURE_NAMES)) and np.isfinite(x).all()
            result.append(x); token_counts.append(count)
            print(f"{name}: {stop:,}/{len(frame):,} rows | elapsed {(perf_counter()-start_time)/60:.1f} min", flush=True)
        return np.vstack(result), np.concatenate(token_counts)


    def lm_load(model_name, revision, device, local_dir=None):
        from transformers import AutoTokenizer, AutoModelForCausalLM
        source = str(local_dir) if local_dir is not None else model_name
        kwargs = {} if local_dir is not None else {"revision": revision}
        tokenizer = AutoTokenizer.from_pretrained(source, trust_remote_code=False, **kwargs)
        dtype = torch.float16 if device.type == "cuda" else torch.float32
        model = AutoModelForCausalLM.from_pretrained(
            source, torch_dtype=dtype, use_safetensors=True, trust_remote_code=False,
            attn_implementation="eager", **kwargs).to(device).eval()
        assert model.config.model_type == "gpt2" and tokenizer.eos_token_id is not None
        model.requires_grad_(False)
        return tokenizer, model


    def s8_dense_pipeline(c):
        # A small nonlinear feature map, followed by regularized Logistic Regression.
        from sklearn.pipeline import make_pipeline
        from sklearn.preprocessing import StandardScaler, PolynomialFeatures
        from sklearn.linear_model import LogisticRegression
        return make_pipeline(StandardScaler(), PolynomialFeatures(degree=2, include_bias=False),
                             StandardScaler(), LogisticRegression(C=c, solver="lbfgs", max_iter=5000,
                                                                   tol=1e-6, random_state=42))


    def s8_saved_predict(texts, model_folder, device=None):
        """Use after executing/importing the helper definitions; accepts arbitrary new text."""
        import joblib
        model_folder = Path(model_folder)
        manifest = json.loads((model_folder / "inference_manifest.json").read_text())
        branch_prob = []
        for family, weight in manifest["sparse_weights"].items():
            bundle = joblib.load(model_folder / manifest["sparse_files"][family])
            branch_prob.append(weight * predict_dropout(bundle, texts))
        baseline = np.sum(branch_prob, axis=0)
        device = torch.device(device or ("cuda" if torch.cuda.is_available() else "cpu"))
        local_model = model_folder / "pretrained_gpt2"
        config = manifest["lm_config"]
        tokenizer, model = lm_load(config["model"], config["revision"], device,
                                   local_model if local_model.is_dir() else None)
        x, _ = lm_extract(list(texts), tokenizer, model, device, config["context"],
                          config["stride"], config["batch_size"])
        dense = joblib.load(model_folder / "lm_logistic.joblib").predict_proba(x)[:, 1]
        result = {"baseline": baseline, "lm_only": dense}
        for name in ["experimental", "selected"]:
            weight = manifest[name + "_lm_weight"]
            result[name] = (1 - weight) * baseline + weight * dense
        del model
        gc.collect()
        if torch.cuda.is_available():
            clear_device_cache()
        return result

    # Original component step (94)
    def embedding_lr():
        return make_pipeline(Normalizer(),StandardScaler(),LogisticRegression(
            C=EMBEDDING_C,solver='lbfgs',max_iter=3000,tol=1e-5,random_state=42))

    def fit_embedding(x,y):
        model=embedding_lr()
        with threadpool_limits(limits=2),warnings.catch_warnings():
            warnings.simplefilter('error',ConvergenceWarning)
            model.fit(x,y)
        return model

    def extract_embeddings(texts,tokenizer,model,device,batch_size=8):
        normalized=[clean_text(t) for t in texts]
        if any(not t for t in normalized):raise ValueError('Blank normalized text')
        encoded=tokenizer(normalized,add_special_tokens=False,truncation=False)['input_ids']
        sequences=[[tokenizer.eos_token_id]+ids for ids in encoded]
        windows=[(r,ids,first) for r,seq in enumerate(sequences) for ids,first in lm_windows(seq,512,256)]
        windows.sort(key=lambda w:len(w[1]))
        totals=np.zeros((len(texts),model.config.n_embd),dtype=np.float64)
        counts=np.zeros(len(texts),dtype=np.int64)
        model.eval()
        with torch.inference_mode():
            for start in range(0,len(windows),batch_size):
                batch=windows[start:start+batch_size];length=max(len(seq) for _,seq,_ in batch)
                ids=torch.full((len(batch),length),tokenizer.eos_token_id,dtype=torch.long,device=device)
                mask=torch.zeros_like(ids)
                for j,(_,seq,_) in enumerate(batch):
                    ids[j,:len(seq)]=torch.tensor(seq,device=device);mask[j,:len(seq)]=1
                hidden=model.transformer(input_ids=ids,attention_mask=mask,use_cache=False).last_hidden_state
                for j,(r,seq,first) in enumerate(batch):
                    totals[r]+=hidden[j,first:len(seq)].float().sum(0).cpu().numpy()
                    counts[r]+=len(seq)-first
                del hidden,ids,mask
        assert np.array_equal(counts,[len(seq)-1 for seq in sequences])
        result=(totals/counts[:,None]).astype(np.float32)
        assert np.isfinite(result).all()
        return result
else:
    print('Skipped: saved components are used in normal mode.')


### 8.5 Optional Component Rebuild: Sparse and Normalized Statistics


In [ ]:
if REBUILD_COMPONENTS:
    # Original component step (84)
    BASELINE_ROOT = OUTPUT_DIR/'frozen_lm_lr'
    S8_DIR = BASELINE_ROOT/'8b1f60e51c7f454a'
    baseline_required=['config.json','selection.json','frozen_lm_features.csv','fold_assignments.csv',
        'oof_predictions.csv','validation_predictions.csv','train_validation_metrics.csv',
        'submission_lm_selected.csv','submission_manifest.json','final_models/inference_manifest.json']
    BASELINE_READY=not FORCE_BASELINE_REBUILD and all((S8_DIR/f).is_file() for f in baseline_required)
    print('Reuse completed baseline:',BASELINE_READY)
    if BASELINE_READY:
        s8_config=json.loads((S8_DIR/'config.json').read_text())
        assert s8_config['train_sha256']==hashlib.sha256((DATA_DIR/'train.csv').read_bytes()).hexdigest()
        assert s8_config['test_sha256']==hashlib.sha256((DATA_DIR/'test.csv').read_bytes()).hexdigest()
        assert s8_config['split_sha256']==hashlib.sha256(assignments.to_csv(index=False).encode()).hexdigest()
        s8_lm_config=s8_config['lm'];s8_revision=s8_lm_config['revision']
        s8_benchmark=json.loads((S8_DIR/'benchmark.json').read_text())
        s8_estimated_minutes=0.
        S9_MODEL_DIR=S8_DIR/'final_models'
        print('Restored baseline:',S8_DIR)
    else:
        S8_HELPER_SHA256 = 'e95ead67592fcc1752a9c8d259c644425b83d771b12630803f83ab2da6f6f72f'

        S8_ROOT = OUTPUT_DIR / "frozen_lm_lr"
        S8_ROOT.mkdir(parents=True, exist_ok=True)
        revision_path = S8_ROOT / "gpt2_revision.json"
        if revision_path.is_file():
            revision_record = json.loads(revision_path.read_text())
            assert revision_record["model"] == S8_MODEL_NAME
            s8_revision = revision_record["revision"]
        else:
            remote_config = AutoConfig.from_pretrained(S8_MODEL_NAME, trust_remote_code=False)
            s8_revision = remote_config._commit_hash
            assert isinstance(s8_revision, str) and len(s8_revision) == 40
            revision_path.write_text(json.dumps({"model": S8_MODEL_NAME, "revision": s8_revision}, indent=2))
        s8_tokenizer, s8_lm = lm_load(S8_MODEL_NAME, s8_revision, S8_DEVICE)
        assert S8_CONTEXT <= s8_lm.config.n_positions
        s8_lm_config = dict(version=LM_FEATURE_VERSION, model=S8_MODEL_NAME, revision=s8_revision,
            helper_sha256=S8_HELPER_SHA256, context=S8_CONTEXT, stride=S8_STRIDE,
            batch_size=S8_BATCH_SIZE, dtype=str(s8_lm.dtype), device_type=S8_DEVICE.type,
            transformers=version("transformers"), torch=version("torch"), normalization=VERSION,
            feature_names=LM_FEATURE_NAMES)
        s8_config = dict(helper_sha256=S8_HELPER_SHA256, lm=s8_lm_config, c_values=S8_C_VALUES,
            blend_weights=S8_BLEND_WEIGHTS, dropout=0.1, sparse_weights=S8_SPARSE_WEIGHTS,
            train_sha256=data_sha, test_sha256=hashlib.sha256((DATA_DIR / "test.csv").read_bytes()).hexdigest(),
            split_sha256=hashlib.sha256(assignments.to_csv(index=False).encode()).hexdigest(),
            versions={p: version(p) for p in ["numpy", "pandas", "scipy", "scikit-learn", "tokenizers", "joblib"]})
        S8_DIR = S8_ROOT / key(s8_config)
        S8_DIR.mkdir(parents=True, exist_ok=True)
        (S8_DIR / "config.json").write_text(json.dumps(s8_config, indent=2))
        s8_all_text = pd.concat([train_df[["id", "text"]], test_df[["id", "text"]]], ignore_index=True)
        s8_by_length = np.argsort(s8_all_text.text.str.len().to_numpy(), kind="stable")
        s8_pilot_idx = s8_by_length[np.linspace(0, len(s8_all_text)-1, 64, dtype=int)]
        lm_extract(s8_all_text.text.iloc[s8_pilot_idx[:4]].tolist(), s8_tokenizer, s8_lm, S8_DEVICE,
                   S8_CONTEXT, S8_STRIDE, S8_BATCH_SIZE)  # Warm up without timing initialization.
        if S8_DEVICE.type == "cuda": torch.cuda.synchronize()
        s8_bench_start = perf_counter()
        s8_pilot_x, s8_pilot_tokens = lm_extract(s8_all_text.text.iloc[s8_pilot_idx].tolist(),
            s8_tokenizer, s8_lm, S8_DEVICE, S8_CONTEXT, S8_STRIDE, S8_BATCH_SIZE)
        if S8_DEVICE.type == "cuda": torch.cuda.synchronize()
        s8_pilot_seconds = perf_counter() - s8_bench_start
        s8_estimated_minutes = s8_pilot_seconds / len(s8_pilot_idx) * len(s8_all_text) / 60
        s8_benchmark = dict(pilot_rows=len(s8_pilot_idx), seconds=s8_pilot_seconds,
            projected_feature_minutes=s8_estimated_minutes, total_rows=len(s8_all_text),
            budget_minutes=S8_FEATURE_BUDGET_MINUTES, device=str(S8_DEVICE))
        (S8_DIR / "benchmark.json").write_text(json.dumps(s8_benchmark, indent=2))
        display(pd.DataFrame([s8_benchmark]))
        print("Experiment:", S8_DIR)
        print("Estimate excludes download, CV, refits and Drive transfer; completed feature shards are reused.")

    # Original component step (86)
    if BASELINE_READY:
        s8_features=pd.read_csv(S8_DIR/'frozen_lm_features.csv').set_index('id')
        s8_fit_x=s8_features.loc[train_split.id].to_numpy();s8_val_x=s8_features.loc[val_split.id].to_numpy()
        s8_y=train_split.label.to_numpy()
        assert s8_features.index.is_unique and np.isfinite(s8_features.to_numpy()).all()
        print('Restored probability features:',s8_features.shape)
    else:
        if s8_estimated_minutes > S8_FEATURE_BUDGET_MINUTES and not S8_ALLOW_LONG_EXTRACTION:
            raise RuntimeError(f"Estimated extraction: {s8_estimated_minutes:.1f} min, budget: "
                f"{S8_FEATURE_BUDGET_MINUTES:.1f}. Select a GPU and rerun 8.1–8.4, "
                "or explicitly set S8_ALLOW_LONG_EXTRACTION = True to continue.")
        s8_all_features, s8_token_counts = lm_cached_extract(s8_all_text, "train_and_test",
            S8_DIR / "feature_cache", s8_tokenizer, s8_lm, S8_DEVICE, s8_lm_config)
        s8_features = pd.DataFrame(s8_all_features, columns=LM_FEATURE_NAMES, index=s8_all_text.id)
        assert s8_features.index.is_unique
        s8_features.index.name = "id"
        s8_features.to_csv(S8_DIR / "frozen_lm_features.csv")
        s8_fit_x = s8_features.loc[train_split.id].to_numpy()
        s8_val_x = s8_features.loc[val_split.id].to_numpy()
        s8_y = train_split.label.to_numpy()
        print("Features:", s8_features.shape, "| Token counts:", int(s8_token_counts.min()), "to", int(s8_token_counts.max()))
        # Release accelerator memory while CPU models run; retain the small frozen model for final export.
        s8_lm.to("cpu")
        if torch.cuda.is_available(): clear_device_cache()

    # Original component step (88)
    if BASELINE_READY:
        fold_assignments=pd.read_csv(S8_DIR/'fold_assignments.csv')
        assert np.array_equal(fold_assignments.id,train_split.id)
        assert np.array_equal(fold_assignments.label,train_split.label)
        stress_folds=[(np.flatnonzero(fold_assignments.fold.to_numpy()!=f),np.flatnonzero(fold_assignments.fold.to_numpy()==f)) for f in range(1,6)]
        s8_likeness=fold_assignments.set_index('id').target_likeness.reindex(train_df.id).fillna(.5)
        assert set(fold_assignments.fold)=={1,2,3,4,5}
    else:
        from sklearn.ensemble import HistGradientBoostingClassifier
        from sklearn.pipeline import make_pipeline

        test_df = pd.read_csv(DATA_DIR / "test.csv")
        assert test_df.id.is_unique and set(train_df.id).isdisjoint(test_df.id)
        source_config = dict(seed=42, folds=5, text_C=10, text_features=30000,
                             style_iterations=150, style_leaves=15, style_l2=10,
                             versions={p: version(p) for p in ["scikit-learn", "numpy", "pandas"]})
        source_hash = hashlib.sha256(canonical(source_config).encode())
        for frame in [train_df[["id", "text"]], test_df[["id", "text"]]]:
            source_hash.update(frame.to_json(orient="split", force_ascii=False).encode())
        SOURCE_DIR = OUTPUT_DIR / "source_diagnostics" / source_hash.hexdigest()[:16]
        SOURCE_DIR.mkdir(parents=True, exist_ok=True)
        source_path = SOURCE_DIR / "source_oof_predictions.csv"
        if source_path.is_file():
            source_table = pd.read_csv(source_path)
        else:
            source_text = pd.concat([train_df.text, test_df.text], ignore_index=True).to_numpy()
            source_y = np.r_[np.zeros(len(train_df), int), np.ones(len(test_df), int)]
            source_style = style_features(source_text.tolist())
            source_folds = list(StratifiedKFold(5, shuffle=True, random_state=42).split(source_text, source_y))
            source_prob = {}
            for kind in ["style_nonlinear", "word_text"]:
                p = np.full(len(source_y), np.nan)
                for fold, (idx, hold) in enumerate(source_folds, 1):
                    if kind == "style_nonlinear":
                        model = HistGradientBoostingClassifier(max_iter=150, max_leaf_nodes=15,
                                                               l2_regularization=10, random_state=42)
                        x = source_style
                    else:
                        model = make_pipeline(TfidfVectorizer(max_features=30000, min_df=3,
                            ngram_range=(1, 2), sublinear_tf=True),
                            LogisticRegression(C=10, solver="liblinear", max_iter=2000))
                        x = source_text
                    with threadpool_limits(limits=2):
                        model.fit(x[idx], source_y[idx])
                        p[hold] = model.predict_proba(x[hold])[:, 1]
                    print("Source diagnostic:", kind, "fold", fold)
                source_prob[kind] = check_prob(p, len(source_y))
            source_table = pd.DataFrame(dict(id=np.r_[train_df.id, test_df.id], source=source_y, **source_prob))
            source_table.to_csv(source_path, index=False)
            # Use the same CSV round trip as the research fold construction.
            source_table = pd.read_csv(source_path)
        assert source_table.id.is_unique
        assert np.array_equal(source_table.id.to_numpy(), np.r_[train_df.id, test_df.id])
        for name in ["style_nonlinear", "word_text"]:
            check_prob(source_table[name].to_numpy(), len(source_table))
            print(name, "train-versus-test AUROC:", roc_auc_score(source_table.source, source_table[name]))
        fold_assignments, stress_folds = make_folds(train_split, source_table)
        display(fold_assignments.groupby(["fold", "label"]).target_likeness.agg(["size", "mean"]))

        fold_assignments.to_csv(S8_DIR / "fold_assignments.csv", index=False)
        # Hash folds separately so cached predictions cannot silently follow a changed assignment.
        s8_fold_hash = hashlib.sha256(fold_assignments.to_csv(index=False).encode()).hexdigest()
        S8_CV_DIR = S8_DIR / ("cv_" + s8_fold_hash[:16])
        S8_CV_DIR.mkdir(parents=True, exist_ok=True)

    # Original component step (90)
    if BASELINE_READY:
        old_oof=pd.read_csv(S8_DIR/'oof_predictions.csv').set_index('id').loc[train_split.id]
        assert np.array_equal(old_oof.label,s8_y)
        s8_sparse_oof=old_oof['sparse'].to_numpy()
        s8_lm_oof={10.:old_oof['lm'].to_numpy()}
        s8_best_c=10.
        print('Baseline OOF predictions restored; parameter search skipped.')
    else:
        s8_sparse_rows, s8_sparse_oof_by_model = search_dropout(
            train_split, fold_assignments, stress_folds, S8_CV_DIR)
        s8_sparse_oof = sum(weight * s8_sparse_oof_by_model[key(dropout_spec(family, 0.1))]
                           for family, weight in S8_SPARSE_WEIGHTS.items())
        s8_lm_oof, s8_dense_rows = {}, []
        for c in S8_C_VALUES:
            oof = np.full(len(s8_y), np.nan)
            for fold, (idx, hold) in enumerate(stress_folds, 1):
                cache = S8_CV_DIR / f"lm_c{c:g}_fold{fold}.npz"
                if cache.is_file():
                    with np.load(cache, allow_pickle=False) as saved:
                        assert np.array_equal(saved["fit"], idx) and np.array_equal(saved["hold"], hold)
                        p = saved["probability"].copy()
                        row = json.loads(saved["metrics"].item())
                else:
                    model = s8_dense_pipeline(c)
                    fit_start = perf_counter()
                    with threadpool_limits(limits=2), warnings.catch_warnings():
                        warnings.simplefilter("error", ConvergenceWarning)
                        model.fit(s8_fit_x[idx], s8_y[idx])
                    p = model.predict_proba(s8_fit_x[hold])[:, 1]
                    row = dict(C=c, fold=fold, auc=roc_auc_score(s8_y[hold], p),
                        holdout_log_loss=log_loss(s8_y[hold], p),
                        clean_train_log_loss=log_loss(s8_y[idx], model.predict_proba(s8_fit_x[idx])[:, 1]),
                        training_seconds=perf_counter()-fit_start,
                        solver_iterations=int(model[-1].n_iter_.max()))
                    lm_atomic_npz(cache, fit=idx, hold=hold, probability=p, metrics=json.dumps(row))
                oof[hold] = check_prob(p, len(hold))
                s8_dense_rows.append(row)
                print(f"LM statistics LR: C={c:g}, fold={fold}/5, AUC={row['auc']:.6f}, "
                      f"train loss={row['clean_train_log_loss']:.4f}, holdout loss={row['holdout_log_loss']:.4f}")
            s8_lm_oof[c] = check_prob(oof, len(s8_y))
        s8_dense_details = pd.DataFrame(s8_dense_rows)
        s8_dense_summary = s8_dense_details.groupby("C", as_index=False).agg(
            mean_auc=("auc", "mean"), worst_auc=("auc", "min"), std_auc=("auc", "std"),
            mean_log_loss=("holdout_log_loss", "mean"), training_seconds=("training_seconds", "sum"))
        s8_dense_summary = s8_dense_summary.sort_values(["mean_auc", "C"], ascending=[False, True])
        s8_best_c = float(s8_dense_summary.iloc[0].C)
        s8_dense_details.to_csv(S8_DIR / "lm_cv_details.csv", index=False)
        s8_dense_summary.to_csv(S8_DIR / "lm_cv_summary.csv", index=False)
        display(s8_dense_summary)
        print("Locked statistical LR C:", s8_best_c)

    # Original component step (92)
    if BASELINE_READY:
        s8_selection=json.loads((S8_DIR/'selection.json').read_text())
        s8_candidate_weight=s8_selected_weight=.4
        s8_best_c=10.
        s8_metrics=pd.read_csv(S8_DIR/'train_validation_metrics.csv')
        display(s8_metrics)
    else:
        s8_best_c=10.;s8_candidate_weight=s8_selected_weight=.4
        s8_selection=dict(C=10.,experimental_lm_weight=.4,selected_lm_weight=.4,
            passes_cv_gate=True,public_auc=None,criterion='Locked historical baseline settings')
        (S8_DIR/'selection.json').write_text(json.dumps(s8_selection,indent=2))
        train_split[['id','label']].assign(sparse=s8_sparse_oof,lm=s8_lm_oof[10.]).to_csv(S8_DIR/'oof_predictions.csv',index=False)
        s8_likeness = source_table.set_index("id")[["style_nonlinear", "word_text"]].mean(axis=1)
        s8_dev_bundles = fit_dropout_bundles(train_split,
            train_split.id.map(s8_likeness).to_numpy(), [0.1], S8_DIR)
        s8_dense_dev = s8_dense_pipeline(s8_best_c)
        with threadpool_limits(limits=2), warnings.catch_warnings():
            warnings.simplefilter("error", ConvergenceWarning)
            s8_dense_dev.fit(s8_fit_x, s8_y)
        joblib.dump(s8_dense_dev, S8_DIR / "lm_logistic_8000.joblib", compress=3)
        s8_metrics = []
        for split_name, frame, matrix in [("train", train_split, s8_fit_x), ("validation", val_split, s8_val_x)]:
            sparse_p = sum(weight * predict_dropout(s8_dev_bundles[key(dropout_spec(family, .1))], frame.text)
                           for family, weight in S8_SPARSE_WEIGHTS.items())
            lm_p = s8_dense_dev.predict_proba(matrix)[:, 1]
            predictions = {"sparse_baseline": sparse_p, "lm_only": lm_p,
                "experimental_blend": (1-s8_candidate_weight)*sparse_p+s8_candidate_weight*lm_p,
                "cv_selected": (1-s8_selected_weight)*sparse_p+s8_selected_weight*lm_p}
            for name, probability in predictions.items():
                probability = check_prob(probability, len(frame))
                s8_metrics.append(dict(model=name, split=split_name,
                    auc=roc_auc_score(frame.label, probability), log_loss=log_loss(frame.label, probability)))
            frame[["id", "label"]].assign(**predictions).to_csv(S8_DIR / f"{split_name}_predictions.csv", index=False)
        s8_metrics = pd.DataFrame(s8_metrics)
        s8_metrics.to_csv(S8_DIR / "train_validation_metrics.csv", index=False)
        display(s8_metrics)
        fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
        for ax, metric in zip(axes, ["auc", "log_loss"]):
            s8_metrics.pivot(index="model", columns="split", values=metric).plot.bar(ax=ax, rot=20)
            ax.set(xlabel="", ylabel=metric, title="AUROC" if metric == "auc" else "Final log loss")
            ax.grid(axis="y", alpha=.2)
        axes[0].set_ylim(0, 1.05)
        fig.savefig(S8_DIR / "train_validation.png", dpi=180)
        plt.show()
        print("Configuration remains locked. Public/private leaderboard scores for this experiment are unknown.")
else:
    print('Skipped: saved components are used in normal mode.')


### 8.6 Optional Component Rebuild: Frozen GPT-2 Embeddings


In [ ]:
if REBUILD_COMPONENTS:
    # Original component step (96)
    EMBEDDING_SAVED_DIR=OUTPUT_DIR/'gpt2_embedding_lr/8d87d6e871124fda'
    all_text=pd.concat([train_df[['id','text']],test_df[['id','text']]],ignore_index=True)
    route_config=dict(version='restore_094189_v1',baseline_config=s8_config,
        embedding_C=EMBEDDING_C,embedding_weight=FINAL_EMBEDDING_WEIGHT,
        data_hash=hashlib.sha256(all_text.to_json(orient='split').encode()).hexdigest())
    ROUTE_DIR=OUTPUT_DIR/'restored_094189'/key(route_config)
    ROUTE_DIR.mkdir(parents=True,exist_ok=True)
    (ROUTE_DIR/'config.json').write_text(json.dumps(route_config,indent=2))
    EMBEDDING_READY=not FORCE_EMBEDDING_REBUILD and (EMBEDDING_SAVED_DIR/'all_embeddings.npz').is_file()
    if EMBEDDING_READY:
        ecfg=json.loads((EMBEDDING_SAVED_DIR/'config.json').read_text())
        assert ecfg['data_order_hash']==route_config['data_hash']
        assert ecfg['baseline_config']==s8_config
        assert ecfg['version']=='gpt2_unique_token_final_layer_mean_v1'
        assert ecfg['context']==512 and ecfg['stride']==256
        with np.load(EMBEDDING_SAVED_DIR/'all_embeddings.npz',allow_pickle=False) as saved:
            assert np.array_equal(saved['ids'],all_text.id.to_numpy())
            vectors=saved['vectors'].copy()
        print('Restored all article embeddings:',vectors.shape)
    else:
        cache=ROUTE_DIR/'embedding_cache';cache.mkdir(exist_ok=True)
        local_weights=S8_DIR/'final_models/pretrained_gpt2'
        tokenizer,model=lm_load(s8_lm_config['model'],s8_lm_config['revision'],S8_DEVICE,
            local_weights if local_weights.is_dir() else None)
        blocks=[]
        for a in range(0,len(all_text),128):
            b=min(a+128,len(all_text));path=cache/f'{a:06d}_{b:06d}.npz'
            if path.is_file():
                with np.load(path,allow_pickle=False) as z:
                    assert np.array_equal(z['ids'],all_text.id.iloc[a:b].to_numpy())
                    v=z['vectors'].copy()
            else:
                v=extract_embeddings(all_text.text.iloc[a:b].tolist(),tokenizer,model,S8_DEVICE)
                lm_atomic_npz(path,ids=all_text.id.iloc[a:b].to_numpy(),vectors=v)
            blocks.append(v);print(f'Embeddings: {b}/{len(all_text)}',flush=True)
        vectors=np.vstack(blocks);model.to('cpu');del model;gc.collect()
        if torch.cuda.is_available():clear_device_cache()
    assert vectors.shape==(len(all_text),768) and np.isfinite(vectors).all()
    embedding_features=pd.DataFrame(vectors,index=all_text.id)
    lm_atomic_npz(ROUTE_DIR/'all_embeddings.npz',ids=all_text.id.to_numpy(),vectors=vectors)

    # Original component step (98)
    embedding_x=embedding_features.loc[train_split.id].to_numpy()
    embedding_oof=np.full(len(train_split),np.nan);embedding_rows=[]
    for fold,(idx,hold) in enumerate(stress_folds,1):
        saved_path=EMBEDDING_SAVED_DIR/'cv'/f'C0.1_fold{fold}.npz'
        path=ROUTE_DIR/f'embedding_C0.1_fold{fold}.npz'
        read_path=saved_path if EMBEDDING_READY and saved_path.is_file() else path
        if read_path.is_file():
            with np.load(read_path,allow_pickle=False) as z:
                assert np.array_equal(z['fit'],idx) and np.array_equal(z['hold'],hold)
                p=z['probability'].copy()
        else:
            model=fit_embedding(embedding_x[idx],s8_y[idx]);p=model.predict_proba(embedding_x[hold])[:,1]
            lm_atomic_npz(path,fit=idx,hold=hold,probability=p)
        embedding_oof[hold]=check_prob(p,len(hold))
        baseline=.6*s8_sparse_oof[hold]+.4*s8_lm_oof[10.][hold]
        combined=.8*baseline+.2*p
        embedding_rows.append(dict(fold=fold,baseline_auc=roc_auc_score(s8_y[hold],baseline),
            embedding_auc=roc_auc_score(s8_y[hold],p),final_auc=roc_auc_score(s8_y[hold],combined)))
    embedding_cv=pd.DataFrame(embedding_rows)
    embedding_cv.to_csv(ROUTE_DIR/'cv_comparison.csv',index=False)
    display(embedding_cv);display(embedding_cv[['baseline_auc','embedding_auc','final_auc']].mean().to_frame('mean_auc'))
    print('Parameters stay locked: C=0.1; embedding weight=0.2. No new search.')

    # Original component step (100)
    old_val=pd.read_csv(S8_DIR/'validation_predictions.csv').set_index('id').loc[val_split.id]
    assert np.array_equal(old_val.label,val_split.label)
    saved_val=EMBEDDING_SAVED_DIR/'validation_predictions.csv'
    if EMBEDDING_READY and saved_val.is_file():
        vp=pd.read_csv(saved_val).set_index('id').loc[val_split.id]
        assert np.array_equal(vp.label,val_split.label)
        embedding_val=vp['embedding'].to_numpy()
    else:
        dev_model=fit_embedding(embedding_x,s8_y)
        joblib.dump(dev_model,ROUTE_DIR/'embedding_lr_8000.joblib',compress=3)
        embedding_val=dev_model.predict_proba(embedding_features.loc[val_split.id].to_numpy())[:,1]
    baseline_val=old_val['cv_selected'].to_numpy()
    final_val=.8*baseline_val+.2*embedding_val
    validation_metrics=pd.DataFrame([dict(model=name,auc=roc_auc_score(val_split.label,p),
        log_loss=log_loss(val_split.label,p)) for name,p in [('baseline_094175',baseline_val),
        ('embedding_only',embedding_val),('final_094189_route',final_val)]])
    validation_metrics.to_csv(ROUTE_DIR/'validation_metrics.csv',index=False)
    val_split[['id','label']].assign(baseline=baseline_val,embedding=embedding_val,final=final_val).to_csv(ROUTE_DIR/'validation_predictions.csv',index=False)
    display(validation_metrics)
    fig,axes=plt.subplots(1,2,figsize=(11,4),constrained_layout=True)
    for name in ['baseline_auc','final_auc']:
        axes[0].plot(embedding_cv.fold.to_numpy(),embedding_cv[name].to_numpy(),'o-',label=name)
    axes[0].set(xlabel='Development fold',ylabel='AUROC');axes[0].legend();axes[0].grid(alpha=.2)
    axes[1].bar(validation_metrics.model.to_numpy(),validation_metrics.auc.to_numpy())
    axes[1].tick_params(axis='x',rotation=20);axes[1].set(ylabel='AUROC',title='Fixed validation')
    fig.savefig(ROUTE_DIR/'comparison.png',dpi=180);plt.show()
else:
    print('Skipped: saved components are used in normal mode.')


### 8.7 Optional Component Rebuild: Raw-Text Statistics and CV


In [ ]:
if REBUILD_COMPONENTS:
    # Original component step (102)
    import types
    RAW_C_VALUES=[.001,.01,.1,1.,10.]
    RAW_BLEND_WEIGHTS=[0.,.05,.1,.2,.3]
    RAW_FEATURE_BUDGET_MINUTES=12.
    RAW_ALLOW_LONG_EXTRACTION=False
    raw_device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    raw_local_weights=S8_DIR/'final_models/pretrained_gpt2'
    raw_tokenizer,raw_lm=lm_load(s8_lm_config['model'],s8_lm_config['revision'],raw_device,
        raw_local_weights if raw_local_weights.is_dir() else None)
    # Give the extraction function its own globals so the baseline cleaner stays unchanged.
    raw_extract=types.FunctionType(lm_extract.__code__,
        {**lm_extract.__globals__,'clean_text':lambda text:text},'raw_extract',lm_extract.__defaults__)
    raw_config=dict(version='original_text_gpt2_statistics_v1',baseline=route_config,
        model=s8_lm_config['model'],revision=s8_lm_config['revision'],
        helper_sha256=s8_config['helper_sha256'],context=512,stride=256,batch_size=4,
        dtype=str(raw_lm.dtype),torch=version('torch'),transformers=version('transformers'),
        C_values=RAW_C_VALUES,blend_weights=RAW_BLEND_WEIGHTS,
        data_hash=hashlib.sha256(all_text.to_json(orient='split').encode()).hexdigest(),
        fold_hash=hashlib.sha256(fold_assignments.to_csv(index=False).encode()).hexdigest())
    RAW_DIR=OUTPUT_DIR/'raw_gpt2_followup'/key(raw_config)
    RAW_DIR.mkdir(parents=True,exist_ok=True)
    (RAW_DIR/'config.json').write_text(json.dumps(raw_config,indent=2))
    raw_cache=RAW_DIR/'features';raw_cache.mkdir(exist_ok=True)
    raw_blocks=[(a,min(a+128,len(all_text))) for a in range(0,len(all_text),128)]
    raw_remaining=sum(b-a for a,b in raw_blocks if not (raw_cache/f'{a:06d}_{b:06d}.npz').is_file())
    if raw_remaining:
        order=np.argsort(all_text.text.str.len().to_numpy(),kind='stable')
        pilot=order[np.linspace(0,len(order)-1,32,dtype=int)]
        raw_extract(all_text.text.iloc[pilot[:2]].tolist(),raw_tokenizer,raw_lm,raw_device)
        synchronize_device(raw_device);t0=perf_counter()
        pilot_features,_=raw_extract(all_text.text.iloc[pilot].tolist(),raw_tokenizer,raw_lm,raw_device)
        synchronize_device(raw_device)
        estimate=(perf_counter()-t0)/len(pilot)*raw_remaining/60
        (RAW_DIR/'benchmark.json').write_text(json.dumps(dict(remaining_rows=raw_remaining,estimated_minutes=estimate),indent=2))
        print(f'Estimated remaining raw feature extraction: {estimate:.1f} minutes; excludes fitting and transfers.')
        if estimate>RAW_FEATURE_BUDGET_MINUTES and not RAW_ALLOW_LONG_EXTRACTION:
            raise RuntimeError('Estimated extraction exceeds budget. Review the estimate before enabling RAW_ALLOW_LONG_EXTRACTION.')
    raw_rows=[]
    for a,b in raw_blocks:
        path=raw_cache/f'{a:06d}_{b:06d}.npz'
        ids=all_text.id.iloc[a:b].to_numpy()
        if path.is_file():
            with np.load(path,allow_pickle=False) as saved:
                assert np.array_equal(saved['ids'],ids)
                x=saved['features'].copy()
        else:
            x,count=raw_extract(all_text.text.iloc[a:b].tolist(),raw_tokenizer,raw_lm,raw_device)
            lm_atomic_npz(path,ids=ids,features=x,token_count=count)
        assert x.shape==(b-a,24) and np.isfinite(x).all()
        raw_rows.append(x)
        print(f'Raw GPT-2 features: {b}/{len(all_text)}',flush=True)
    raw_features=pd.DataFrame(np.vstack(raw_rows),index=all_text.id,columns=LM_FEATURE_NAMES)
    raw_features.index.name='id';raw_features.to_csv(RAW_DIR/'raw_features.csv')
    raw_lm.to('cpu');del raw_lm;gc.collect();clear_device_cache()
    print('New experiment:',RAW_DIR)

    # Original component step (104)
    raw_x=raw_features.loc[train_split.id].to_numpy()
    raw_oof={};raw_cv_rows=[]
    raw_cv_dir=RAW_DIR/'cv';raw_cv_dir.mkdir(exist_ok=True)
    for c in RAW_C_VALUES:
        oof=np.full(len(s8_y),np.nan)
        for fold,(idx,hold) in enumerate(stress_folds,1):
            path=raw_cv_dir/f'C{c:g}_fold{fold}.npz'
            if path.is_file():
                with np.load(path,allow_pickle=False) as saved:
                    assert np.array_equal(saved['fit'],idx) and np.array_equal(saved['hold'],hold)
                    p=saved['probability'].copy();row=json.loads(saved['metrics'].item())
            else:
                t0=perf_counter();model=s8_dense_pipeline(c)
                with threadpool_limits(limits=2),warnings.catch_warnings():
                    warnings.simplefilter('error',ConvergenceWarning)
                    model.fit(raw_x[idx],s8_y[idx])
                p=model.predict_proba(raw_x[hold])[:,1]
                row=dict(C=c,fold=fold,auc=roc_auc_score(s8_y[hold],p),
                    train_loss=log_loss(s8_y[idx],model.predict_proba(raw_x[idx])[:,1]),
                    holdout_loss=log_loss(s8_y[hold],p),seconds=perf_counter()-t0)
                lm_atomic_npz(path,fit=idx,hold=hold,probability=p,metrics=json.dumps(row))
            oof[hold]=check_prob(p,len(hold));raw_cv_rows.append(row)
            print(f"C={c:g}, fold={fold}: AUC={row['auc']:.6f}; train loss={row['train_loss']:.4f}; holdout loss={row['holdout_loss']:.4f}")
        raw_oof[c]=check_prob(oof,len(s8_y))
    raw_cv_details=pd.DataFrame(raw_cv_rows)
    raw_cv_summary=raw_cv_details.groupby('C',as_index=False).agg(mean_auc=('auc','mean'),worst_auc=('auc','min'))
    raw_cv_summary=raw_cv_summary.sort_values(['mean_auc','C'],ascending=[False,True])
    raw_best_c=float(raw_cv_summary.iloc[0].C)
    raw_cv_details.to_csv(RAW_DIR/'cv_details.csv',index=False)
    raw_cv_summary.to_csv(RAW_DIR/'cv_summary.csv',index=False)
    display(raw_cv_summary)

    # Original component step (106)
    control_oof=.8*(.6*s8_sparse_oof+.4*s8_lm_oof[10.])+.2*embedding_oof
    raw_blend_rows=[]
    for w in RAW_BLEND_WEIGHTS:
        p=(1-w)*control_oof+w*raw_oof[raw_best_c]
        for fold,(_,hold) in enumerate(stress_folds,1):
            raw_blend_rows.append(dict(weight=w,fold=fold,auc=roc_auc_score(s8_y[hold],p[hold])))
    raw_blend_details=pd.DataFrame(raw_blend_rows)
    raw_blends=raw_blend_details.groupby('weight',as_index=False).agg(mean_auc=('auc','mean'),worst_auc=('auc','min'))
    base=raw_blends[raw_blends.weight.eq(0)].iloc[0]
    base_folds=raw_blend_details[raw_blend_details.weight.eq(0)].set_index('fold').auc
    raw_blends['mean_gain']=raw_blends.mean_auc-base.mean_auc
    raw_blends['worst_gain']=raw_blends.worst_auc-base.worst_auc
    raw_blends['improved_folds']=[int((raw_blend_details[raw_blend_details.weight.eq(w)].set_index('fold').auc>base_folds).sum()) for w in raw_blends.weight]
    raw_blends['eligible']=(raw_blends.weight>0)&(raw_blends.mean_gain>=.0005)&(raw_blends.worst_gain>=0)&(raw_blends.improved_folds>=3)
    ranked=raw_blends[raw_blends.weight>0].sort_values(['mean_auc','weight'],ascending=[False,True])
    eligible=ranked[ranked.eligible]
    raw_candidate_weight=float((eligible if len(eligible) else ranked).iloc[0].weight)
    raw_selected_weight=raw_candidate_weight if len(eligible) else 0.
    raw_selection=dict(C=raw_best_c,experimental_raw_weight=raw_candidate_weight,
        selected_raw_weight=raw_selected_weight,passes_cv_gate=bool(len(eligible)),public_auc=None)
    (RAW_DIR/'selection.json').write_text(json.dumps(raw_selection,indent=2))
    raw_blends.to_csv(RAW_DIR/'blend_cv_summary.csv',index=False)
    train_split[['id','label']].assign(control=control_oof,raw=raw_oof[raw_best_c]).to_csv(RAW_DIR/'oof_predictions.csv',index=False)
    display(raw_blends);print('Locked selection:',raw_selection)

    # Original component step (108)
    raw_dev=s8_dense_pipeline(raw_best_c)
    with threadpool_limits(limits=2),warnings.catch_warnings():
        warnings.simplefilter('error',ConvergenceWarning)
        raw_dev.fit(raw_x,s8_y)
    joblib.dump(raw_dev,RAW_DIR/'raw_lr_8000.joblib',compress=3)
    raw_val=raw_dev.predict_proba(raw_features.loc[val_split.id].to_numpy())[:,1]
    raw_val_predictions=dict(control_094189=final_val,raw_only=raw_val,
        experimental=(1-raw_candidate_weight)*final_val+raw_candidate_weight*raw_val,
        selected=(1-raw_selected_weight)*final_val+raw_selected_weight*raw_val)
    raw_val_metrics=pd.DataFrame([dict(model=name,auc=roc_auc_score(val_split.label,p),log_loss=log_loss(val_split.label,p)) for name,p in raw_val_predictions.items()])
    raw_val_metrics.to_csv(RAW_DIR/'validation_metrics.csv',index=False)
    val_split[['id','label']].assign(**raw_val_predictions).to_csv(RAW_DIR/'validation_predictions.csv',index=False)
    display(raw_val_metrics)
    fig,axes=plt.subplots(1,2,figsize=(11,4),constrained_layout=True)
    for w in sorted(set([0.,raw_candidate_weight])):
        r=raw_blend_details[raw_blend_details.weight.eq(w)]
        axes[0].plot(r.fold.to_numpy(),r.auc.to_numpy(),'o-',label=f'Raw weight {w:g}')
    axes[0].set(xlabel='Development fold',ylabel='AUROC');axes[0].legend();axes[0].grid(alpha=.2)
    r=raw_cv_details[raw_cv_details.C.eq(raw_best_c)]
    axes[1].plot(r.fold.to_numpy(),r.train_loss.to_numpy(),'o-',label='Training loss')
    axes[1].plot(r.fold.to_numpy(),r.holdout_loss.to_numpy(),'o-',label='Holdout loss')
    axes[1].set(xlabel='Fold',ylabel='Final log loss');axes[1].legend();axes[1].grid(alpha=.2)
    fig.savefig(RAW_DIR/'comparison.png',dpi=180);plt.show()
else:
    print('Skipped: saved components are used in normal mode.')


### 8.8 Optional Component Rebuild: Saved Final Component Models


In [ ]:
if REBUILD_COMPONENTS:
    # Original component step (111)
    S9_MODEL_DIR=S8_DIR/'final_models'
    baseline_final_ready=(S9_MODEL_DIR/'inference_manifest.json').is_file() and (S8_DIR/'submission_lm_selected.csv').is_file()
    if not baseline_final_ready or (REFIT_FINAL_MODELS and not BASELINE_READY):
        s9_started = perf_counter()
        S9_MODEL_DIR = S8_DIR / "final_models"
        S9_MODEL_DIR.mkdir(parents=True, exist_ok=True)
        s9_bundles = fit_dropout_bundles(train_df, train_df.id.map(s8_likeness).to_numpy(), [0.1], S8_DIR)
        s9_dense = s8_dense_pipeline(s8_best_c)
        with threadpool_limits(limits=2), warnings.catch_warnings():
            warnings.simplefilter("error", ConvergenceWarning)
            s9_dense.fit(s8_features.loc[train_df.id].to_numpy(), train_df.label.to_numpy())
        joblib.dump(s9_dense, S9_MODEL_DIR / "lm_logistic.joblib", compress=3)
        sparse_files = {}
        for family in S8_SPARSE_WEIGHTS:
            model_key = key(dropout_spec(family, .1))
            filename = f"sparse_{model_key}.joblib"
            joblib.dump(s9_bundles[model_key], S9_MODEL_DIR / filename, compress=3)
            sparse_files[family] = filename
        if S8_SAVE_PRETRAINED_WEIGHTS:
            s8_lm.save_pretrained(S9_MODEL_DIR / "pretrained_gpt2", safe_serialization=True)
            s8_tokenizer.save_pretrained(S9_MODEL_DIR / "pretrained_gpt2")
        s9_manifest = dict(training_rows=len(train_df), train_sha256=data_sha, lm_config=s8_lm_config,
            sparse_weights=S8_SPARSE_WEIGHTS, sparse_files=sparse_files, **s8_selection,
            sklearn_version=version("scikit-learn"), note="Final refit includes former validation rows.")
        (S9_MODEL_DIR / "inference_manifest.json").write_text(json.dumps(s9_manifest, indent=2))
        print("Saved models:", S9_MODEL_DIR)
        print(f"Final-refit/export-model elapsed: {(perf_counter()-s9_started)/60:.2f} min")

        s9_ordered = test_df.set_index("id", drop=False).loc[sample_submission_df.id].reset_index(drop=True)
        s9_test_x = s8_features.loc[s9_ordered.id].to_numpy()
        s9_sparse_p = sum(weight * predict_dropout(s9_bundles[key(dropout_spec(family, .1))], s9_ordered.text)
                          for family, weight in S8_SPARSE_WEIGHTS.items())
        s9_lm_p = s9_dense.predict_proba(s9_test_x)[:, 1]
        s9_loaded_sparse = sum(weight * predict_dropout(joblib.load(S9_MODEL_DIR / sparse_files[family]), s9_ordered.text)
                               for family, weight in S8_SPARSE_WEIGHTS.items())
        s9_loaded_lm = joblib.load(S9_MODEL_DIR / "lm_logistic.joblib").predict_proba(s9_test_x)[:, 1]
        np.testing.assert_allclose(s9_sparse_p, s9_loaded_sparse, atol=1e-12, rtol=0)
        np.testing.assert_allclose(s9_lm_p, s9_loaded_lm, atol=1e-12, rtol=0)
        s8_lm.to(S8_DEVICE)
        check_x, _ = lm_extract(s9_ordered.text.iloc[:3].tolist(), s8_tokenizer, s8_lm, S8_DEVICE,
                                S8_CONTEXT, S8_STRIDE, S8_BATCH_SIZE)
        # Different padding/batching can introduce small reduced-precision differences.
        np.testing.assert_allclose(check_x, s9_test_x[:3], atol=.02, rtol=.01)
        s9_probabilities = dict(baseline=check_prob(s9_sparse_p, 3000),
            experimental=check_prob((1-s8_candidate_weight)*s9_sparse_p+s8_candidate_weight*s9_lm_p, 3000),
            selected=check_prob((1-s8_selected_weight)*s9_sparse_p+s8_selected_weight*s9_lm_p, 3000))
        print("Reload verification passed for all test predictions; feature re-extraction passed for three texts.")

        s9_exports = []
        for name, probability in s9_probabilities.items():
            submission = sample_submission_df[["id"]].copy()
            submission["label"] = probability
            assert submission.columns.tolist() == ["id", "label"]
            assert submission.id.is_unique and len(submission) == 3000
            assert np.array_equal(submission.id.to_numpy(), sample_submission_df.id.to_numpy())
            check_prob(submission.label.to_numpy(), len(submission))
            assert submission.label.nunique() > 2, "Export probabilities, not hard classes."
            path = S8_DIR / f"submission_lm_{name}.csv"
            submission.to_csv(path, index=False)
            reread = pd.read_csv(path)
            assert reread.columns.tolist() == ["id", "label"]
            assert np.array_equal(reread.id.to_numpy(), sample_submission_df.id.to_numpy())
            np.testing.assert_allclose(reread.label, probability, atol=1e-12, rtol=0)
            s9_exports.append(dict(variant=name, path=str(path), rows=len(submission),
                sha256=hashlib.sha256(path.read_bytes()).hexdigest(), public_auc=None, private_auc=None))
        pd.DataFrame(s9_exports).to_csv(S8_DIR / "submission_manifest.csv", index=False)
        (S8_DIR / "submission_manifest.json").write_text(json.dumps(s9_exports, indent=2))
        SUBMISSION_PATH = S8_DIR / "submission_lm_selected.csv"
        EXPERIMENTAL_SUBMISSION_PATH = S8_DIR / "submission_lm_experimental.csv"
        display(pd.DataFrame(s9_exports)[["variant", "path", "rows"]])
        display(pd.read_csv(EXPERIMENTAL_SUBMISSION_PATH).head())
        print("CV-selected submission:", SUBMISSION_PATH)
        print("Nonzero LM-weight experiment:", EXPERIMENTAL_SUBMISSION_PATH)
        print("All files are also visible in the project's Google Drive outputs folder.")

    else:
        s9_manifest=json.loads((S9_MODEL_DIR/'inference_manifest.json').read_text())
    print('Baseline final model directory:',S9_MODEL_DIR)

    # Original component step (113)
    embedding_final_path=EMBEDDING_SAVED_DIR/'embedding_lr_10000.joblib'
    if REFIT_FINAL_MODELS or not embedding_final_path.is_file():
        final_embedding_model=fit_embedding(embedding_features.loc[train_df.id].to_numpy(),train_df.label.to_numpy())
        embedding_final_path=ROUTE_DIR/'embedding_lr_10000.joblib'
        joblib.dump(final_embedding_model,embedding_final_path,compress=3)
    else:
        print('Reuse final embedding model:',embedding_final_path)
    print('Final training rows: 10,000; test rows: 3,000.')

    # Original component step (115)
    baseline_csv=S8_DIR/'submission_lm_selected.csv'
    baseline_records=json.loads((S8_DIR/'submission_manifest.json').read_text())
    base_record=next(r for r in baseline_records if r['variant']=='selected')
    assert hashlib.sha256(baseline_csv.read_bytes()).hexdigest()==base_record['sha256']
    baseline_submission=pd.read_csv(baseline_csv)
    assert np.array_equal(baseline_submission.id,sample_submission_df.id)
    saved_csv=EMBEDDING_SAVED_DIR/'submission_embedding_selected.csv'
    FINAL_SUBMISSION_PATH=ROUTE_DIR/'submission_094189.csv'
    RESTORED_MEASURED_SUBMISSION=EMBEDDING_READY and saved_csv.is_file() and not REFIT_FINAL_MODELS
    if RESTORED_MEASURED_SUBMISSION:
        selection=json.loads((EMBEDDING_SAVED_DIR/'selection.json').read_text())
        assert selection['C']==.1 and selection['selected_embedding_weight']==.2
        records=json.loads((EMBEDDING_SAVED_DIR/'submission_manifest.json').read_text())
        selected=next(r for r in records if r['variant']=='selected')
        assert hashlib.sha256(saved_csv.read_bytes()).hexdigest()==selected['sha256']
        FINAL_SUBMISSION_PATH.write_bytes(saved_csv.read_bytes())
    else:
        if 'final_embedding_model' not in globals():
            assert version('scikit-learn')==s9_manifest['sklearn_version'], 'Install the saved sklearn version before loading models.'
            final_embedding_model=joblib.load(embedding_final_path)
        p=final_embedding_model.predict_proba(embedding_features.loc[sample_submission_df.id].to_numpy())[:,1]
        output=sample_submission_df[['id']].copy()
        output['label']=check_prob(.8*baseline_submission.label.to_numpy()+.2*p,len(output))
        output.to_csv(FINAL_SUBMISSION_PATH,index=False)
    submission=pd.read_csv(FINAL_SUBMISSION_PATH)
    assert submission.columns.tolist()==['id','label'] and len(submission)==3000 and submission.id.is_unique
    assert np.array_equal(submission.id,sample_submission_df.id)
    check_prob(submission.label.to_numpy(),len(submission));assert submission.label.nunique()>2
    submission_record=dict(path=str(FINAL_SUBMISSION_PATH),sha256=hashlib.sha256(FINAL_SUBMISSION_PATH.read_bytes()).hexdigest(),
        restored_measured_file=RESTORED_MEASURED_SUBMISSION,public_auc=.94189 if RESTORED_MEASURED_SUBMISSION else None,
        baseline_weight=.8,embedding_weight=.2,embedding_C=.1)
    (ROUTE_DIR/'submission_manifest.json').write_text(json.dumps(submission_record,indent=2))
    print('Submission:',FINAL_SUBMISSION_PATH)
    print('Restored the original submitted bytes:',RESTORED_MEASURED_SUBMISSION)
    display(submission.head())
    if DOWNLOAD_SUBMISSION and IN_COLAB:
        from google.colab import files
        files.download(str(FINAL_SUBMISSION_PATH))
    elif not IN_COLAB:
        from IPython.display import FileLink
        display(FileLink(str(FINAL_SUBMISSION_PATH)))

    # Original component step (117)
    raw_final=s8_dense_pipeline(raw_best_c)
    with threadpool_limits(limits=2),warnings.catch_warnings():
        warnings.simplefilter('error',ConvergenceWarning)
        raw_final.fit(raw_features.loc[train_df.id].to_numpy(),train_df.label.to_numpy())
    raw_model_path=RAW_DIR/'raw_lr_10000.joblib'
    joblib.dump(raw_final,raw_model_path,compress=3)
    raw_test_x=raw_features.loc[sample_submission_df.id].to_numpy()
    raw_test=raw_final.predict_proba(raw_test_x)[:,1]
    np.testing.assert_allclose(raw_test[:16],joblib.load(raw_model_path).predict_proba(raw_test_x[:16])[:,1],atol=1e-12,rtol=0)
    control=pd.read_csv(FINAL_SUBMISSION_PATH)
    assert np.array_equal(control.id,sample_submission_df.id)
    raw_exports=[]
    for name,w in [('experimental',raw_candidate_weight),('selected',raw_selected_weight)]:
        path=RAW_DIR/f'submission_raw_gpt2_{name}.csv'
        if w==0:
            path.write_bytes(FINAL_SUBMISSION_PATH.read_bytes())
        else:
            output=sample_submission_df[['id']].copy()
            output['label']=check_prob((1-w)*control.label.to_numpy()+w*raw_test,len(control))
            output.to_csv(path,index=False)
        output=pd.read_csv(path)
        assert output.columns.tolist()==['id','label'] and output.id.is_unique
        assert len(output)==3000 and np.array_equal(output.id,sample_submission_df.id)
        check_prob(output.label.to_numpy(),len(output));assert output.label.nunique()>2
        raw_exports.append(dict(variant=name,path=str(path),raw_weight=w,
            sha256=hashlib.sha256(path.read_bytes()).hexdigest(),public_auc=None))
    (RAW_DIR/'submission_manifest.json').write_text(json.dumps(raw_exports,indent=2))
    print('Candidate files:',RAW_DIR)
    display(pd.DataFrame(raw_exports))
    if raw_selected_weight==0:print('CV gate failed: selected is the unchanged 0.94189 control. Experimental is a separate candidate.')
    if DOWNLOAD_SUBMISSION and IN_COLAB:
        from google.colab import files
        files.download(str(RAW_DIR/'submission_raw_gpt2_selected.csv'))

    # Original component step (122)
    BASE_HELPER_SOURCE='import importlib.util\nimport subprocess\nimport sys\nif importlib.util.find_spec("tokenizers") is None:\n    subprocess.check_call([sys.executable, "-m", "pip", "install", "tokenizers"])\n"""Training-only five-fold research on sparse text and observable style features.\n\nRun as a script locally. Import functions in Colab after making this file available.\nNo test-set labels, external labeled data, or pretrained encoders are used.\n"""\nimport argparse\nfrom collections import Counter\nimport gc\nimport hashlib\nfrom importlib.metadata import version\nimport json\nfrom pathlib import Path\nimport re\nfrom time import perf_counter\nimport warnings\n\nimport joblib\nimport numpy as np\nimport pandas as pd\nfrom scipy.sparse import csr_matrix, hstack\nfrom sklearn.exceptions import ConvergenceWarning\nfrom sklearn.feature_extraction.text import TfidfVectorizer\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.metrics import accuracy_score, log_loss, roc_auc_score\nfrom sklearn.model_selection import StratifiedKFold\nfrom sklearn.preprocessing import StandardScaler, normalize\nfrom threadpoolctl import threadpool_limits\n\nSEED = 42\nC_VALUES = [0.1, 1.0, 10.0, 30.0, 100.0]\nWORD = dict(lowercase=True, strip_accents=None, stop_words=None,\n            ngram_range=(1, 2), min_df=2, max_df=1.0, max_features=50_000,\n            sublinear_tf=True, norm="l2", dtype=np.float32)\nCHAR = dict(analyzer="char", ngram_range=(3, 5), lowercase=False,\n            min_df=2, max_df=1.0, max_features=200_000,\n            sublinear_tf=True, norm="l2", dtype=np.float32)\nPUNCTUATION = [",", ".", ";", ":", "!", "?", "-", "—", "–", "\'", \'"\',\n               "’", "“", "”", "(", ")", "[", "]", "\\n", "\\t"]\nSTYLE_NAMES = ["log_characters", "log_words", "word_length_mean", "word_length_sd",\n               "word_length_p90", "type_token_ratio", "hapax_ratio", "repeated_bigram_ratio",\n               "repeated_trigram_ratio", "log_sentence_segments", "segment_words_mean",\n               "segment_words_sd", "segment_words_cv", "uppercase_ratio", "digit_ratio",\n               "whitespace_ratio", "non_ascii_ratio"] + [f"frequency_{p!r}" for p in PUNCTUATION]\n\n\n"""Five-fold target-like development of format-invariant sparse detectors.\n\nTarget text supplies only source-membership diagnostics; no target authorship labels\nare accessed. Folds remain balanced by training authorship labels. These stress\nscores do not estimate an unbiased leaderboard score.\n"""\nimport argparse\nfrom collections import defaultdict\nimport gc\nimport hashlib\nfrom importlib.metadata import version\nimport json\nfrom pathlib import Path\nfrom time import perf_counter\nimport unicodedata\n\nimport joblib\nimport numpy as np\nimport pandas as pd\nfrom scipy.sparse import csr_matrix, hstack\nfrom sklearn.feature_extraction.text import TfidfVectorizer\nfrom sklearn.metrics import accuracy_score, log_loss, roc_auc_score\nfrom sklearn.preprocessing import normalize, StandardScaler\nfrom tokenizers import Tokenizer, models, pre_tokenizers, trainers\n\n\nVERSION=\'format_invariant_sparse_v1\'\nCS=[0.1,1.0,10.0,30.0,100.0]\nBPE_CONFIG=dict(vocab_size=4000,min_frequency=2)\nTRANSLATION=str.maketrans({\'’\':"\'",\'‘\':"\'",\'“\':\'"\',\'”\':\'"\',\'—\':\'-\',\'–\':\'-\',\'…\':\'...\'})\n\n\n"""Controlled follow-up to the user-reported 0.90551 format-robust submission.\n\nSelect a limited addition to the incumbent using the same five source-likeness\nfolds. No held-out authorship labels enter any fitted feature operation.\n"""\nimport argparse\nimport gc\nimport hashlib\nfrom importlib.metadata import version\nimport json\nfrom pathlib import Path\nimport shutil\nfrom time import perf_counter\nimport warnings\n\nimport joblib\nimport numpy as np\nimport pandas as pd\nfrom scipy.sparse import csr_matrix,hstack\nfrom sklearn.exceptions import ConvergenceWarning\nfrom sklearn.feature_extraction.text import TfidfVectorizer\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.metrics import roc_auc_score,log_loss,accuracy_score\nfrom sklearn.preprocessing import normalize,StandardScaler,PolynomialFeatures\nfrom threadpoolctl import threadpool_limits\nfrom tokenizers import Tokenizer,models,pre_tokenizers,trainers\n\n\nEXPERIMENT_VERSION=\'format_robust_increment_v2\'\nCS=[10.0,30.0,100.0,300.0,1000.0]\nINCUMBENT=[dict(family=\'clean_nbstyle_0.3\',C=100.0),dict(family=\'bpe_nb\',C=100.0)]\nFAMILIES=[\'clean_nbstyle_0.3\',\'bpe_nb\',\'clean_poly_0.1\',\'clean_weighted_0.5\',\n          \'bpe_weighted_0.5\',\'bpe_long_4000\',\'bpe_long_8000\']\n\n\n"""Paired, five-fold feature-dropout experiment for the existing sparse LR route.\n\nOne clean view and one fixed corrupted view share each original row\'s weight.\nThis is Monte Carlo input augmentation, not per-iteration neural-network dropout.\n"""\nimport argparse\nimport gc\nimport hashlib\nfrom importlib.metadata import version\nimport json\nfrom pathlib import Path\nfrom time import perf_counter\n\nimport joblib\nimport numpy as np\nimport pandas as pd\nfrom scipy.sparse import vstack\nfrom sklearn.metrics import roc_auc_score, log_loss, accuracy_score\n\n\nDROPOUT_VERSION = \'sparse_lr_dropout_v1\'\nDROP_RATES = [0.0, 0.02, 0.05, 0.10]\nSTRESS_RATES = [0.0, 0.05, 0.10, 0.20]\nDROP_FAMILIES = [\'clean_nbstyle_0.3\', \'clean_poly_0.1\', \'bpe_nb\']\nDROP_C = 100.0\nDROP_SEED = 42\nBLENDS = {\n    \'public_route\': {\'clean_nbstyle_0.3\': 0.5, \'bpe_nb\': 0.5},\n    \'v2_route\': {\'clean_nbstyle_0.3\': 0.25, \'clean_poly_0.1\': 0.5, \'bpe_nb\': 0.25},\n}\nMIN_MEAN_GAIN = 0.0005  # A practical adoption gate, not a significance threshold.\n\ndef style_features(texts):\n    rows = []\n    for text in texts:\n        n = max(len(text), 1)\n        words = re.findall(r"\\b[\\w\']+\\b", text.lower())\n        counts = Counter(words)\n        lengths = np.array([len(w) for w in words] or [0], dtype=float)\n        segments = [s for s in re.split(r"[.!?]+", text) if s.strip()]\n        segment_lengths = np.array([len(re.findall(r"\\b[\\w\']+\\b", s)) for s in segments] or [0], dtype=float)\n        repeats = []\n        for k in (2, 3):\n            grams = list(zip(*(words[i:] for i in range(k))))\n            repeats.append(1 - len(set(grams)) / len(grams) if grams else 0.0)\n        rows.append([\n            np.log1p(len(text)), np.log1p(len(words)), lengths.mean(), lengths.std(),\n            np.percentile(lengths, 90), len(counts) / max(len(words), 1),\n            sum(v == 1 for v in counts.values()) / max(len(words), 1), *repeats,\n            np.log1p(len(segments)), segment_lengths.mean(), segment_lengths.std(),\n            segment_lengths.std() / max(segment_lengths.mean(), 1.0),\n            sum(c.isupper() for c in text) / n, sum(c.isdigit() for c in text) / n,\n            sum(c.isspace() for c in text) / n, sum(ord(c) > 127 for c in text) / n,\n            *[text.count(c) / n for c in PUNCTUATION],\n        ])\n    result = np.asarray(rows, dtype=np.float32)\n    assert result.shape == (len(texts), len(STYLE_NAMES)) and np.isfinite(result).all()\n    return result\n\n\ndef canonical(value):\n    return json.dumps(value, sort_keys=True, default=str)\n\n\ndef key(spec):\n    return hashlib.sha256(canonical(spec).encode()).hexdigest()[:16]\n\n\ndef check_prob(p, n):\n    p = np.asarray(p, dtype=float)\n    assert p.shape == (n,) and np.isfinite(p).all() and ((p >= 0) & (p <= 1)).all()\n    return p\n\n\ndef binary_matrix(x):\n    result = x.copy()\n    result.data.fill(1.0)\n    return result\n\n\ndef nb_ratio(binary_x, y, alpha=1.0):\n    positive = np.asarray(binary_x[y == 1].sum(axis=0)).ravel().astype(float) + alpha\n    negative = np.asarray(binary_x[y == 0].sum(axis=0)).ravel().astype(float) + alpha\n    return np.log((positive / positive.sum()) / (negative / negative.sum())).astype(np.float32)\n\ndef clean_text(text):\n    text=unicodedata.normalize(\'NFKC\',text).translate(TRANSLATION).casefold()\n    text=\'\'.join(c for c in text if unicodedata.category(c)!=\'Cf\')\n    return \' \'.join(text.split())\n\n\ndef encode(tokenizer,texts):\n    return [\' \'.join(\'t\'+str(t) for t in e.ids) for e in tokenizer.encode_batch(list(texts))]\n\n\ndef make_folds(fit,source):\n    """Contiguous target-likeness quintiles per class, not random folds."""\n    table=fit[[\'id\',\'label\']].copy()\n    source=source.set_index(\'id\')\n    table[\'target_likeness\']=(table.id.map(source.style_nonlinear)+table.id.map(source.word_text))/2\n    assert table.target_likeness.notna().all()\n    table[\'fold\']=-1\n    for label in [0,1]:\n        ordered=table[table.label.eq(label)].sort_values([\'target_likeness\',\'id\']).index.to_numpy()\n        for fold,indices in enumerate(np.array_split(ordered,5),1):table.loc[indices,\'fold\']=fold\n    folds=[(np.flatnonzero(table.fold.to_numpy()!=f),np.flatnonzero(table.fold.to_numpy()==f)) for f in range(1,6)]\n    return table,folds\n\ndef row_weights(y,likeness,power=.5):\n    # Tempered, clipped density-ratio proxy. Source probabilities need not be\n    # calibrated density ratios; this is an empirical covariate-shift experiment.\n    p=np.clip(np.asarray(likeness,dtype=float),.01,.99)\n    weights=np.clip((p/(1-p)*(10000/3000))**power,.25,4.0)\n    for label in [0,1]:weights[y==label]/=weights[y==label].mean()\n    assert np.isfinite(weights).all() and np.all(weights>0)\n    return weights\n\n\ndef train_classifier(x,y,c,weights=None):\n    model=LogisticRegression(C=c,solver=\'liblinear\',max_iter=5000,tol=1e-4,random_state=42)\n    with threadpool_limits(limits=2),warnings.catch_warnings():\n        warnings.simplefilter(\'error\',ConvergenceWarning)\n        model.fit(x,y,sample_weight=weights)\n    assert np.array_equal(model.classes_,[0,1])\n    return model\n\n\ndef style_poly(bundle,raw_style):\n    z=np.clip(bundle[\'scaler\'].transform(raw_style),-8,8)\n    x=bundle[\'poly\'].transform(z)\n    return np.clip(bundle[\'poly_scaler\'].transform(x),-8,8).astype(np.float32)\n\n\ndef matrices(fit_text,hold_text,y,likeness,families):\n    clean_fit=[clean_text(t) for t in fit_text];clean_hold=[clean_text(t) for t in hold_text]\n    weights=row_weights(y,likeness)\n    if any(f.startswith(\'clean_\') for f in families):\n        word=TfidfVectorizer(**WORD);char=TfidfVectorizer(**CHAR)\n        wf=word.fit_transform(clean_fit);wh=word.transform(clean_hold)\n        cf=char.fit_transform(clean_fit);ch=char.transform(clean_hold)\n        xf=hstack([wf*.5,cf],format=\'csr\');xh=hstack([wh*.5,ch],format=\'csr\')\n        bf=binary_matrix(xf);bh=binary_matrix(xh)\n        ratio=nb_ratio(bf,y);weighted_ratio=nb_ratio(bf.multiply(weights[:,None]).tocsr(),y)\n        nf=normalize(bf.multiply(ratio)).tocsr();nh=normalize(bh.multiply(ratio)).tocsr()\n        raw_sf=style_features(clean_fit);raw_sh=style_features(clean_hold)\n        scaler=StandardScaler().fit(raw_sf);sf=scaler.transform(raw_sf);sh=scaler.transform(raw_sh)\n        base=dict(word=word,char=char,ratio=ratio,scaler=scaler,preprocessing=VERSION)\n        for family in families:\n            if family==\'clean_nbstyle_0.3\':\n                a=hstack([nf,csr_matrix(sf*.3)],format=\'csr\');b=hstack([nh,csr_matrix(sh*.3)],format=\'csr\');bundle=base;sw=None\n            elif family==\'clean_weighted_0.5\':\n                a=normalize(bf.multiply(weighted_ratio)).tocsr();b=normalize(bh.multiply(weighted_ratio)).tocsr()\n                a=hstack([a,csr_matrix(sf*.3)],format=\'csr\');b=hstack([b,csr_matrix(sh*.3)],format=\'csr\')\n                bundle={**base,\'ratio\':weighted_ratio};sw=weights\n            elif family==\'clean_poly_0.1\':\n                poly=PolynomialFeatures(degree=2,include_bias=False)\n                expanded=poly.fit_transform(np.clip(sf,-8,8));ps=StandardScaler().fit(expanded)\n                bundle={**base,\'poly\':poly,\'poly_scaler\':ps}\n                a=hstack([nf,csr_matrix(style_poly(bundle,raw_sf)*.1)],format=\'csr\')\n                b=hstack([nh,csr_matrix(style_poly(bundle,raw_sh)*.1)],format=\'csr\');sw=None\n            else:continue\n            yield family,a,b,bundle,sw\n        del word,char,wf,wh,cf,ch,xf,xh,bf,bh,nf,nh,raw_sf,raw_sh,sf,sh,base,bundle,a,b\n        gc.collect()\n    requested=[f for f in families if f.startswith(\'bpe_\')]\n    for vocab in [4000,8000]:\n        group=[f for f in requested if (8000 if f==\'bpe_long_8000\' else 4000)==vocab]\n        if not group:continue\n        tokenizer=Tokenizer(models.BPE(unk_token=\'[UNK]\'));tokenizer.pre_tokenizer=pre_tokenizers.ByteLevel(add_prefix_space=True)\n        trainer=trainers.BpeTrainer(vocab_size=vocab,min_frequency=2,special_tokens=[\'[UNK]\'],\n                  initial_alphabet=pre_tokenizers.ByteLevel.alphabet(),show_progress=False)\n        tokenizer.train_from_iterator(clean_fit,trainer=trainer)\n        encoded_fit=encode(tokenizer,clean_fit);encoded_hold=encode(tokenizer,clean_hold)\n        for gram in [(1,3),(2,5)]:\n            gram_group=[f for f in group if ((2,5) if f.startswith(\'bpe_long_\') else (1,3))==gram]\n            if not gram_group:continue\n            tfidf=TfidfVectorizer(ngram_range=gram,min_df=2,max_features=250000 if gram==(1,3) else 300000,\n                        lowercase=False,token_pattern=r\'(?u)\\bt\\d+\\b\',sublinear_tf=True,dtype=np.float32)\n            xf=tfidf.fit_transform(encoded_fit);xh=tfidf.transform(encoded_hold)\n            bf=binary_matrix(xf);bh=binary_matrix(xh)\n            ratio=nb_ratio(bf,y)\n            base=dict(tokenizer_json=tokenizer.to_str(),vectorizer=tfidf,ratio=ratio,preprocessing=VERSION)\n            for family in gram_group:\n                sw=weights if family==\'bpe_weighted_0.5\' else None\n                used_ratio=nb_ratio(bf.multiply(weights[:,None]).tocsr(),y) if sw is not None else ratio\n                a=normalize(bf.multiply(used_ratio)).tocsr();b=normalize(bh.multiply(used_ratio)).tocsr()\n                yield family,a,b,{**base,\'ratio\':used_ratio},sw\n            del tfidf,xf,xh,bf,bh,a,b,base\n            gc.collect()\n\ndef dropout_spec(family, rate):\n    return dict(family=family, C=DROP_C, dropout=float(rate))\n\n\ndef dropout_seed(fold, family, offset=0):\n    return DROP_SEED + 1000 * int(fold) + 17 * DROP_FAMILIES.index(family) + offset\n\n\ndef feature_dropout(x, text_columns, rate, seed):\n    """Independent entry dropout on the text block; keep all style columns.\n\n    Inverted scaling preserves each feature\'s expected value. Do not renormalize\n    rows afterward. The same seed gives nested masks for different rates.\n    """\n    if not 0 <= rate < 1:\n        raise ValueError(\'Dropout rate must be in [0, 1).\')\n    if not 0 < text_columns <= x.shape[1]:\n        raise ValueError(\'Invalid text feature boundary.\')\n    result = x.copy().tocsr()\n    if rate == 0:\n        return result\n    eligible = result.indices < text_columns\n    positions = np.flatnonzero(eligible)\n    rng = np.random.default_rng(seed)\n    keep = rng.random(len(positions), dtype=np.float32) >= rate\n    result.data[positions] *= keep / (1.0 - rate)\n    result.eliminate_zeros()\n    assert np.isfinite(result.data).all()\n    return result\n\n\ndef fit_dropout(x, y, text_columns, rate, seed):\n    started = perf_counter()\n    if rate == 0:\n        model = train_classifier(x, y, DROP_C)\n        augmented_loss = log_loss(y, model.predict_proba(x)[:, 1])\n    else:\n        corrupted = feature_dropout(x, text_columns, rate, seed)\n        augmented = vstack([x, corrupted], format=\'csr\')\n        targets = np.concatenate([y, y])\n        # Sum of weights stays equal to the number of original training rows.\n        weights = np.full(len(targets), 0.5)\n        model = train_classifier(augmented, targets, DROP_C, weights)\n        augmented_loss = log_loss(targets, model.predict_proba(augmented)[:, 1], sample_weight=weights)\n        del augmented, corrupted, targets, weights\n    diagnostics = dict(\n        training_seconds=perf_counter() - started,\n        clean_train_log_loss=float(log_loss(y, model.predict_proba(x)[:, 1])),\n        augmented_train_log_loss=float(augmented_loss),\n        solver_iterations=int(np.max(model.n_iter_)),\n        original_training_rows=len(y),\n        effective_sample_weight=float(len(y)),\n    )\n    return model, diagnostics\n\n\ndef feature_matrix(bundle, texts):\n    """Uncorrupted inference features; identical to the preceding v2 route."""\n    clean = [clean_text(t) for t in texts]\n    family = bundle[\'spec\'][\'family\']\n    if family == \'bpe_nb\':\n        tokenizer = Tokenizer.from_str(bundle[\'tokenizer_json\'])\n        x = bundle[\'vectorizer\'].transform(encode(tokenizer, clean))\n        return normalize(binary_matrix(x).multiply(bundle[\'ratio\'])).tocsr()\n    x = hstack([bundle[\'word\'].transform(clean) * .5,\n                  bundle[\'char\'].transform(clean)], format=\'csr\')\n    x = normalize(binary_matrix(x).multiply(bundle[\'ratio\'])).tocsr()\n    raw_style = style_features(clean)\n    if family == \'clean_poly_0.1\':\n        s = style_poly(bundle, raw_style) * .1\n    else:\n        s = bundle[\'scaler\'].transform(raw_style) * .3\n    return hstack([x, csr_matrix(s)], format=\'csr\')\n\n\ndef predict_dropout(bundle, texts):\n    x = feature_matrix(bundle, texts)\n    return check_prob(bundle[\'classifier\'].predict_proba(x)[:, 1], len(texts))\n\n\ndef search_dropout(fit, fold_table, folds, out, max_folds=None):\n    folder = out / \'cv\'\n    folder.mkdir(exist_ok=True)\n    specs = [dropout_spec(f, p) for f in DROP_FAMILIES for p in DROP_RATES]\n    y = fit.label.to_numpy()\n    rows = []\n    oof = {key(s): np.full(len(fit), np.nan) for s in specs}\n    for fold, (idx, hold) in enumerate(folds, 1):\n        if max_folds is not None and fold > max_folds:\n            break\n        needed = []\n        for spec in specs:\n            path = folder / f\'{key(spec)}_{fold}.npz\'\n            if path.is_file():\n                with np.load(path) as saved:\n                    assert np.array_equal(saved[\'fit\'], idx) and np.array_equal(saved[\'hold\'], hold)\n                    p = check_prob(saved[\'p\'], len(hold))\n                    diagnostics = json.loads(saved[\'diagnostics\'].item())\n                oof[key(spec)][hold] = p\n                rows.append(dict(**spec, fold=fold, auc=roc_auc_score(y[hold], p),\n                                 holdout_log_loss=log_loss(y[hold], p), **diagnostics))\n            else:\n                needed.append(spec)\n        if needed:\n            families = [f for f in DROP_FAMILIES if any(s[\'family\'] == f for s in needed)]\n            feature_started = perf_counter()\n            for family, xf, xh, bundle, _ in matrices(\n                    fit.iloc[idx].text, fit.iloc[hold].text, y[idx],\n                    fold_table.target_likeness.to_numpy()[idx], families):\n                print(f\'Features ready: fold={fold}/5, family={family}, shape={xf.shape}, \'\n                      f\'elapsed={perf_counter()-feature_started:.1f}s\', flush=True)\n                columns = len(bundle[\'ratio\'])\n                for spec in [s for s in needed if s[\'family\'] == family]:\n                    model, diagnostics = fit_dropout(xf, y[idx], columns, spec[\'dropout\'], dropout_seed(fold, family))\n                    p = check_prob(model.predict_proba(xh)[:, 1], len(hold))\n                    path = folder / f\'{key(spec)}_{fold}.npz\'\n                    np.savez_compressed(path, fit=idx, hold=hold, p=p,\n                                        diagnostics=json.dumps(diagnostics))\n                    oof[key(spec)][hold] = p\n                    row = dict(**spec, fold=fold, auc=roc_auc_score(y[hold], p),\n                               holdout_log_loss=log_loss(y[hold], p), **diagnostics)\n                    rows.append(row)\n                    pd.DataFrame(rows).to_csv(out / \'model_fold_results.csv\', index=False)\n                    print(f"Completed fold={fold}/5, {family}, dropout={spec[\'dropout\']:.0%}, "\n                          f"AUC={row[\'auc\']:.6f}, loss={row[\'holdout_log_loss\']:.5f}, "\n                          f"fit={diagnostics[\'training_seconds\']:.1f}s", flush=True)\n                    del model\n                del xf, xh, bundle\n                gc.collect()\n        pd.DataFrame(rows).to_csv(out / \'model_fold_results.csv\', index=False)\n    if max_folds is None:\n        for p in oof.values():\n            check_prob(p, len(fit))\n        fit[[\'id\', \'label\']].assign(**oof).to_csv(out / \'oof.csv\', index=False)\n    return pd.DataFrame(rows), oof\n\n\ndef fit_dropout_bundles(frame, likeness, rates, out):\n    folder = out / f\'fitted_{len(frame)}\'\n    folder.mkdir(exist_ok=True)\n    frame_hash = hashlib.sha256(frame.to_json(orient=\'split\', force_ascii=False).encode()).hexdigest()\n    bundles = {}\n    missing = []\n    for family in DROP_FAMILIES:\n        for rate in sorted(set(rates)):\n            spec = dropout_spec(family, rate)\n            path = folder / (key(spec) + \'.joblib\')\n            if path.is_file():\n                bundle = joblib.load(path)\n                assert bundle[\'training_hash\'] == frame_hash and bundle[\'spec\'] == spec\n                bundles[key(spec)] = bundle\n            else:\n                missing.append(spec)\n    if missing:\n        families = [f for f in DROP_FAMILIES if any(s[\'family\'] == f for s in missing)]\n        for family, x, _, features, _ in matrices(frame.text, frame.text.iloc[:1],\n                                                    frame.label.to_numpy(), likeness, families):\n            for spec in [s for s in missing if s[\'family\'] == family]:\n                model, diagnostics = fit_dropout(x, frame.label.to_numpy(), len(features[\'ratio\']),\n                                                spec[\'dropout\'], dropout_seed(len(frame), family))\n                bundle = dict(**features, classifier=model, spec=spec, training_hash=frame_hash,\n                              training_rows=len(frame), diagnostics=diagnostics, dropout_version=DROPOUT_VERSION)\n                # Verify persisted inference applies exactly the fitted representation.\n                direct = model.predict_proba(x[:16])[:, 1]\n                np.testing.assert_allclose(direct, predict_dropout(bundle, frame.text.iloc[:16]), atol=1e-12, rtol=0)\n                path = folder / (key(spec) + \'.joblib\')\n                temporary = path.with_suffix(\'.tmp\')\n                joblib.dump(bundle, temporary, compress=3)\n                temporary.replace(path)\n                bundles[key(spec)] = bundle\n                print(\'Refit\', len(frame), spec, diagnostics, flush=True)\n            del x, features\n            gc.collect()\n    return bundles\n\n"""Frozen GPT-2 statistics; labels never enter feature extraction."""\nimport gc\nimport hashlib\nimport json\nfrom pathlib import Path\nfrom time import perf_counter\n\nimport numpy as np\nimport torch\n\nLM_FEATURE_VERSION = "gpt2_normalized_fulltext_statistics_v1"\nLM_FEATURE_NAMES = [\n    "nll_mean", "nll_std", "nll_q10", "nll_q25", "nll_median", "nll_q75", "nll_q90",\n    "nll_above_5", "nll_above_10", "nll_adjacent_change", "nll_top_decile_mean",\n    "entropy_mean", "entropy_std", "surprisal_gap_mean", "surprisal_gap_std",\n    "standardized_gap_mean", "standardized_gap_std", "conditional_discrepancy",\n    "quarter_1_nll", "quarter_2_nll", "quarter_3_nll", "quarter_4_nll",\n    "last_minus_first_nll", "quarter_nll_std",\n]\n\n\ndef lm_windows(token_ids, context=512, stride=256):\n    """Yield (window, first target offset); each target is scored exactly once."""\n    if not 1 <= stride < context:\n        raise ValueError("Require 1 <= stride < context.")\n    previous_end = 1  # The explicit EOS context token is never a prediction target.\n    for begin in range(0, len(token_ids) - 1, stride):\n        end = min(begin + context, len(token_ids))\n        if end <= previous_end:\n            break\n        first_target = max(1, previous_end - begin)\n        yield token_ids[begin:end], first_target\n        previous_end = end\n        if end == len(token_ids):\n            break\n\n\ndef lm_token_statistics(logits, targets, chunk_size=128):\n    """Float32 reductions, with vocabulary work chunked to limit peak memory."""\n    pieces = []\n    for start in range(0, len(targets), chunk_size):\n        logp = torch.log_softmax(logits[start:start + chunk_size].float(), dim=-1)\n        p = logp.exp()\n        observed = -logp.gather(1, targets[start:start + chunk_size, None]).squeeze(1)\n        entropy = -(p * logp).sum(-1)\n        # Center before squaring to avoid cancellation for near-uniform distributions.\n        variance = (p * (logp + entropy[:, None]).square()).sum(-1).clamp_min(1e-8)\n        pieces.append(torch.stack([observed, entropy, variance], -1).cpu().numpy())\n    return np.concatenate(pieces, axis=0)\n\n\ndef lm_summarize(values):\n    nll, entropy, variance = np.asarray(values, dtype=np.float64).T\n    gap = nll - entropy\n    z = gap / np.sqrt(variance)\n    quarters = [float(x.mean()) if len(x) else float(nll.mean()) for x in np.array_split(nll, 4)]\n    q = np.quantile(nll, [.1, .25, .5, .75, .9])\n    result = [nll.mean(), nll.std(), *q,\n              np.mean(nll > 5), np.mean(nll > 10),\n              np.abs(np.diff(nll)).mean() if len(nll) > 1 else 0.,\n              np.sort(nll)[-max(1, int(np.ceil(len(nll) * .1))):].mean(),\n              entropy.mean(), entropy.std(), gap.mean(), gap.std(), z.mean(), z.std(),\n              -gap.sum() / np.sqrt(variance.sum()), *quarters,\n              quarters[-1] - quarters[0], np.std(quarters)]\n    result = np.asarray(result, dtype=np.float32)\n    assert len(result) == len(LM_FEATURE_NAMES) and np.isfinite(result).all()\n    return result\n\n\ndef lm_extract(texts, tokenizer, model, device, context=512, stride=256, batch_size=4):\n    """No truncation: sliding windows cover every normalized text token."""\n    cleaned = [clean_text(t) for t in texts]\n    if any(not t for t in cleaned):\n        raise ValueError("Empty normalized text cannot be scored.")\n    encoded = tokenizer(cleaned, add_special_tokens=False, truncation=False)["input_ids"]\n    sequences = [[tokenizer.eos_token_id] + ids for ids in encoded]\n    windows = [(row, ids, first) for row, seq in enumerate(sequences)\n               for ids, first in lm_windows(seq, context, stride)]\n    # Sorting reduces padding; per-document results are restored to token order below.\n    ordered = sorted(enumerate(windows), key=lambda item: len(item[1][1]))\n    outputs = [None] * len(windows)\n    model.eval()\n    with torch.inference_mode():\n        for begin in range(0, len(ordered), batch_size):\n            batch = ordered[begin:begin + batch_size]\n            length = max(len(w[1]) for _, w in batch)\n            ids = torch.full((len(batch), length), tokenizer.eos_token_id, dtype=torch.long, device=device)\n            mask = torch.zeros_like(ids)\n            for j, (_, (_, seq, _)) in enumerate(batch):\n                ids[j, :len(seq)] = torch.tensor(seq, dtype=torch.long, device=device)\n                mask[j, :len(seq)] = 1\n            logits = model(input_ids=ids, attention_mask=mask, use_cache=False).logits\n            for j, (window_number, (_, seq, first)) in enumerate(batch):\n                # Logit t predicts token t+1. Right padding and overlap are excluded.\n                outputs[window_number] = lm_token_statistics(\n                    logits[j, first - 1:len(seq) - 1], ids[j, first:len(seq)])\n            del logits, ids, mask\n    values = [[] for _ in sequences]\n    for (row, _, _), stats in zip(windows, outputs):\n        values[row].append(stats)\n    result = []\n    for seq, chunks in zip(sequences, values):\n        stats = np.concatenate(chunks)\n        assert len(stats) == len(seq) - 1, "Missing or repeated target tokens."\n        result.append(lm_summarize(stats))\n    return np.stack(result), np.asarray([len(s) - 1 for s in sequences])\n\n\ndef lm_atomic_npz(path, **values):\n    temporary = Path(str(path) + ".tmp")\n    with temporary.open("wb") as handle:\n        np.savez_compressed(handle, **values)\n    temporary.replace(path)\n\n\ndef lm_cached_extract(frame, name, folder, tokenizer, model, device, config, shard_size=128):\n    """Fingerprint data + model + numerical settings; resume completed shards."""\n    fingerprint = hashlib.sha256(json.dumps(config, sort_keys=True).encode()\n                                 + frame[["id", "text"]].to_json(orient="split").encode()).hexdigest()\n    cache_dir = Path(folder) / f"{name}_{fingerprint[:16]}"\n    cache_dir.mkdir(parents=True, exist_ok=True)\n    result, token_counts = [], []\n    start_time = perf_counter()\n    for start in range(0, len(frame), shard_size):\n        stop = min(start + shard_size, len(frame))\n        path = cache_dir / f"{start:06d}_{stop:06d}.npz"\n        if path.is_file():\n            with np.load(path, allow_pickle=False) as saved:\n                assert saved["fingerprint"].item() == fingerprint\n                assert np.array_equal(saved["ids"], frame.id.iloc[start:stop].astype(str).to_numpy())\n                x, count = saved["features"].copy(), saved["token_count"].copy()\n        else:\n            x, count = lm_extract(frame.text.iloc[start:stop].tolist(), tokenizer, model, device,\n                                  context=config["context"], stride=config["stride"],\n                                  batch_size=config["batch_size"])\n            lm_atomic_npz(path, fingerprint=fingerprint,\n                          ids=frame.id.iloc[start:stop].astype(str).to_numpy(dtype=str),\n                          features=x, token_count=count)\n        assert x.shape == (stop - start, len(LM_FEATURE_NAMES)) and np.isfinite(x).all()\n        result.append(x); token_counts.append(count)\n        print(f"{name}: {stop:,}/{len(frame):,} rows | elapsed {(perf_counter()-start_time)/60:.1f} min", flush=True)\n    return np.vstack(result), np.concatenate(token_counts)\n\n\ndef lm_load(model_name, revision, device, local_dir=None):\n    from transformers import AutoTokenizer, AutoModelForCausalLM\n    source = str(local_dir) if local_dir is not None else model_name\n    kwargs = {} if local_dir is not None else {"revision": revision}\n    tokenizer = AutoTokenizer.from_pretrained(source, trust_remote_code=False, **kwargs)\n    dtype = torch.float16 if device.type == "cuda" else torch.float32\n    model = AutoModelForCausalLM.from_pretrained(\n        source, torch_dtype=dtype, use_safetensors=True, trust_remote_code=False,\n        attn_implementation="eager", **kwargs).to(device).eval()\n    assert model.config.model_type == "gpt2" and tokenizer.eos_token_id is not None\n    model.requires_grad_(False)\n    return tokenizer, model\n\n\ndef s8_dense_pipeline(c):\n    # A small nonlinear feature map, followed by regularized Logistic Regression.\n    from sklearn.pipeline import make_pipeline\n    from sklearn.preprocessing import StandardScaler, PolynomialFeatures\n    from sklearn.linear_model import LogisticRegression\n    return make_pipeline(StandardScaler(), PolynomialFeatures(degree=2, include_bias=False),\n                         StandardScaler(), LogisticRegression(C=c, solver="lbfgs", max_iter=5000,\n                                                               tol=1e-6, random_state=42))\n\n\ndef s8_saved_predict(texts, model_folder, device=None):\n    """Use after executing/importing the helper definitions; accepts arbitrary new text."""\n    import joblib\n    model_folder = Path(model_folder)\n    manifest = json.loads((model_folder / "inference_manifest.json").read_text())\n    branch_prob = []\n    for family, weight in manifest["sparse_weights"].items():\n        bundle = joblib.load(model_folder / manifest["sparse_files"][family])\n        branch_prob.append(weight * predict_dropout(bundle, texts))\n    baseline = np.sum(branch_prob, axis=0)\n    device = torch.device(device or ("cuda" if torch.cuda.is_available() else "cpu"))\n    local_model = model_folder / "pretrained_gpt2"\n    config = manifest["lm_config"]\n    tokenizer, model = lm_load(config["model"], config["revision"], device,\n                               local_model if local_model.is_dir() else None)\n    x, _ = lm_extract(list(texts), tokenizer, model, device, config["context"],\n                      config["stride"], config["batch_size"])\n    dense = joblib.load(model_folder / "lm_logistic.joblib").predict_proba(x)[:, 1]\n    result = {"baseline": baseline, "lm_only": dense}\n    for name in ["experimental", "selected"]:\n        weight = manifest[name + "_lm_weight"]\n        result[name] = (1 - weight) * baseline + weight * dense\n    del model\n    gc.collect()\n    if torch.cuda.is_available():\n        clear_device_cache()\n    return result\n'
    assert hashlib.sha256(BASE_HELPER_SOURCE.encode()).hexdigest()==s8_config['helper_sha256']
    (ROUTE_DIR/'baseline_inference_helpers.py').write_text(BASE_HELPER_SOURCE,encoding='utf-8')
    EMBEDDING_HELPER_SOURCE="def embedding_lr():\n    return make_pipeline(Normalizer(),StandardScaler(),LogisticRegression(\n        C=EMBEDDING_C,solver='lbfgs',max_iter=3000,tol=1e-5,random_state=42))\n\ndef fit_embedding(x,y):\n    model=embedding_lr()\n    with threadpool_limits(limits=2),warnings.catch_warnings():\n        warnings.simplefilter('error',ConvergenceWarning)\n        model.fit(x,y)\n    return model\n\ndef extract_embeddings(texts,tokenizer,model,device,batch_size=8):\n    normalized=[clean_text(t) for t in texts]\n    if any(not t for t in normalized):raise ValueError('Blank normalized text')\n    encoded=tokenizer(normalized,add_special_tokens=False,truncation=False)['input_ids']\n    sequences=[[tokenizer.eos_token_id]+ids for ids in encoded]\n    windows=[(r,ids,first) for r,seq in enumerate(sequences) for ids,first in lm_windows(seq,512,256)]\n    windows.sort(key=lambda w:len(w[1]))\n    totals=np.zeros((len(texts),model.config.n_embd),dtype=np.float64)\n    counts=np.zeros(len(texts),dtype=np.int64)\n    model.eval()\n    with torch.inference_mode():\n        for start in range(0,len(windows),batch_size):\n            batch=windows[start:start+batch_size];length=max(len(seq) for _,seq,_ in batch)\n            ids=torch.full((len(batch),length),tokenizer.eos_token_id,dtype=torch.long,device=device)\n            mask=torch.zeros_like(ids)\n            for j,(_,seq,_) in enumerate(batch):\n                ids[j,:len(seq)]=torch.tensor(seq,device=device);mask[j,:len(seq)]=1\n            hidden=model.transformer(input_ids=ids,attention_mask=mask,use_cache=False).last_hidden_state\n            for j,(r,seq,first) in enumerate(batch):\n                totals[r]+=hidden[j,first:len(seq)].float().sum(0).cpu().numpy()\n                counts[r]+=len(seq)-first\n            del hidden,ids,mask\n    assert np.array_equal(counts,[len(seq)-1 for seq in sequences])\n    result=(totals/counts[:,None]).astype(np.float32)\n    assert np.isfinite(result).all()\n    return result\n"
    (ROUTE_DIR/'embedding_helpers.py').write_text(EMBEDDING_HELPER_SOURCE,encoding='utf-8')
    inference_recipe=dict(baseline_models=str(S9_MODEL_DIR),embedding_model=str(embedding_final_path),
        base_helper='baseline_inference_helpers.py',embedding_helper='embedding_helpers.py',
        frozen_gpt2=s8_lm_config,baseline_weight=.8,embedding_weight=.2,embedding_C=.1,
        note='Import base helpers first; embedding helpers also need make_pipeline, Normalizer and StandardScaler.')
    (ROUTE_DIR/'inference_recipe.json').write_text(json.dumps(inference_recipe,indent=2))
    (ROUTE_DIR/'environment.json').write_text(json.dumps(dict(python=sys.version,
        packages={p:version(p) for p in ['numpy','pandas','scipy','scikit-learn','torch','transformers','tokenizers','joblib']}),indent=2))
    print('Saved recipe, helper sources, and environment.')
else:
    print('Skipped: saved components are used in normal mode.')


### 8.9 Optional Pythia Setup and Benchmark

Enable RUN_PYTHIA_RESEARCH only to reproduce the second-LM experiment. Uses publicly
available Pythia-410M weights (Apache-2.0), frozen full-precision inference, original text,
sliding windows and token-rank statistics. Actual feature extraction previously took 21 minutes
on T4. Cached features skip model loading; library/version changes can create a new cache.


In [ ]:
if RUN_PYTHIA_RESEARCH:
    # A. Model setup and extraction helpers
    import sys,importlib.util,subprocess,time,gc,json,hashlib
    from pathlib import Path
    import numpy as np
    import pandas as pd
    ensure_packages({"torch": "torch", "transformers": "transformers==4.57.1", "safetensors": "safetensors"})
    import torch
    from IPython.display import display
    from importlib.metadata import version
    if importlib.util.find_spec('transformers') is None:
        subprocess.check_call([sys.executable,'-m','pip','install','-q','transformers==4.57.1'])
    from transformers import AutoConfig,AutoTokenizer,AutoModelForCausalLM
    P_MODEL='EleutherAI/pythia-410m'
    P_FEATURE_BUDGET_MIN=20.  # Feature extraction only; download, CV and refits are extra.
    P_ALLOW_LONG=False
    P_BATCH=8
    P_CONTEXT=512
    P_STRIDE=256
    P_DOWNLOAD=False
    P_DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    """Frozen causal-LM feature extraction; token coverage and padding are explicit."""
    import numpy as np
    import torch

    NAMES=['nll_mean','nll_std','nll_q10','nll_median','nll_q90','nll_top10_mean','nll_above5','nll_above10','nll_adjacent_change','entropy_mean','entropy_std','gap_mean','gap_std','standardized_gap_mean','conditional_discrepancy','quarter1_nll','quarter2_nll','quarter3_nll','quarter4_nll','last_minus_first','quarter_std','logrank_mean','logrank_std','logrank_q90','top1_share','top10_share','top100_share','nll_logrank_ratio','logrank_adjacent_change','last_minus_first_logrank']

    def windows(ids,context=512,stride=256):
        assert 1<=stride<context
        end_previous=1
        for begin in range(0,len(ids)-1,stride):
            end=min(begin+context,len(ids))
            if end<=end_previous:break
            yield ids[begin:end],max(1,end_previous-begin)
            end_previous=end
            if end==len(ids):break

    def token_stats(logits,targets,chunk=64):
        parts=[]
        for a in range(0,len(targets),chunk):
            z=logits[a:a+chunk].float(); actual=z.gather(1,targets[a:a+chunk,None]).squeeze(1)
            rank=1+(z>actual[:,None]).sum(1)
            lp=torch.log_softmax(z,dim=-1);p=lp.exp()
            nll=-lp.gather(1,targets[a:a+chunk,None]).squeeze(1)
            entropy=-(p*lp).sum(1)
            variance=(p*(lp+entropy[:,None]).square()).sum(1).clamp_min(1e-8)
            parts.append(torch.stack([nll,entropy,variance,rank.float()],1).cpu().numpy())
        return np.concatenate(parts)

    def summarize(values):
        n,e,v,r=np.asarray(values,dtype=np.float64).T;g=n-e;lr=np.log(r)
        avg=lambda x:float(np.mean(x)) if len(x) else 0.
        quarters=[avg(x) if len(x) else avg(n) for x in np.array_split(n,4)]
        rq=[avg(x) if len(x) else avg(lr) for x in np.array_split(lr,4)]
        out=[avg(n),n.std(),*np.quantile(n,[.1,.5,.9]),avg(np.sort(n)[-max(1,int(np.ceil(len(n)*.1))):]),avg(n>5),avg(n>10),avg(np.abs(np.diff(n))),avg(e),e.std(),avg(g),g.std(),avg(g/np.sqrt(v)),-g.sum()/np.sqrt(v.sum()),*quarters,quarters[-1]-quarters[0],np.std(quarters),avg(lr),lr.std(),np.quantile(lr,.9),avg(r==1),avg(r<=10),avg(r<=100),avg(n)/max(avg(lr),1e-8),avg(np.abs(np.diff(lr))),rq[-1]-rq[0]]
        assert len(out)==len(NAMES) and np.isfinite(out).all()
        return np.asarray(out,dtype=np.float32)

    def extract(texts,tokenizer,model,device,context=512,stride=256,batch=8):
        enc=tokenizer(list(texts),add_special_tokens=False,truncation=False)['input_ids']
        assert all(len(ids)>0 for ids in enc)
        seqs=[[tokenizer.eos_token_id]+ids for ids in enc]
        work=[(r,ids,first) for r,seq in enumerate(seqs) for ids,first in windows(seq,context,stride)]
        ordered=sorted(enumerate(work),key=lambda x:len(x[1][1]));values=[None]*len(work)
        model.eval()
        with torch.inference_mode():
            for a in range(0,len(ordered),batch):
                block=ordered[a:a+batch];length=max(len(w[1]) for _,w in block)
                ids=torch.full((len(block),length),tokenizer.eos_token_id,dtype=torch.long,device=device);mask=torch.zeros_like(ids)
                for j,(_,(_,seq,_)) in enumerate(block):
                    ids[j,:len(seq)]=torch.tensor(seq,device=device);mask[j,:len(seq)]=1
                logits=model(input_ids=ids,attention_mask=mask,use_cache=False).logits
                for j,(idx,(_,seq,first)) in enumerate(block):
                    values[idx]=token_stats(logits[j,first-1:len(seq)-1],ids[j,first:len(seq)])
                del logits,ids,mask
        byrow=[[] for _ in seqs]
        for (row,_,_),v in zip(work,values):byrow[row].append(v)
        result=[]
        for seq,parts in zip(seqs,byrow):
            v=np.concatenate(parts);assert len(v)==len(seq)-1
            result.append(summarize(v))
        return np.stack(result)

    P_ROOT=OUTPUT_DIR/'second_lm';P_ROOT.mkdir(parents=True,exist_ok=True)
    revision_path=P_ROOT/'pythia_410m_revision.json'
    if revision_path.is_file():
        revision=json.loads(revision_path.read_text())['revision']
    else:
        revision=__import__('huggingface_hub').HfApi().model_info(P_MODEL).sha
        assert isinstance(revision,str) and len(revision)==40
        revision_path.write_text(json.dumps(dict(model=P_MODEL,revision=revision),indent=2))
    p_train=pd.read_csv(DATA_DIR/'train.csv');p_test=pd.read_csv(DATA_DIR/'test.csv')
    p_all=pd.concat([p_train[['id','text']],p_test[['id','text']]],ignore_index=True)
    assert len(p_train)==10000 and len(p_test)==3000 and p_all.id.is_unique
    helper_source='"""Frozen causal-LM feature extraction; token coverage and padding are explicit."""\nimport numpy as np\nimport torch\n\nNAMES=[\'nll_mean\',\'nll_std\',\'nll_q10\',\'nll_median\',\'nll_q90\',\'nll_top10_mean\',\'nll_above5\',\'nll_above10\',\'nll_adjacent_change\',\'entropy_mean\',\'entropy_std\',\'gap_mean\',\'gap_std\',\'standardized_gap_mean\',\'conditional_discrepancy\',\'quarter1_nll\',\'quarter2_nll\',\'quarter3_nll\',\'quarter4_nll\',\'last_minus_first\',\'quarter_std\',\'logrank_mean\',\'logrank_std\',\'logrank_q90\',\'top1_share\',\'top10_share\',\'top100_share\',\'nll_logrank_ratio\',\'logrank_adjacent_change\',\'last_minus_first_logrank\']\n\ndef windows(ids,context=512,stride=256):\n    assert 1<=stride<context\n    end_previous=1\n    for begin in range(0,len(ids)-1,stride):\n        end=min(begin+context,len(ids))\n        if end<=end_previous:break\n        yield ids[begin:end],max(1,end_previous-begin)\n        end_previous=end\n        if end==len(ids):break\n\ndef token_stats(logits,targets,chunk=64):\n    parts=[]\n    for a in range(0,len(targets),chunk):\n        z=logits[a:a+chunk].float(); actual=z.gather(1,targets[a:a+chunk,None]).squeeze(1)\n        rank=1+(z>actual[:,None]).sum(1)\n        lp=torch.log_softmax(z,dim=-1);p=lp.exp()\n        nll=-lp.gather(1,targets[a:a+chunk,None]).squeeze(1)\n        entropy=-(p*lp).sum(1)\n        variance=(p*(lp+entropy[:,None]).square()).sum(1).clamp_min(1e-8)\n        parts.append(torch.stack([nll,entropy,variance,rank.float()],1).cpu().numpy())\n    return np.concatenate(parts)\n\ndef summarize(values):\n    n,e,v,r=np.asarray(values,dtype=np.float64).T;g=n-e;lr=np.log(r)\n    avg=lambda x:float(np.mean(x)) if len(x) else 0.\n    quarters=[avg(x) if len(x) else avg(n) for x in np.array_split(n,4)]\n    rq=[avg(x) if len(x) else avg(lr) for x in np.array_split(lr,4)]\n    out=[avg(n),n.std(),*np.quantile(n,[.1,.5,.9]),avg(np.sort(n)[-max(1,int(np.ceil(len(n)*.1))):]),avg(n>5),avg(n>10),avg(np.abs(np.diff(n))),avg(e),e.std(),avg(g),g.std(),avg(g/np.sqrt(v)),-g.sum()/np.sqrt(v.sum()),*quarters,quarters[-1]-quarters[0],np.std(quarters),avg(lr),lr.std(),np.quantile(lr,.9),avg(r==1),avg(r<=10),avg(r<=100),avg(n)/max(avg(lr),1e-8),avg(np.abs(np.diff(lr))),rq[-1]-rq[0]]\n    assert len(out)==len(NAMES) and np.isfinite(out).all()\n    return np.asarray(out,dtype=np.float32)\n\ndef extract(texts,tokenizer,model,device,context=512,stride=256,batch=8):\n    enc=tokenizer(list(texts),add_special_tokens=False,truncation=False)[\'input_ids\']\n    assert all(len(ids)>0 for ids in enc)\n    seqs=[[tokenizer.eos_token_id]+ids for ids in enc]\n    work=[(r,ids,first) for r,seq in enumerate(seqs) for ids,first in windows(seq,context,stride)]\n    ordered=sorted(enumerate(work),key=lambda x:len(x[1][1]));values=[None]*len(work)\n    model.eval()\n    with torch.inference_mode():\n        for a in range(0,len(ordered),batch):\n            block=ordered[a:a+batch];length=max(len(w[1]) for _,w in block)\n            ids=torch.full((len(block),length),tokenizer.eos_token_id,dtype=torch.long,device=device);mask=torch.zeros_like(ids)\n            for j,(_,(_,seq,_)) in enumerate(block):\n                ids[j,:len(seq)]=torch.tensor(seq,device=device);mask[j,:len(seq)]=1\n            logits=model(input_ids=ids,attention_mask=mask,use_cache=False).logits\n            for j,(idx,(_,seq,first)) in enumerate(block):\n                values[idx]=token_stats(logits[j,first-1:len(seq)-1],ids[j,first:len(seq)])\n            del logits,ids,mask\n    byrow=[[] for _ in seqs]\n    for (row,_,_),v in zip(work,values):byrow[row].append(v)\n    result=[]\n    for seq,parts in zip(seqs,byrow):\n        v=np.concatenate(parts);assert len(v)==len(seq)-1\n        result.append(summarize(v))\n    return np.stack(result)\n'
    p_config=dict(version='pythia_statistics_rank_v1',model=P_MODEL,revision=revision,license='Apache-2.0',model_card='https://huggingface.co/EleutherAI/pythia-410m',context=P_CONTEXT,stride=P_STRIDE,batch=P_BATCH,dtype='float32',torch=version('torch'),transformers=version('transformers'),features=NAMES,helper_sha=hashlib.sha256(helper_source.encode()).hexdigest(),train_sha=sha256_file(DATA_DIR/'train.csv'),test_sha=sha256_file(DATA_DIR/'test.csv'),raw_text=True)
    P_DIR=P_ROOT/hashlib.sha256(json.dumps(p_config,sort_keys=True).encode()).hexdigest()[:16]
    P_DIR.mkdir(parents=True,exist_ok=True)
    (P_DIR/'config.json').write_text(json.dumps(p_config,indent=2))
    (P_DIR/'feature_helpers.py').write_text(helper_source)
    p_cached=P_DIR/'features.csv'
    p_tokenizer=None;p_lm=None
    if not p_cached.is_file():
        if P_DEVICE.type!='cuda': print('CPU extraction is supported but can be very slow; the benchmark budget still applies.')
        p_tokenizer=AutoTokenizer.from_pretrained(P_MODEL,revision=revision,trust_remote_code=False)
        p_lm=AutoModelForCausalLM.from_pretrained(P_MODEL,revision=revision,torch_dtype=torch.float32,trust_remote_code=False,use_safetensors=True,attn_implementation='eager').to(P_DEVICE).eval()
        p_lm.requires_grad_(False)
        assert p_lm.config.model_type=='gpt_neox' and p_tokenizer.eos_token_id is not None
        order=np.argsort(p_all.text.str.len().to_numpy(),kind='stable')
        pilot_idx=order[np.linspace(0,len(order)-1,48,dtype=int)]
        extract(p_all.text.iloc[pilot_idx[:4]],p_tokenizer,p_lm,P_DEVICE,P_CONTEXT,P_STRIDE,P_BATCH)
        synchronize_device(P_DEVICE);start=time.perf_counter()
        pilot=extract(p_all.text.iloc[pilot_idx],p_tokenizer,p_lm,P_DEVICE,P_CONTEXT,P_STRIDE,P_BATCH)
        synchronize_device(P_DEVICE);elapsed=time.perf_counter()-start
        p_estimate=elapsed/len(pilot_idx)*len(p_all)/60
        benchmark=dict(pilot_rows=len(pilot_idx),seconds=elapsed,estimated_feature_minutes=p_estimate,excludes='download, CV, final refits, Drive transfer',device=(torch.cuda.get_device_name(0) if P_DEVICE.type == 'cuda' else 'cpu'))
        (P_DIR/'benchmark.json').write_text(json.dumps(benchmark,indent=2));display(pd.DataFrame([benchmark]))
    else:
        p_estimate=0.;print('Feature cache exists; no GPU model loaded.')
    print('Experiment:',P_DIR)
    print('Next cell extracts all 13,000 rows; estimates above 20 minutes stop unless P_ALLOW_LONG=True.')
else:
    print('Skipped: optional experiment is disabled.')


### 8.10 Optional Pythia Full-Text Feature Extraction


In [ ]:
if RUN_PYTHIA_RESEARCH:
    # B. Each token is scored once, with sliding-window context and no truncation.
    if p_cached.is_file():
        p_features=pd.read_csv(p_cached).set_index('id')
    else:
        if p_estimate>P_FEATURE_BUDGET_MIN and not P_ALLOW_LONG:
            raise RuntimeError(f'Estimated feature time {p_estimate:.1f} min exceeds {P_FEATURE_BUDGET_MIN:.0f} min. Review benchmark before enabling P_ALLOW_LONG.')
        shards=P_DIR/'shards';shards.mkdir(exist_ok=True);blocks=[];start=time.perf_counter()
        for a in range(0,len(p_all),128):
            b=min(a+128,len(p_all));path=shards/f'{a:06d}_{b:06d}.npz'
            if path.is_file():
                with np.load(path) as z:
                    assert np.array_equal(z['ids'],p_all.id.iloc[a:b])
                    v=z['features'].copy()
            else:
                v=extract(p_all.text.iloc[a:b],p_tokenizer,p_lm,P_DEVICE,P_CONTEXT,P_STRIDE,P_BATCH)
                tmp=path.with_suffix('.tmp')
                with tmp.open('wb') as h:np.savez_compressed(h,ids=p_all.id.iloc[a:b].to_numpy(),features=v)
                tmp.replace(path)
            assert v.shape==(b-a,len(NAMES)) and np.isfinite(v).all()
            blocks.append(v);print(f'Pythia: {b}/{len(p_all)}, elapsed {(time.perf_counter()-start)/60:.1f} min',flush=True)
        p_features=pd.DataFrame(np.vstack(blocks),index=p_all.id,columns=NAMES)
        p_features.index.name='id';p_features.to_csv(p_cached)
    assert np.array_equal(p_features.index,p_all.id) and p_features.columns.tolist()==NAMES
    assert np.isfinite(p_features.to_numpy()).all()
    if p_lm is not None:
        p_lm.to('cpu');p_lm=None;gc.collect();clear_device_cache()
    print('Frozen Pythia features ready:',p_features.shape)
else:
    print('Skipped: optional experiment is disabled.')


## 9. Controlled Comparisons and Experiment Results

Training development uses 8,000 labeled rows and the original five folds. The same fixed
2,000-row validation set is assessed after model/weight selection. Repeated use makes these
development diagnostics; they are not independent estimates of Kaggle public/private AUROC.
All preprocessing learned from labels/data inside a fold is fitted on that fold's training rows.


### 9.1 Restore the Joint Experiment or Prepare a New Search


In [ ]:
if RUN_JOINT_SEARCH:
    # 10.6 Joint Raw + Normalized GPT-2 Statistics: cached 5-fold experiment
    from pathlib import Path
    import json, hashlib, time, warnings
    import numpy as np
    import pandas as pd
    import sklearn, joblib
    from sklearn.pipeline import make_pipeline
    from sklearn.preprocessing import StandardScaler, PolynomialFeatures
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import roc_auc_score, log_loss
    from sklearn.exceptions import ConvergenceWarning
    from threadpoolctl import threadpool_limits
    from IPython.display import display
    joint_root=PROJECT_DIR
    jr=joint_root/'outputs/raw_gpt2_followup/9259247aa4f3e359'
    jb=joint_root/'outputs/frozen_lm_lr/8b1f60e51c7f454a'
    raw_frame=pd.read_csv(jr/'raw_features.csv').set_index('id')
    norm_frame=pd.read_csv(jb/'frozen_lm_features.csv').set_index('id')
    assert raw_frame.index.is_unique and norm_frame.index.is_unique
    assert raw_frame.columns.tolist()==norm_frame.columns.tolist() and len(raw_frame.columns)==24
    # Quadratic cross-products can connect raw-format and normalized statistics.
    joint_features=pd.concat([raw_frame.add_prefix('raw__'),norm_frame.loc[raw_frame.index].add_prefix('norm__')],axis=1)
    assert np.isfinite(joint_features.to_numpy()).all()
    joof=pd.read_csv(jr/'oof_predictions.csv').set_index('id')
    jfolds=pd.read_csv(jb/'fold_assignments.csv').set_index('id').loc[joof.index]
    assert np.array_equal(joof.label,jfolds.label) and set(jfolds.fold)=={1,2,3,4,5}
    jx=joint_features.loc[joof.index].to_numpy(dtype=np.float64)
    jy=joof.label.to_numpy()
    jcontrol=.7*joof.control.to_numpy()+.3*joof.raw.to_numpy()
    jconfig=dict(version='joint_raw_normalized_polynomial_v1',C_values=[.01,.1,1.,10.,100.],weights=[0.,.1,.2,.3,.4,.5,.7,1.],seed=42,sklearn=sklearn.__version__,control_public_auc=.96719,features={name:hashlib.sha256(path.read_bytes()).hexdigest() for name,path in [('raw',jr/'raw_features.csv'),('normalized',jb/'frozen_lm_features.csv'),('folds',jb/'fold_assignments.csv'),('oof',jr/'oof_predictions.csv')]})
    jdir=joint_root/'outputs/joint_gpt2_statistics'/hashlib.sha256(json.dumps(jconfig,sort_keys=True).encode()).hexdigest()[:16]
    jdir.mkdir(parents=True,exist_ok=True)
    (jdir/'config.json').write_text(json.dumps(jconfig,indent=2))
    def joint_model(c):
        return make_pipeline(StandardScaler(),PolynomialFeatures(2,include_bias=False),StandardScaler(),LogisticRegression(C=c,max_iter=2000,tol=1e-5,solver='lbfgs',random_state=42))
else:
    print('No feature extraction or search: restoring the confirmed joint experiment.')

if not RUN_JOINT_SEARCH:
    jdir = CONFIRMED_JOINT_DIR


### 9.2 Five-Fold Parameter Search and Fusion Comparison


In [ ]:
if RUN_JOINT_SEARCH:
    jrows=[]; joint_oofs={}

    for c in jconfig['C_values']:
        p=np.full(len(jy),np.nan)
        for fold in range(1,6):
            hold=np.flatnonzero(jfolds.fold.to_numpy()==fold)
            fit=np.flatnonzero(jfolds.fold.to_numpy()!=fold)
            cache=jdir/f'C{c:g}_fold{fold}.npz'
            if cache.exists():
                with np.load(cache) as saved:
                    assert np.array_equal(saved['hold'],hold)
                    ph=saved['p']; seconds=float(saved['seconds'])
            else:
                start=time.perf_counter()
                with warnings.catch_warnings():
                    warnings.simplefilter('error',ConvergenceWarning)
                    with threadpool_limits(limits=2):
                        model=joint_model(c).fit(jx[fit],jy[fit])
                        ph=model.predict_proba(jx[hold])[:,1]
                seconds=time.perf_counter()-start
                np.savez_compressed(cache,hold=hold,p=ph,seconds=seconds)
            p[hold]=ph
            auc=roc_auc_score(jy[hold],ph)
            jrows.append(dict(C=c,fold=fold,auc=auc,seconds=seconds))
            print(f'Joint C={c:g}, fold={fold}/5, AUC={auc:.6f}, {seconds:.1f}s',flush=True)
        assert np.isfinite(p).all()
        joint_oofs[c]=p
    jdetails=pd.DataFrame(jrows)
    jsummary=jdetails.groupby('C',as_index=False).agg(mean_auc=('auc','mean'),worst_auc=('auc','min'),seconds=('seconds','sum')).sort_values(['mean_auc','C'],ascending=[False,True])
    jbest_c=float(jsummary.iloc[0].C)
    jp=joint_oofs[jbest_c]
    jblend_rows=[]
    for w in jconfig['weights']:
        for fold in range(1,6):
            mask=jfolds.fold.to_numpy()==fold
            jblend_rows.append(dict(weight=w,fold=fold,auc=roc_auc_score(jy[mask],((1-w)*jcontrol+w*jp)[mask])))
    jbd=pd.DataFrame(jblend_rows)
    jbs=jbd.groupby('weight',as_index=False).agg(mean_auc=('auc','mean'),worst_auc=('auc','min'))
    jbc=jbs[jbs.weight.eq(0)].iloc[0]
    jbf=jbd[jbd.weight.eq(0)].set_index('fold').auc
    jbs['mean_gain']=jbs.mean_auc-jbc.mean_auc
    jbs['worst_gain']=jbs.worst_auc-jbc.worst_auc
    jbs['improved_folds']=[int((jbd[jbd.weight.eq(w)].set_index('fold').auc>jbf).sum()) for w in jbs.weight]
    jbs['eligible']=(jbs.weight>0)&(jbs.mean_gain>=.0005)&(jbs.worst_gain>=0)&(jbs.improved_folds>=3)
    jranked=jbs[jbs.weight.gt(0)].sort_values(['mean_auc','weight'],ascending=[False,True])
    je=jranked[jranked.eligible]
    jexperimental=float((je if len(je) else jranked).iloc[0].weight)
    jselected=jexperimental if len(je) else 0.
    jselection=dict(C=jbest_c,experimental_joint_weight=jexperimental,selected_joint_weight=jselected,passes_cv_gate=bool(len(je)),public_auc=None)
    jdetails.to_csv(jdir/'cv_details.csv',index=False)
    jsummary.to_csv(jdir/'cv_summary.csv',index=False)
    jbs.to_csv(jdir/'blend_cv_summary.csv',index=False)
    (jdir/'selection.json').write_text(json.dumps(jselection,indent=2))
    pd.DataFrame(dict(id=joof.index,label=jy,control=jcontrol,joint=jp)).to_csv(jdir/'oof_predictions.csv',index=False)
    display(jsummary);display(jbs);print('Locked selection:',jselection)
else:

    jselection = read_json(jdir/'selection.json')
    jsummary = pd.read_csv(jdir/'cv_summary.csv')
    jbs = pd.read_csv(jdir/'blend_cv_summary.csv')
    jbest_c = float(jselection['C'])
    jexperimental = float(jselection['experimental_joint_weight'])
    jselected = float(jselection['selected_joint_weight'])
    display(jsummary)
    display(jbs)
    print('Historical CV selection:', jselection)
    print('Confirmed best uses the experimental 10% joint blend (public AUROC 0.96875).')


### 9.3 Fixed Validation: Compare Without Retuning


In [ ]:
if RUN_JOINT_SEARCH:

    jvp=pd.read_csv(jr/'validation_predictions.csv')
    assert set(jvp.id).isdisjoint(joof.index)
    with threadpool_limits(limits=2):
        jdev=joint_model(jbest_c).fit(jx,jy)
        jval=jdev.predict_proba(joint_features.loc[jvp.id].to_numpy(dtype=np.float64))[:,1]
    joblib.dump(jdev,jdir/'joint_lr_8000.joblib',compress=3)
    jmetrics=[]
    for name,w in [('control_096719',0.),('experimental',jexperimental),('selected',jselected)]:
        p=(1-w)*jvp.selected.to_numpy()+w*jval
        jmetrics.append(dict(model=name,joint_weight=w,auc=roc_auc_score(jvp.label,p),log_loss=log_loss(jvp.label,p)))
    jvm=pd.DataFrame(jmetrics);jvm.to_csv(jdir/'validation_metrics.csv',index=False);display(jvm)
    pd.DataFrame(dict(id=jvp.id,label=jvp.label,control=jvp.selected,joint=jval)).to_csv(jdir/'validation_predictions.csv',index=False)
else:
    print('Loading saved validation results; no model fitting.')

if not RUN_JOINT_SEARCH:
    jvm = pd.read_csv(jdir/'validation_metrics.csv')
    display(jvm)


### 9.4 Visual Comparison of the Saved Results


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(jbs['weight'].to_numpy(), jbs['mean_auc'].to_numpy(), marker='o')
axes[0].axvline(.1, color='tab:green', linestyle='--', label='Confirmed best: 10% joint')
axes[0].set(xlabel='Joint-model weight', ylabel='Mean 5-fold AUROC', title='Development CV')
axes[0].legend()
shown = jvm[jvm.model.isin(['control_096719', 'experimental'])]
axes[1].bar(['0.96719 baseline', '10% joint candidate'], shown['auc'].to_numpy(), color=['gray', 'tab:green'])
axes[1].set(ylabel='AUROC', title='Fixed validation (repeatedly reused)')
axes[1].set_ylim(max(0, shown.auc.min()-.001), min(1, shown.auc.max()+.0005))
for i, auc in enumerate(shown['auc']):
    axes[1].text(i, auc, f'{auc:.6f}', ha='center', va='bottom')
for ax in axes:
    ax.grid(axis='y', alpha=.2)
fig.tight_layout()
plt.show()


### 9.5 Structural Tree Features and Cache Fingerprints


In [ ]:
if RUN_TREE_RESEARCH:
    # A. Dependencies, cached features, and original folds (no GPT-2 extraction)
    import importlib.util, subprocess, re, time
    from collections import Counter
    if importlib.util.find_spec('catboost') is None:
        subprocess.check_call([sys.executable,'-m','pip','install','-q','catboost==1.2.8'])
    import catboost
    from catboost import CatBoostClassifier
    from sklearn.metrics import roc_auc_score, log_loss
    TREE_CONFIGS=[dict(depth=d,l2_leaf_reg=l2) for d in [4,6] for l2 in [10.,30.]]
    TREE_WEIGHTS=[0.,.05,.1,.2,.3]
    TREE_DOWNLOAD=False

    def tree_structure(t):
        words=re.findall(r"[A-Za-z]+(?:['’][A-Za-z]+)?",t.lower())
        lines=[l for l in t.splitlines() if l.strip()]; n=max(1,len(lines)); length=max(1,len(t))
        normlines=[' '.join(l.lower().split()) for l in lines]
        grams=list(zip(words,words[1:],words[2:])); counts=Counter(grams)
        repeat=lambda w: 1-len(set(zip(w,w[1:],w[2:])))/max(1,len(w)-2) if len(w)>=3 else 0.
        half=len(words)//2
        return dict(words=len(t.split()),characters=len(t),blank_blocks=len([z for z in re.split(r'\n\s*\n',t) if z.strip()]),
            newlines=t.count('\n'),nonempty_lines=len(lines),hardwrap_ratio=sum(60<=len(l)<=85 for l in lines)/n,
            line_length_std=float(np.std([len(l) for l in lines])) if lines else 0.,
            bullet_ratio=sum(bool(re.match(r'^\s*[-*•]\s+',l)) for l in lines)/n,
            numbered_ratio=sum(bool(re.match(r'^\s*\d+[.)]\s+',l)) for l in lines)/n,
            html_breaks=len(re.findall(r'<br\s*/?>',t,re.I)),bold_markers=t.count('**'),
            repeated_line_ratio=1-len(set(normlines))/n if lines else 0.,
            trigram_repeat=repeat(words),max_trigram_ratio=max(counts.values(),default=0)/max(1,len(grams)),
            repetition_tail_change=repeat(words[half:])-repeat(words[:half]),
            exclamation_ratio=t.count('!')/length,question_ratio=t.count('?')/length,
            curly_quote_ratio=sum(t.count(c) for c in '’‘“”')/length,
            digit_ratio=sum(c.isdigit() for c in t)/length,
            transition_count=len(re.findall(r'\b(overall|in conclusion|furthermore|moreover|in summary)\b',t,re.I)))

    tree_train=pd.read_csv(DATA_DIR/'train.csv');tree_test=pd.read_csv(DATA_DIR/'test.csv')
    tree_all=pd.concat([tree_train[['id','text']],tree_test[['id','text']]],ignore_index=True)
    assert tree_all.id.is_unique
    ids=tree_all.id
    raw=pd.read_csv(RAW_BASELINE_DIR/'raw_features.csv').set_index('id').loc[ids]
    norm=pd.read_csv(BASELINE_DIR/'frozen_lm_features.csv').set_index('id').loc[ids]
    assert raw.columns.tolist()==norm.columns.tolist()
    layout=pd.DataFrame([tree_structure(t) for t in tree_all.text],index=ids)
    tree_features=pd.concat([raw.add_prefix('raw__'),norm.add_prefix('norm__'),(raw-norm).add_prefix('delta__'),layout.add_prefix('layout__')],axis=1)
    assert np.isfinite(tree_features.to_numpy()).all()
    tree_oof_control=pd.read_csv(CONFIRMED_JOINT_DIR/'oof_predictions.csv').set_index('id')
    tree_folds=pd.read_csv(BASELINE_DIR/'fold_assignments.csv').set_index('id').loc[tree_oof_control.index]
    assert np.array_equal(tree_folds.label,tree_oof_control.label) and set(tree_folds.fold)=={1,2,3,4,5}
    tree_y=tree_oof_control.label.to_numpy();tree_x=tree_features.loc[tree_oof_control.index]
    assert len(tree_y)==8000
    assert np.array_equal(tree_train.set_index('id').loc[tree_oof_control.index,'label'],tree_y)
    # Exact 0.96875 route: 90% raw baseline + 10% joint model.
    tree_baseline_oof=.9*tree_oof_control.control.to_numpy()+.1*tree_oof_control.joint.to_numpy()
    tree_config=dict(version='structure_catboost_copy_v1',catboost=catboost.__version__,configs=TREE_CONFIGS,iterations=600,learning_rate=.04,seed=42,weights=TREE_WEIGHTS,
        feature_hash=hashlib.sha256(pd.util.hash_pandas_object(tree_features,index=True).values.tobytes()).hexdigest(),
        train_hash=sha256_file(DATA_DIR/'train.csv'),folds_hash=sha256_file(BASELINE_DIR/'fold_assignments.csv'),control_oof_hash=sha256_file(CONFIRMED_JOINT_DIR/'oof_predictions.csv'),
        feature_names=tree_features.columns.tolist(),best_submission_sha=COMPONENT_BASELINE['sha256'])
    TREE_DIR=OUTPUT_DIR/'structure_catboost'/hashlib.sha256(json.dumps(tree_config,sort_keys=True).encode()).hexdigest()[:16]
    TREE_DIR.mkdir(parents=True,exist_ok=True)
    (TREE_DIR/'config.json').write_text(json.dumps(tree_config,indent=2))
    def tree_model(spec):
        return CatBoostClassifier(**spec,iterations=600,learning_rate=.04,loss_function='Logloss',random_seed=42,thread_count=2,verbose=False,allow_writing_files=False)
    print('Features:',tree_features.shape,'| Experiment:',TREE_DIR)
    print('Next: 4 configurations x 5 folds. Fixed 600 rounds; CPU, no early stopping on holdout labels.')
else:
    print('Skipped: optional experiment is disabled.')


### 9.6 CatBoost Five-Fold Search and Initial Fusion


In [ ]:
if RUN_TREE_RESEARCH:
    # B. Five-fold search, followed by OOF fusion selection
    rows=[];tree_oofs={}
    for k,spec in enumerate(TREE_CONFIGS):
        pred=np.full(len(tree_y),np.nan)
        for fold in range(1,6):
            hold=np.flatnonzero(tree_folds.fold.to_numpy()==fold);fit=np.flatnonzero(tree_folds.fold.to_numpy()!=fold)
            cache=TREE_DIR/f'config{k}_fold{fold}.npz'
            if cache.is_file():
                with np.load(cache) as z:
                    assert np.array_equal(z['hold'],hold)
                    p=z['p'];seconds=float(z['seconds'])
            else:
                started=time.perf_counter();model=tree_model(spec).fit(tree_x.iloc[fit],tree_y[fit])
                p=model.predict_proba(tree_x.iloc[hold])[:,1];seconds=time.perf_counter()-started
                temporary=cache.with_suffix('.tmp')
                with temporary.open('wb') as handle: np.savez_compressed(handle,hold=hold,p=p,seconds=seconds)
                temporary.replace(cache)
            pred[hold]=p;auc=roc_auc_score(tree_y[hold],p)
            rows.append(dict(config=k,fold=fold,auc=auc,seconds=seconds))
            print(f'Config {k+1}/4 {spec}, fold {fold}/5, AUROC={auc:.6f}, {seconds:.1f}s',flush=True)
        assert np.isfinite(pred).all();tree_oofs[k]=pred
    cv=pd.DataFrame(rows);cv.to_csv(TREE_DIR/'cv_folds.csv',index=False)
    cv_summary=cv.groupby('config',as_index=False).agg(mean_auc=('auc','mean'),worst_auc=('auc','min'),seconds=('seconds','sum')).sort_values(['mean_auc','config'],ascending=[False,True])
    tree_best_k=int(cv_summary.iloc[0]['config']);tree_best_oof=tree_oofs[tree_best_k]
    rows=[]
    for w in TREE_WEIGHTS:
        p=(1-w)*tree_baseline_oof+w*tree_best_oof
        for fold in range(1,6):
            mask=tree_folds.fold.to_numpy()==fold
            rows.append(dict(weight=w,fold=fold,auc=roc_auc_score(tree_y[mask],p[mask])))
    blend_folds=pd.DataFrame(rows);blend=blend_folds.groupby('weight',as_index=False).agg(mean_auc=('auc','mean'),worst_auc=('auc','min'))
    b=blend[blend.weight.eq(0)].iloc[0];bf=blend_folds[blend_folds.weight.eq(0)].set_index('fold').auc
    blend['mean_gain']=blend.mean_auc-b.mean_auc;blend['worst_gain']=blend.worst_auc-b.worst_auc
    blend['improved_folds']=[int((blend_folds[blend_folds.weight.eq(w)].set_index('fold').auc>bf).sum()) for w in blend.weight]
    blend['eligible']=(blend.weight>0)&(blend.mean_gain>=.0005)&(blend.worst_gain>=0)&(blend.improved_folds>=3)
    ranked=blend[blend.weight.gt(0)].sort_values(['mean_auc','weight'],ascending=[False,True]);eligible=ranked[ranked.eligible]
    tree_experimental_weight=float((eligible if len(eligible) else ranked).iloc[0].weight)
    tree_selected_weight=tree_experimental_weight if len(eligible) else 0.
    tree_selection=dict(config=TREE_CONFIGS[tree_best_k],experimental_tree_weight=tree_experimental_weight,selected_tree_weight=tree_selected_weight,passes_cv_gate=bool(len(eligible)),public_auc=None)
    cv_summary.to_csv(TREE_DIR/'cv_summary.csv',index=False);blend.to_csv(TREE_DIR/'blend_cv_summary.csv',index=False)
    (TREE_DIR/'selection.json').write_text(json.dumps(tree_selection,indent=2))
    pd.DataFrame(dict(id=tree_oof_control.index,label=tree_y,control=tree_baseline_oof,tree=tree_best_oof)).to_csv(TREE_DIR/'oof_predictions.csv',index=False)
    display(cv_summary);display(blend);print('Locked selection:',tree_selection)
else:
    print('Skipped: optional experiment is disabled.')


### 9.7 Fixed Validation and Feature Importance


In [ ]:
if RUN_TREE_RESEARCH:
    # C. Fixed validation after locking parameters; never retune below
    import joblib
    raw_val=pd.read_csv(RAW_BASELINE_DIR/'validation_predictions.csv').set_index('id')
    joint_val=pd.read_csv(CONFIRMED_JOINT_DIR/'validation_predictions.csv').set_index('id').loc[raw_val.index]
    assert np.array_equal(raw_val.label,joint_val.label) and set(raw_val.index).isdisjoint(tree_oof_control.index)
    val_control=.9*raw_val.selected.to_numpy()+.1*joint_val.joint.to_numpy()
    tree_dev=tree_model(TREE_CONFIGS[tree_best_k]).fit(tree_x,tree_y)
    tree_val=tree_dev.predict_proba(tree_features.loc[raw_val.index])[:,1]
    joblib.dump(tree_dev,TREE_DIR/'tree_8000.joblib',compress=3)
    rows=[]
    for name,w in [('control_096875',0.),('experimental',tree_experimental_weight),('selected',tree_selected_weight)]:
        p=(1-w)*val_control+w*tree_val
        rows.append(dict(model=name,tree_weight=w,auc=roc_auc_score(raw_val.label,p),log_loss=log_loss(raw_val.label,p)))
    tree_validation=pd.DataFrame(rows);tree_validation.to_csv(TREE_DIR/'validation_metrics.csv',index=False)
    pd.DataFrame(dict(id=raw_val.index,label=raw_val.label,control=val_control,tree=tree_val)).to_csv(TREE_DIR/'validation_predictions.csv',index=False)
    importance=pd.DataFrame(dict(feature=tree_features.columns,importance=tree_dev.feature_importances_)).sort_values('importance',ascending=False)
    importance.to_csv(TREE_DIR/'feature_importance.csv',index=False)
    display(tree_validation);display(importance.head(20))
    import matplotlib.pyplot as plt
    fig,axes=plt.subplots(1,2,figsize=(12,4))
    axes[0].plot(blend.weight.to_numpy(),blend.mean_auc.to_numpy(),marker='o')
    axes[0].set(xlabel='Tree weight',ylabel='Mean fold AUROC',title='Development OOF fusion')
    importance.head(12).iloc[::-1].plot.barh(x='feature',y='importance',ax=axes[1],legend=False)
    axes[1].set_title('Feature importance (descriptive, not causal)')
    fig.tight_layout();fig.savefig(TREE_DIR/'tree_comparison.png',dpi=160,bbox_inches='tight');plt.show()
    print('Repeated validation is diagnostic. No weight changes after inspecting these results.')
else:
    print('Skipped: optional experiment is disabled.')


### 9.8 Restore the Measured Tree and Weight Comparisons

At 30% tree weight, public AUROC was 0.97988. Extending to 60% produced 0.98270.
The old conservative mean-CV improvement gate was not passed by the weight extension;
that historical result is retained. Scores below are cached development results.


In [ ]:
tree_cv=pd.read_csv(TREE_SOURCE_DIR/'cv_summary.csv')
initial_blend_cv=pd.read_csv(TREE_SOURCE_DIR/'blend_cv_summary.csv')
weight_cv=pd.read_csv(WEIGHT_SOURCE_DIR/'cv_summary.csv')
weight_validation=pd.read_csv(WEIGHT_SOURCE_DIR/'validation_metrics.csv')
display(tree_cv);display(weight_cv);display(weight_validation)
import matplotlib.pyplot as plt
fig,ax=plt.subplots(figsize=(7,4))
ax.plot(weight_cv.weight.to_numpy(),weight_cv.mean_auc.to_numpy(),marker='o')
ax.axvline(.6,color='tab:green',linestyle='--',label='Scored best: 60% tree')
ax.set(xlabel='CatBoost weight',ylabel='Mean five-fold AUROC',title='Saved fusion comparison')
ax.legend();ax.grid(alpha=.2);fig.tight_layout();plt.show()


### 9.9 Optional Weight Extension: Saved Predictions Only


In [ ]:
if RUN_WEIGHT_COMPARISON:
    # 10.6 Extend CatBoost Fusion Weights — No Training or Feature Extraction
    from pathlib import Path
    import json, hashlib
    import numpy as np
    import pandas as pd
    from sklearn.metrics import roc_auc_score, log_loss
    from IPython.display import display
    extension_root = Path(PROJECT_DIR)
    source_dir=extension_root/'outputs/structure_catboost/404ce0b7b71319b1'
    base_dir=extension_root/'outputs/frozen_lm_lr/8b1f60e51c7f454a'
    def extension_sha(path): return hashlib.sha256(Path(path).read_bytes()).hexdigest()
    source_selection=json.loads((source_dir/'selection.json').read_text())
    assert source_selection['selected_tree_weight']==.3
    source_record=next(r for r in json.loads((source_dir/'submission_manifest.json').read_text()) if r['variant']=='selected')
    scored_path=source_dir/'submission_tree_selected.csv'
    assert extension_sha(scored_path)==source_record['sha256']
    weights=[.3,.35,.4,.5,.6,.8,1.]
    config=dict(version='tree_weight_extension_v1',control_public_auc=.97988,source_sha=source_record['sha256'],weights=weights,
        oof_sha=extension_sha(source_dir/'oof_predictions.csv'),folds_sha=extension_sha(base_dir/'fold_assignments.csv'))
    ext_dir=extension_root/'outputs/tree_weight_extension_097988'/hashlib.sha256(json.dumps(config,sort_keys=True).encode()).hexdigest()[:16]
    ext_dir.mkdir(parents=True,exist_ok=True)
    (ext_dir/'config.json').write_text(json.dumps(config,indent=2))
    (ext_dir/'control_097988.csv').write_bytes(scored_path.read_bytes())
    oof=pd.read_csv(source_dir/'oof_predictions.csv').set_index('id')
    folds=pd.read_csv(base_dir/'fold_assignments.csv').set_index('id').loc[oof.index]
    assert oof.index.is_unique and len(oof)==8000 and np.array_equal(oof.label,folds.label)
    assert set(folds.fold)=={1,2,3,4,5} and np.isfinite(oof[['control','tree']].to_numpy()).all()
    rows=[]
    for w in weights:
        p=(1-w)*oof.control.to_numpy()+w*oof.tree.to_numpy()
        for fold in range(1,6):
            mask=folds.fold.to_numpy()==fold
            rows.append(dict(weight=w,fold=fold,auc=roc_auc_score(oof.label.to_numpy()[mask],p[mask])))
    details=pd.DataFrame(rows)
    summary=details.groupby('weight',as_index=False).agg(mean_auc=('auc','mean'),worst_auc=('auc','min'))
    control=summary[summary.weight.eq(.3)].iloc[0]
    cf=details[details.weight.eq(.3)].set_index('fold').auc
    summary['mean_gain']=summary.mean_auc-control.mean_auc
    summary['worst_gain']=summary.worst_auc-control.worst_auc
    summary['improved_folds']=[int((details[details.weight.eq(w)].set_index('fold').auc>cf).sum()) for w in summary.weight]
    summary['eligible']=(summary.weight>.3)&(summary.mean_gain>=.0005)&(summary.worst_gain>=0)&(summary.improved_folds>=3)
    ranked=summary[summary.weight.gt(.3)].sort_values(['mean_auc','weight'],ascending=[False,True])
    eligible=ranked[ranked.eligible]
    experimental_w=float((eligible if len(eligible) else ranked).iloc[0].weight)
    selected_w=experimental_w if len(eligible) else .3
    selection=dict(experimental_tree_weight=experimental_w,selected_tree_weight=selected_w,passes_cv_gate=bool(len(eligible)),public_auc=None)
    summary.to_csv(ext_dir/'cv_summary.csv',index=False);details.to_csv(ext_dir/'cv_details.csv',index=False)
    (ext_dir/'selection.json').write_text(json.dumps(selection,indent=2))
    display(summary);print('Locked selection:',selection)
    # Fixed validation is inspected after selection; no parameter changes below.
    vp=pd.read_csv(source_dir/'validation_predictions.csv')
    assert vp.id.is_unique and set(vp.id).isdisjoint(oof.index)
    metrics=[]
    for name,w in [('control_097988',.3),('experimental',experimental_w),('selected',selected_w)]:
        p=(1-w)*vp.control.to_numpy()+w*vp.tree.to_numpy()
        metrics.append(dict(model=name,tree_weight=w,auc=roc_auc_score(vp.label,p),log_loss=log_loss(vp.label,p)))
    validation=pd.DataFrame(metrics);validation.to_csv(ext_dir/'validation_metrics.csv',index=False);display(validation)
    # Recover exact tree test probabilities from the previously scored 70/30 blend.
    # This avoids loading models, recomputation, or floating-point refit differences.
    old_path=extension_root/'outputs/best_096875/submission_best_096875.csv'
    old_manifest=json.loads((old_path.parent/'manifest.json').read_text())
    assert extension_sha(old_path)==old_manifest['sha256']
    old=pd.read_csv(old_path);scored=pd.read_csv(scored_path)
    data_dir=next(extension_root/d for d in ['data','Data'] if (extension_root/d/'sample_submission.csv').is_file())
    template=pd.read_csv(data_dir/'sample_submission.csv')
    assert np.array_equal(old.id,template.id) and np.array_equal(scored.id,template.id)
    tree_test=(scored.label.to_numpy()-.7*old.label.to_numpy())/.3
    assert np.isfinite(tree_test).all() and tree_test.min()>=-1e-10 and tree_test.max()<=1+1e-10
    tree_test=np.clip(tree_test,0,1)
    exports=[]
    for name,w in [('experimental',experimental_w),('selected',selected_w)]:
        path=ext_dir/f'submission_tree_weight_{name}.csv'
        if w==.3: path.write_bytes(scored_path.read_bytes())
        else:
            out=template[['id']].copy();out['label']=(1-w)*old.label.to_numpy()+w*tree_test;out.to_csv(path,index=False)
        out=pd.read_csv(path)
        assert out.columns.tolist()==['id','label'] and len(out)==3000 and out.id.is_unique
        assert np.array_equal(out.id,template.id) and np.isfinite(out.label).all() and out.label.between(0,1).all()
        exports.append(dict(variant=name,path=str(path),tree_weight=w,sha256=extension_sha(path),public_auc=None))
    (ext_dir/'submission_manifest.json').write_text(json.dumps(exports,indent=2))
    display(pd.DataFrame(exports));print('Files:',ext_dir)
    print('New selected candidate passed CV; public score unknown.' if selected_w!=.3 else 'Gate failed: selected is the exact 0.97988 file; do not resubmit it.')
    print('Repeated CV and validation are development diagnostics, not independent public-score estimates.')
else:
    print('Skipped: optional experiment is disabled.')


### 9.10 Optional Configuration Averaging: Negative Follow-Up

The highest-mean candidate improved CV AUROC by only about 0.000014 and reduced
worst-fold performance. It did not replace the 0.98270 baseline or generate a new submission.
Enable only to reproduce that experiment; model refits may run if its positive-signal condition passes.


In [ ]:
if RUN_AVERAGING:
    # 10.7 CatBoost Configuration Averaging — Reuse Existing OOF Caches
    from pathlib import Path
    import json,hashlib,time
    import numpy as np
    import pandas as pd
    from sklearn.metrics import roc_auc_score,log_loss
    from catboost import CatBoostClassifier
    from IPython.display import display
    root=Path(PROJECT_DIR)
    source=root/'outputs/structure_catboost/404ce0b7b71319b1'
    weight_dir=root/'outputs/tree_weight_extension_097988/2e838e8ef8a0d53c'
    scored=weight_dir/'submission_tree_weight_experimental.csv'
    sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
    scored_record=next(r for r in json.loads((weight_dir/'submission_manifest.json').read_text()) if r['variant']=='experimental')
    assert scored_record['tree_weight']==.6 and sha(scored)==scored_record['sha256']
    source_config=json.loads((source/'config.json').read_text())
    specs=source_config['configs']
    oof=pd.read_csv(source/'oof_predictions.csv').set_index('id')
    folds=pd.read_csv(root/'outputs/frozen_lm_lr/8b1f60e51c7f454a/fold_assignments.csv').set_index('id').loc[oof.index]
    assert np.array_equal(oof.label,folds.label)
    y=oof.label.to_numpy();component_oofs={}
    for k in range(4):
        p=np.full(len(y),np.nan)
        for fold in range(1,6):
            hold=np.flatnonzero(folds.fold.to_numpy()==fold)
            with np.load(source/f'config{k}_fold{fold}.npz') as z:
                assert np.array_equal(z['hold'],hold)
                p[hold]=z['p']
        assert np.isfinite(p).all();component_oofs[k]=p
    np.testing.assert_allclose(component_oofs[2],oof.tree,atol=1e-12,rtol=0)
    # Configuration 2 is depth=6, l2=10, the incumbent individual tree model.
    sets={'incumbent':[2],'depth_diverse':[0,2],'regularization_diverse':[2,3],'depth_regularization_diverse':[1,2],'all_four':[0,1,2,3]}
    weights=[.4,.5,.6,.7,.8]
    config=dict(version='catboost_average_v1',source_config_sha=sha(source/'config.json'),source_oof_sha=sha(source/'oof_predictions.csv'),control_sha=sha(scored),control_public_auc=.98270,sets=sets,weights=weights)
    folder=root/'outputs/catboost_averaging'/hashlib.sha256(json.dumps(config,sort_keys=True).encode()).hexdigest()[:16]
    folder.mkdir(parents=True,exist_ok=True)
    (folder/'config.json').write_text(json.dumps(config,indent=2));(folder/'control_098270.csv').write_bytes(scored.read_bytes())
    base=.4*oof.control.to_numpy()+.6*oof.tree.to_numpy()
    base_folds=np.array([roc_auc_score(y[folds.fold.to_numpy()==f],base[folds.fold.to_numpy()==f]) for f in range(1,6)])
    rows=[]
    for name,indices in sets.items():
        pt=np.mean([component_oofs[k] for k in indices],axis=0)
        for w in weights:
            p=(1-w)*oof.control.to_numpy()+w*pt
            aucs=np.array([roc_auc_score(y[folds.fold.to_numpy()==f],p[folds.fold.to_numpy()==f]) for f in range(1,6)])
            rows.append(dict(ensemble=name,weight=w,mean_auc=aucs.mean(),worst_auc=aucs.min(),mean_gain=(aucs-base_folds).mean(),worst_gain=aucs.min()-base_folds.min(),improved_folds=int((aucs>base_folds).sum())))
    results=pd.DataFrame(rows).sort_values(['mean_auc','weight','ensemble'],ascending=[False,True,True])
    results.to_csv(folder/'cv_summary.csv',index=False);display(results)
    choices=results[~(results.ensemble.eq('incumbent')&results.weight.eq(.6))]
    best=choices.iloc[0];name=str(best.ensemble);w=float(best.weight);indices=sets[name]
    positive=bool(best.mean_gain>0 and best.worst_gain>=0 and best.improved_folds>=3)
    selection=dict(ensemble=name,components=indices,tree_weight=w,positive_cv_signal=positive,passes_conservative_gate=bool(positive and best.mean_gain>=.0005),public_auc=None)
    (folder/'selection.json').write_text(json.dumps(selection,indent=2));print('Locked candidate:',selection)
    if not positive:
        print('No consistent positive CV signal. No new final models or submission are generated.')
    else:
        # Everything below follows the locked CV choice. Validation never changes it.
        assert 'tree_features' in globals(), 'Run 10.5 A first to restore the cached feature table.'
        actual_hash=hashlib.sha256(pd.util.hash_pandas_object(tree_features,index=True).values.tobytes()).hexdigest()
        assert actual_hash==source_config['feature_hash'], 'Feature table changed.'
        assert sha(DATA_DIR/'train.csv')==source_config['train_hash']
        train=pd.read_csv(DATA_DIR/'train.csv').set_index('id')
        vp=pd.read_csv(source/'validation_predictions.csv').set_index('id')
        assert set(vp.index).isdisjoint(oof.index) and np.array_equal(vp.label,train.loc[vp.index,'label'])
        assert np.array_equal(y,train.loc[oof.index,'label'])
        def average_model(k):
            return CatBoostClassifier(**specs[k],iterations=600,learning_rate=.04,loss_function='Logloss',random_seed=42,thread_count=2,verbose=False,allow_writing_files=False)
        valparts=[];testparts=[]
        template=pd.read_csv(DATA_DIR/'sample_submission.csv')
        for k in indices:
            start=time.perf_counter()
            dev=average_model(k).fit(tree_features.loc[oof.index],y)
            valparts.append(dev.predict_proba(tree_features.loc[vp.index])[:,1])
            dev.save_model(str(folder/f'config{k}_8000.cbm'))
            final=average_model(k)
            if k==2:
                final.load_model(str(source/'tree_10000.cbm'))
            else:
                final.fit(tree_features.loc[train.index],train.label.to_numpy())
            final.save_model(str(folder/f'config{k}_10000.cbm'))
            testparts.append(final.predict_proba(tree_features.loc[template.id])[:,1])
            print(f'Component {k} ready: {(time.perf_counter()-start):.1f}s',flush=True)
        pval=(1-w)*vp.control.to_numpy()+w*np.mean(valparts,axis=0)
        baseline_val=.4*vp.control.to_numpy()+.6*vp.tree.to_numpy()
        metrics=pd.DataFrame([dict(model='control_098270',auc=roc_auc_score(vp.label,baseline_val),log_loss=log_loss(vp.label,baseline_val)),dict(model='experimental',auc=roc_auc_score(vp.label,pval),log_loss=log_loss(vp.label,pval))])
        metrics.to_csv(folder/'validation_metrics.csv',index=False);display(metrics)
        old=root/'outputs/best_096875/submission_best_096875.csv'
        old_record=json.loads((old.parent/'manifest.json').read_text())
        assert sha(old)==old_record['sha256']
        control=pd.read_csv(old);assert np.array_equal(control.id,template.id)
        output=template[['id']].copy();output['label']=(1-w)*control.label.to_numpy()+w*np.mean(testparts,axis=0)
        assert len(output)==3000 and output.id.is_unique and np.isfinite(output.label).all() and output.label.between(0,1).all()
        path=folder/'submission_catboost_average_experimental.csv';output.to_csv(path,index=False)
        (folder/'submission_manifest.json').write_text(json.dumps(dict(path=str(path),sha256=sha(path),selection=selection,public_auc=None),indent=2))
        print('Experimental candidate:',path)
        print('Public score unknown. The exact 0.98270 control remains preserved.')
    print('Experiment folder:',folder)
else:
    print('Skipped: optional experiment is disabled.')


### 9.11 Optional Pythia Classifier Search and Fusion


In [ ]:
if RUN_PYTHIA_RESEARCH:
    # C. 3 LR configurations + 1 tree configuration, each on the same 5 folds.
    from sklearn.pipeline import make_pipeline
    from sklearn.preprocessing import StandardScaler,PolynomialFeatures
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import roc_auc_score,log_loss
    from sklearn.exceptions import ConvergenceWarning
    from threadpoolctl import threadpool_limits
    import warnings,joblib,sklearn
    if importlib.util.find_spec('catboost') is None:
        subprocess.check_call([sys.executable,'-m','pip','install','-q','catboost==1.2.8'])
    import catboost
    from catboost import CatBoostClassifier
    source=OUTPUT_DIR/'structure_catboost/404ce0b7b71319b1'
    weight_dir=OUTPUT_DIR/'tree_weight_extension_097988/2e838e8ef8a0d53c'
    control_csv=weight_dir/'submission_tree_weight_experimental.csv'
    rec=next(r for r in read_json(weight_dir/'submission_manifest.json') if r['variant']=='experimental')
    assert rec['tree_weight']==.6 and sha256_file(control_csv)==rec['sha256']
    oof=pd.read_csv(source/'oof_predictions.csv').set_index('id')
    folds=pd.read_csv(BASELINE_DIR/'fold_assignments.csv').set_index('id').loc[oof.index]
    y=oof.label.to_numpy();baseline=.4*oof.control.to_numpy()+.6*oof.tree.to_numpy()
    assert len(y)==8000 and np.array_equal(y,folds.label) and set(folds.fold)=={1,2,3,4,5}
    raw=pd.read_csv(RAW_BASELINE_DIR/'raw_features.csv').set_index('id')
    norm=pd.read_csv(BASELINE_DIR/'frozen_lm_features.csv').set_index('id')
    # No direct NLL ratios across models with different tokenizers.
    views={'pythia':p_features,'joint':pd.concat([p_features.add_prefix('pythia__'),raw.loc[p_features.index].add_prefix('gpt2_raw__'),norm.loc[p_features.index].add_prefix('gpt2_norm__')],axis=1)}
    specs=[dict(kind='lr',C=c,view='pythia') for c in [.01,.1,1.]]+[dict(kind='tree',view='joint')]
    training_config=dict(feature_config=p_config,sklearn=sklearn.__version__,catboost=catboost.__version__,specs=specs,fold_sha=sha256_file(BASELINE_DIR/'fold_assignments.csv'),control_oof_sha=sha256_file(source/'oof_predictions.csv'),control_sha=rec['sha256'],raw_sha=sha256_file(RAW_BASELINE_DIR/'raw_features.csv'),norm_sha=sha256_file(BASELINE_DIR/'frozen_lm_features.csv'))
    P_RUN=P_DIR/'classifiers'/hashlib.sha256(json.dumps(training_config,sort_keys=True).encode()).hexdigest()[:16]
    P_RUN.mkdir(parents=True,exist_ok=True);(P_RUN/'config.json').write_text(json.dumps(training_config,indent=2));(P_RUN/'control_098270.csv').write_bytes(control_csv.read_bytes())
    def p_classifier(spec):
        if spec['kind']=='lr':return make_pipeline(StandardScaler(),PolynomialFeatures(2,include_bias=False),StandardScaler(),LogisticRegression(C=spec['C'],solver='lbfgs',max_iter=3000,tol=1e-5,random_state=42))
        return CatBoostClassifier(depth=4,l2_leaf_reg=10.,iterations=600,learning_rate=.04,loss_function='Logloss',random_seed=42,thread_count=2,verbose=False,allow_writing_files=False)
    rows=[];predictions={}
    for k,spec in enumerate(specs):
        x=views[spec['view']].loc[oof.index].to_numpy();p=np.full(len(y),np.nan)
        for fold in range(1,6):
            fit=np.flatnonzero(folds.fold.to_numpy()!=fold);hold=np.flatnonzero(folds.fold.to_numpy()==fold);cache=P_RUN/f'config{k}_fold{fold}.npz'
            if cache.is_file():
                with np.load(cache) as z:
                    assert np.array_equal(z['hold'],hold)
                    ph=z['p'];seconds=float(z['seconds'])
            else:
                start=time.perf_counter()
                with threadpool_limits(limits=2),warnings.catch_warnings():
                    warnings.simplefilter('error',ConvergenceWarning)
                    m=p_classifier(spec).fit(x[fit],y[fit]);ph=m.predict_proba(x[hold])[:,1]
                seconds=time.perf_counter()-start;tmp=cache.with_suffix('.tmp')
                with tmp.open('wb') as h:np.savez_compressed(h,hold=hold,p=ph,seconds=seconds)
                tmp.replace(cache)
            p[hold]=ph;auc=roc_auc_score(y[hold],ph);rows.append(dict(config=k,fold=fold,auc=auc,seconds=seconds))
            print(f'Config {k+1}/4 {spec}, fold {fold}/5: AUC={auc:.6f}, {seconds:.1f}s',flush=True)
        assert np.isfinite(p).all();predictions[k]=p
    cv=pd.DataFrame(rows);cv.to_csv(P_RUN/'cv_folds.csv',index=False)
    summary=cv.groupby('config',as_index=False).agg(mean_auc=('auc','mean'),worst_auc=('auc','min')).sort_values(['mean_auc','config'],ascending=[False,True])
    p_best_k=int(summary.iloc[0]['config']);p_best_spec=specs[p_best_k];candidate=predictions[p_best_k]
    bf=np.array([roc_auc_score(y[folds.fold.to_numpy()==f],baseline[folds.fold.to_numpy()==f]) for f in range(1,6)])
    rows=[]
    for w in [0.,.05,.1,.2,.3,.5]:
        p=(1-w)*baseline+w*candidate
        af=np.array([roc_auc_score(y[folds.fold.to_numpy()==f],p[folds.fold.to_numpy()==f]) for f in range(1,6)])
        rows.append(dict(weight=w,mean_auc=af.mean(),worst_auc=af.min(),mean_gain=(af-bf).mean(),worst_gain=af.min()-bf.min(),improved_folds=int((af>bf).sum())))
    blend=pd.DataFrame(rows);blend['eligible']=(blend.weight>0)&(blend.mean_gain>=.0005)&(blend.worst_gain>=0)&(blend.improved_folds>=3)
    ranked=blend[blend.weight.gt(0)].sort_values(['mean_auc','weight'],ascending=[False,True]);eligible=ranked[ranked.eligible]
    best=(eligible if len(eligible) else ranked).iloc[0];p_weight=float(best.weight)
    p_positive=bool(best.mean_gain>0 and best.worst_gain>=0 and best.improved_folds>=3)
    p_selection=dict(classifier=p_best_spec,experimental_weight=p_weight,positive_cv_signal=p_positive,passes_cv_gate=bool(len(eligible)),public_auc=None)
    summary.to_csv(P_RUN/'cv_summary.csv',index=False);blend.to_csv(P_RUN/'blend_cv_summary.csv',index=False)
    (P_RUN/'selection.json').write_text(json.dumps(p_selection,indent=2));pd.DataFrame(dict(id=oof.index,label=y,control=baseline,pythia=candidate)).to_csv(P_RUN/'oof_predictions.csv',index=False)
    display(summary);display(blend);print('Locked candidate:',p_selection)
else:
    print('Skipped: optional experiment is disabled.')


### 9.12 Restore the Scored Pythia and All-Feature Comparisons

Pythia candidate: 80% previous ensemble + 20% dual-LM statistics tree, public 0.98431.
All-feature candidate: 50% of that ensemble + 50% joint-feature tree, public 0.98677.
Scores below are saved experiments, not newly trained results.


In [ ]:
pythia_cv=pd.read_csv(PYTHIA_MODEL_DIR/'blend_cv_summary.csv')
pythia_validation=pd.read_csv(PYTHIA_MODEL_DIR/'validation_metrics.csv')
all_feature_cv=pd.read_csv(ALL_FEATURES_DIR/'blend_cv_summary.csv')
all_feature_validation=pd.read_csv(ALL_FEATURES_DIR/'validation_metrics.csv')
all_feature_importance=pd.read_csv(ALL_FEATURES_DIR/'feature_importance.csv')
display(pythia_cv);display(pythia_validation)
display(all_feature_cv);display(all_feature_validation)
import matplotlib.pyplot as plt
fig,axes=plt.subplots(1,2,figsize=(12,4))
axes[0].plot(all_feature_cv.weight.to_numpy(),all_feature_cv.mean_auc.to_numpy(),marker='o')
axes[0].axvline(.5,color='tab:green',linestyle='--',label='Scored best: 50% joint tree')
axes[0].set(xlabel='All-feature tree weight',ylabel='Mean five-fold AUROC',title='Saved fusion comparison')
axes[0].legend();axes[0].grid(alpha=.2)
all_feature_importance.head(12).iloc[::-1].plot.barh(x='feature',y='importance',ax=axes[1],legend=False)
axes[1].set_title('Feature importance (descriptive, not causal)')
fig.tight_layout();plt.show()


### 9.13 Optional Three-Branch Weight Search: Marginal Gain


In [ ]:
if RUN_THREE_BRANCH_SEARCH:
    # 10.9 Joint Three-Branch Weight Search — No Training or Feature Extraction
    from pathlib import Path
    import json,hashlib
    import numpy as np
    import pandas as pd
    from sklearn.metrics import roc_auc_score,log_loss
    from IPython.display import display
    root = Path(PROJECT_DIR)
    outputs=root/'outputs'
    source=outputs/'structure_catboost/404ce0b7b71319b1'
    pdir=outputs/'second_lm/c496bdce6ff4eb8a/classifiers/1963be0721ebc96f'
    wdir=outputs/'tree_weight_extension_097988/2e838e8ef8a0d53c'
    sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
    p_record=json.loads((pdir/'submission_manifest.json').read_text())
    assert p_record['selection']['experimental_weight']==.2
    scored_path=pdir/'submission_pythia_experimental.csv'
    assert sha(scored_path)==p_record['sha256']
    toof=pd.read_csv(source/'oof_predictions.csv').set_index('id')
    poof=pd.read_csv(pdir/'oof_predictions.csv').set_index('id').loc[toof.index]
    folds=pd.read_csv(outputs/'frozen_lm_lr/8b1f60e51c7f454a/fold_assignments.csv').set_index('id').loc[toof.index]
    assert toof.index.is_unique and len(toof)==8000 and np.array_equal(toof.label,poof.label) and np.array_equal(toof.label,folds.label)
    np.testing.assert_allclose(poof.control,.4*toof.control+.6*toof.tree,atol=1e-12,rtol=0)
    y=toof.label.to_numpy();branches=np.column_stack([toof.control,toof.tree,poof.pythia])
    assert np.isfinite(branches).all() and ((branches>=0)&(branches<=1)).all()
    incumbent=np.array([.32,.48,.20])
    # A limited 5-percentage-point simplex grid (231 candidates), plus exact incumbent.
    weight_grid=[(i/20,j/20,(20-i-j)/20) for i in range(21) for j in range(21-i)]
    weight_grid.append(tuple(incumbent));weight_grid=sorted(set(weight_grid))
    config=dict(version='three_branch_simplex_v1',grid_step=.05,weights=weight_grid,control_public_auc=.98431,control_sha=sha(scored_path),tree_oof_sha=sha(source/'oof_predictions.csv'),pythia_oof_sha=sha(pdir/'oof_predictions.csv'),fold_sha=sha(outputs/'frozen_lm_lr/8b1f60e51c7f454a/fold_assignments.csv'))
    folder=outputs/'three_branch_weights'/hashlib.sha256(json.dumps(config,sort_keys=True).encode()).hexdigest()[:16]
    folder.mkdir(parents=True,exist_ok=True)
    (folder/'config.json').write_text(json.dumps(config,indent=2));(folder/'control_098431.csv').write_bytes(scored_path.read_bytes())
    base=branches@incumbent
    masks=[folds.fold.to_numpy()==f for f in range(1,6)]
    assert set(folds.fold)=={1,2,3,4,5}
    bf=np.array([roc_auc_score(y[m],base[m]) for m in masks])
    rows=[];fold_rows=[]
    for weights in weight_grid:
        p=branches@np.array(weights)
        af=np.array([roc_auc_score(y[m],p[m]) for m in masks])
        rows.append(dict(old_weight=weights[0],structure_weight=weights[1],pythia_joint_weight=weights[2],mean_auc=af.mean(),worst_auc=af.min(),mean_gain=(af-bf).mean(),worst_gain=af.min()-bf.min(),improved_folds=int((af>bf).sum())))
        for f,auc in enumerate(af,1):fold_rows.append(dict(old_weight=weights[0],structure_weight=weights[1],pythia_joint_weight=weights[2],fold=f,auc=auc))
    results=pd.DataFrame(rows).sort_values(['mean_auc','pythia_joint_weight','structure_weight'],ascending=[False,True,True])
    results['eligible']=(results.mean_gain>=.0005)&(results.worst_gain>=0)&(results.improved_folds>=3)
    eligible=results[results.eligible]
    best=(eligible if len(eligible) else results).iloc[0]
    new_weights=np.array([best.old_weight,best.structure_weight,best.pythia_joint_weight])
    selected_weights=new_weights if len(eligible) else incumbent
    selection=dict(experimental_weights=new_weights.tolist(),selected_weights=selected_weights.tolist(),branch_order=['old_096875','structure_tree','gpt2_pythia_tree'],passes_cv_gate=bool(len(eligible)),public_auc=None)
    results.to_csv(folder/'cv_summary.csv',index=False);pd.DataFrame(fold_rows).to_csv(folder/'cv_folds.csv',index=False)
    (folder/'selection.json').write_text(json.dumps(selection,indent=2))
    display(results.head(12));print('Locked selection:',selection)
    print('Incumbent mean/worst CV:',bf.mean(),bf.min())
    # Validation is inspected only after locking the weights; no retuning below.
    tval=pd.read_csv(source/'validation_predictions.csv').set_index('id')
    pval=pd.read_csv(pdir/'validation_predictions.csv').set_index('id').loc[tval.index]
    assert set(tval.index).isdisjoint(toof.index) and tval.index.is_unique and np.array_equal(tval.label,pval.label)
    np.testing.assert_allclose(pval.control,.4*tval.control+.6*tval.tree,atol=1e-12,rtol=0)
    vbranches=np.column_stack([tval.control,tval.tree,pval.pythia])
    metrics=[]
    for name,weights in [('control_098431',incumbent),('experimental',new_weights),('selected',selected_weights)]:
        p=vbranches@weights
        metrics.append(dict(model=name,old_weight=weights[0],structure_weight=weights[1],pythia_joint_weight=weights[2],auc=roc_auc_score(tval.label,p),log_loss=log_loss(tval.label,p)))
    validation=pd.DataFrame(metrics);validation.to_csv(folder/'validation_metrics.csv',index=False);display(validation)
    # Recover exact saved branch predictions algebraically; no model load/refit is needed.
    old_path=outputs/'best_096875/submission_best_096875.csv'
    old_record=json.loads((old_path.parent/'manifest.json').read_text());assert sha(old_path)==old_record['sha256']
    tree_path=source/'submission_tree_selected.csv'
    tree_record=next(r for r in json.loads((source/'submission_manifest.json').read_text()) if r['variant']=='selected')
    assert tree_record['tree_weight']==.3 and sha(tree_path)==tree_record['sha256']
    control9827_path=wdir/'submission_tree_weight_experimental.csv'
    weight_record=next(r for r in json.loads((wdir/'submission_manifest.json').read_text()) if r['variant']=='experimental')
    assert weight_record['tree_weight']==.6 and sha(control9827_path)==weight_record['sha256']
    old=pd.read_csv(old_path);tree_scored=pd.read_csv(tree_path);control9827=pd.read_csv(control9827_path);scored=pd.read_csv(scored_path)
    data=next(root/d for d in ['data','Data'] if (root/d/'sample_submission.csv').is_file())
    template=pd.read_csv(data/'sample_submission.csv')
    assert all(np.array_equal(df.id,template.id) for df in [old,tree_scored,control9827,scored])
    a=old.label.to_numpy();b=(tree_scored.label.to_numpy()-.7*a)/.3
    np.testing.assert_allclose(.4*a+.6*b,control9827.label,atol=1e-12,rtol=0)
    c=(scored.label.to_numpy()-.8*control9827.label.to_numpy())/.2
    for p in [b,c]:assert np.isfinite(p).all() and p.min()>=-1e-10 and p.max()<=1+1e-10
    b=np.clip(b,0,1);c=np.clip(c,0,1);test_branches=np.column_stack([a,b,c])
    np.testing.assert_allclose(test_branches@incumbent,scored.label,atol=1e-12,rtol=0)
    exports=[]
    for name,weights in [('experimental',new_weights),('selected',selected_weights)]:
        path=folder/f'submission_three_branch_{name}.csv'
        if np.array_equal(weights,incumbent):path.write_bytes(scored_path.read_bytes())
        else:
            out=template[['id']].copy();out['label']=test_branches@weights;out.to_csv(path,index=False)
        out=pd.read_csv(path)
        assert out.columns.tolist()==['id','label'] and len(out)==3000 and out.id.is_unique
        assert np.array_equal(out.id,template.id) and np.isfinite(out.label).all() and out.label.between(0,1).all()
        exports.append(dict(variant=name,path=str(path),weights=weights.tolist(),sha256=sha(path),public_auc=None))
    (folder/'submission_manifest.json').write_text(json.dumps(exports,indent=2));display(pd.DataFrame(exports))
    print('Files:',folder)
    print('Selected retains exact 0.98431 incumbent.' if not len(eligible) else 'New selected passed the CV gate; public score unknown.')
    print('Experimental is a separate candidate. Repeated CV/validation are development diagnostics.')
else:
    print('Skipped: optional experiment is disabled.')


### 9.14 Optional All-Feature Table and Cache Fingerprints


In [ ]:
if RUN_ALL_FEATURES_RESEARCH:
    # 10.10 Joint Layout + GPT-2 + Pythia CatBoost: Feature-Level Fusion
    # The entire feature table is deterministic and label-free. Only fold training rows fit trees.
    from pathlib import Path
    from collections import Counter
    import json,hashlib,re,time,importlib.util,subprocess,sys
    import numpy as np
    import pandas as pd
    from sklearn.metrics import roc_auc_score,log_loss
    from IPython.display import display
    if importlib.util.find_spec('catboost') is None:
        subprocess.check_call([sys.executable,'-m','pip','install','-q','catboost==1.2.8'])
    import catboost
    from catboost import CatBoostClassifier
    joint_root = Path(PROJECT_DIR)
    out_root=joint_root/'outputs'
    data=next(joint_root/d for d in ['data','Data'] if (joint_root/d/'train.csv').is_file())
    bdir=out_root/'frozen_lm_lr/8b1f60e51c7f454a'
    rdir=out_root/'raw_gpt2_followup/9259247aa4f3e359'
    tdir=out_root/'structure_catboost/404ce0b7b71319b1'
    pfeature_dir=out_root/'second_lm/c496bdce6ff4eb8a'
    pmodel_dir=pfeature_dir/'classifiers/1963be0721ebc96f'
    sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
    control_csv=pmodel_dir/'submission_pythia_experimental.csv'
    control_record=json.loads((pmodel_dir/'submission_manifest.json').read_text())
    assert control_record['selection']['experimental_weight']==.2 and sha(control_csv)==control_record['sha256']
    train=pd.read_csv(data/'train.csv');test=pd.read_csv(data/'test.csv')
    base_config=json.loads((bdir/'config.json').read_text())
    assert sha(data/'train.csv')==base_config['train_sha256'] and sha(data/'test.csv')==base_config['test_sha256']
    all_text=pd.concat([train[['id','text']],test[['id','text']]],ignore_index=True)
    assert len(train)==10000 and len(test)==3000 and all_text.id.is_unique

    def joint_layout(t):
        words=re.findall(r"[A-Za-z]+(?:['’][A-Za-z]+)?",t.lower())
        lines=[l for l in t.splitlines() if l.strip()];n=max(1,len(lines));length=max(1,len(t))
        normlines=[' '.join(l.lower().split()) for l in lines]
        grams=list(zip(words,words[1:],words[2:]));counts=Counter(grams)
        repeat=lambda w:1-len(set(zip(w,w[1:],w[2:])))/max(1,len(w)-2) if len(w)>=3 else 0.
        half=len(words)//2
        return dict(words=len(t.split()),characters=len(t),blank_blocks=len([z for z in re.split(r'\n\s*\n',t) if z.strip()]),
            newlines=t.count('\n'),nonempty_lines=len(lines),hardwrap_ratio=sum(60<=len(l)<=85 for l in lines)/n,
            line_length_std=float(np.std([len(l) for l in lines])) if lines else 0.,
            bullet_ratio=sum(bool(re.match(r'^\s*[-*•]\s+',l)) for l in lines)/n,
            numbered_ratio=sum(bool(re.match(r'^\s*\d+[.)]\s+',l)) for l in lines)/n,
            html_breaks=len(re.findall(r'<br\s*/?>',t,re.I)),bold_markers=t.count('**'),
            repeated_line_ratio=1-len(set(normlines))/n if lines else 0.,
            trigram_repeat=repeat(words),max_trigram_ratio=max(counts.values(),default=0)/max(1,len(grams)),
            repetition_tail_change=repeat(words[half:])-repeat(words[:half]),
            exclamation_ratio=t.count('!')/length,question_ratio=t.count('?')/length,
            curly_quote_ratio=sum(t.count(c) for c in '’‘“”')/length,
            digit_ratio=sum(c.isdigit() for c in t)/length,
            transition_count=len(re.findall(r'\b(overall|in conclusion|furthermore|moreover|in summary)\b',t,re.I)))

    ids=all_text.id
    raw=pd.read_csv(rdir/'raw_features.csv').set_index('id').loc[ids]
    norm=pd.read_csv(bdir/'frozen_lm_features.csv').set_index('id').loc[ids]
    pythia=pd.read_csv(pfeature_dir/'features.csv').set_index('id').loc[ids]
    assert raw.columns.tolist()==norm.columns.tolist()
    layout=pd.DataFrame([joint_layout(t) for t in all_text.text],index=ids)
    features=pd.concat([raw.add_prefix('raw__'),norm.add_prefix('norm__'),(raw-norm).add_prefix('delta__'),layout.add_prefix('layout__'),pythia.add_prefix('pythia__')],axis=1)
    assert features.index.is_unique and features.columns.is_unique and np.isfinite(features.to_numpy()).all()
    # Verify that the old 93-feature tree input is reproduced before adding Pythia.
    old_names=json.loads((tdir/'config.json').read_text())['feature_names']
    old_hash=hashlib.sha256(pd.util.hash_pandas_object(features[old_names],index=True).values.tobytes()).hexdigest()
    assert old_hash==json.loads((tdir/'config.json').read_text())['feature_hash']
    toof=pd.read_csv(tdir/'oof_predictions.csv').set_index('id')
    poof=pd.read_csv(pmodel_dir/'oof_predictions.csv').set_index('id').loc[toof.index]
    folds=pd.read_csv(bdir/'fold_assignments.csv').set_index('id').loc[toof.index]
    y=toof.label.to_numpy();x=features.loc[toof.index]
    assert len(y)==8000 and np.array_equal(y,poof.label) and np.array_equal(y,folds.label)
    assert np.array_equal(train.set_index('id').loc[toof.index,'label'],y)
    np.testing.assert_allclose(poof.control,.4*toof.control+.6*toof.tree,atol=1e-12,rtol=0)
    incumbent=.8*poof.control.to_numpy()+.2*poof.pythia.to_numpy()
    specs=[dict(depth=d,l2_leaf_reg=l2) for d in [4,6] for l2 in [10.,30.]]
    weights=[0.,.05,.1,.2,.3,.5,.7,1.]
    config=dict(version='all_features_catboost_v1',catboost=catboost.__version__,specs=specs,weights=weights,iterations=600,learning_rate=.04,seed=42,feature_names=features.columns.tolist(),feature_sha=hashlib.sha256(pd.util.hash_pandas_object(features,index=True).values.tobytes()).hexdigest(),control_sha=sha(control_csv),control_public_auc=.98431,train_sha=sha(data/'train.csv'),fold_sha=sha(bdir/'fold_assignments.csv'),tree_oof_sha=sha(tdir/'oof_predictions.csv'),pythia_oof_sha=sha(pmodel_dir/'oof_predictions.csv'))
    folder=out_root/'all_features_catboost'/hashlib.sha256(json.dumps(config,sort_keys=True).encode()).hexdigest()[:16]
    folder.mkdir(parents=True,exist_ok=True);(folder/'config.json').write_text(json.dumps(config,indent=2));(folder/'control_098431.csv').write_bytes(control_csv.read_bytes())
    def joint_tree(spec):
        return CatBoostClassifier(**spec,iterations=600,learning_rate=.04,loss_function='Logloss',random_seed=42,thread_count=2,verbose=False,allow_writing_files=False)
else:
    print('Skipped: optional experiment is disabled.')


### 9.15 Optional All-Feature Five-Fold Search and Weight Selection


In [ ]:
if RUN_ALL_FEATURES_RESEARCH:
    rows=[];predictions={}

    print('Joint feature table:',features.shape,'| Experiment:',folder)
    for k,spec in enumerate(specs):
        p=np.full(len(y),np.nan)
        for fold in range(1,6):
            fit=np.flatnonzero(folds.fold.to_numpy()!=fold);hold=np.flatnonzero(folds.fold.to_numpy()==fold)
            cache=folder/f'config{k}_fold{fold}.npz'
            if cache.is_file():
                with np.load(cache) as z:
                    assert np.array_equal(z['hold'],hold)
                    ph=z['p'];seconds=float(z['seconds'])
            else:
                start=time.perf_counter();m=joint_tree(spec).fit(x.iloc[fit],y[fit]);ph=m.predict_proba(x.iloc[hold])[:,1]
                seconds=time.perf_counter()-start;tmp=cache.with_suffix('.tmp')
                with tmp.open('wb') as h:np.savez_compressed(h,hold=hold,p=ph,seconds=seconds)
                tmp.replace(cache)
            p[hold]=ph;auc=roc_auc_score(y[hold],ph);rows.append(dict(config=k,fold=fold,auc=auc,seconds=seconds))
            print(f'Config {k+1}/4 {spec}, fold {fold}/5: AUC={auc:.6f}, {seconds:.1f}s',flush=True)
        assert np.isfinite(p).all();predictions[k]=p
    cv=pd.DataFrame(rows);cv.to_csv(folder/'cv_folds.csv',index=False)
    summary=cv.groupby('config',as_index=False).agg(mean_auc=('auc','mean'),worst_auc=('auc','min'),seconds=('seconds','sum')).sort_values(['mean_auc','config'],ascending=[False,True])
    best_k=int(summary.iloc[0]['config']);candidate=predictions[best_k]
    masks=[folds.fold.to_numpy()==f for f in range(1,6)]
    bf=np.array([roc_auc_score(y[m],incumbent[m]) for m in masks])
    rows=[]
    for w in weights:
        p=(1-w)*incumbent+w*candidate;af=np.array([roc_auc_score(y[m],p[m]) for m in masks])
        rows.append(dict(weight=w,mean_auc=af.mean(),worst_auc=af.min(),mean_gain=(af-bf).mean(),worst_gain=af.min()-bf.min(),improved_folds=int((af>bf).sum())))
    blend=pd.DataFrame(rows);blend['eligible']=(blend.weight>0)&(blend.mean_gain>=.0005)&(blend.worst_gain>=0)&(blend.improved_folds>=3)
    ranked=blend[blend.weight.gt(0)].sort_values(['mean_auc','weight'],ascending=[False,True]);eligible=ranked[ranked.eligible]
    best=(eligible if len(eligible) else ranked).iloc[0];w=float(best.weight)
    positive=bool(best.mean_gain>0 and best.worst_gain>=0 and best.improved_folds>=3)
    selection=dict(spec=specs[best_k],experimental_weight=w,passes_cv_gate=bool(len(eligible)),positive_cv_signal=positive,public_auc=None)
    summary.to_csv(folder/'cv_summary.csv',index=False);blend.to_csv(folder/'blend_cv_summary.csv',index=False)
    (folder/'selection.json').write_text(json.dumps(selection,indent=2));pd.DataFrame(dict(id=toof.index,label=y,control=incumbent,joint=candidate)).to_csv(folder/'oof_predictions.csv',index=False)
    display(summary);display(blend);print('Locked selection:',selection)
else:
    print('Skipped: optional experiment is disabled.')


### 9.16 Optional Fixed Validation and Feature Importance


In [ ]:
if RUN_ALL_FEATURES_RESEARCH and positive:
    tv=pd.read_csv(tdir/'validation_predictions.csv').set_index('id')
    pv=pd.read_csv(pmodel_dir/'validation_predictions.csv').set_index('id').loc[tv.index]
    assert tv.index.is_unique and set(tv.index).isdisjoint(toof.index) and np.array_equal(tv.label,pv.label)
    assert np.array_equal(train.set_index('id').loc[tv.index,'label'],tv.label)
    np.testing.assert_allclose(pv.control,.4*tv.control+.6*tv.tree,atol=1e-12,rtol=0)
    baseline_val=.8*pv.control.to_numpy()+.2*pv.pythia.to_numpy()
    dev=joint_tree(specs[best_k]).fit(x,y);val=dev.predict_proba(features.loc[tv.index])[:,1]
    dev.save_model(str(folder/'model_8000.cbm'))
    metrics=[]
    for name,weight in [('control_098431',0.),('experimental',w)]:
        p=(1-weight)*baseline_val+weight*val
        metrics.append(dict(model=name,joint_weight=weight,auc=roc_auc_score(tv.label,p),log_loss=log_loss(tv.label,p)))
    metrics=pd.DataFrame(metrics);metrics.to_csv(folder/'validation_metrics.csv',index=False);display(metrics)
    pd.DataFrame(dict(id=tv.index,label=tv.label,control=baseline_val,joint=val)).to_csv(folder/'validation_predictions.csv',index=False)
    importance=pd.DataFrame(dict(feature=features.columns,importance=dev.feature_importances_)).sort_values('importance',ascending=False)
    importance.to_csv(folder/'feature_importance.csv',index=False);display(importance.head(15))
else:
    print('Skipped: optional experiment is disabled.')


### 9.17 Optional Cached Embeddings and Statistical Feature Preparation


In [ ]:
if RUN_EMBEDDING_RESEARCH:
    # A. Restore 122 statistical/layout features and 768-D cached GPT-2 vectors
    from pathlib import Path
    from collections import Counter
    import re,json,hashlib,time,sys,subprocess,importlib.util
    import numpy as np
    import pandas as pd
    from IPython.display import display
    from sklearn.decomposition import PCA
    from sklearn.metrics import roc_auc_score,log_loss
    from threadpoolctl import threadpool_limits
    import joblib,sklearn
    if importlib.util.find_spec('catboost') is None:
        subprocess.check_call([sys.executable,'-m','pip','install','-q','catboost==1.2.8'])
    import catboost
    from catboost import CatBoostClassifier
    E_ROOT = Path(PROJECT_DIR)
    E_OUT=E_ROOT/'outputs'
    E_DATA=next(E_ROOT/d for d in ['data','Data'] if (E_ROOT/d/'train.csv').is_file())
    E_BASE=E_OUT/'frozen_lm_lr/8b1f60e51c7f454a'
    E_RAW=E_OUT/'raw_gpt2_followup/9259247aa4f3e359'
    E_PY=E_OUT/'second_lm/c496bdce6ff4eb8a'
    E_ALL=E_OUT/'all_features_catboost/b28c916d42a794e2'
    E_EMB=E_OUT/'gpt2_embedding_lr/8d87d6e871124fda/all_embeddings.npz'
    def e_sha(p):return hashlib.sha256(Path(p).read_bytes()).hexdigest()
    e_train=pd.read_csv(E_DATA/'train.csv');e_test=pd.read_csv(E_DATA/'test.csv')
    e_all_text=pd.concat([e_train[['id','text']],e_test[['id','text']]],ignore_index=True)
    assert len(e_train)==10000 and len(e_test)==3000 and e_all_text.id.is_unique
    base_config=json.loads((E_BASE/'config.json').read_text())
    assert e_sha(E_DATA/'train.csv')==base_config['train_sha256'] and e_sha(E_DATA/'test.csv')==base_config['test_sha256']
    e_control_file=E_ALL/'submission_all_features_experimental.csv'
    e_record=json.loads((E_ALL/'submission_manifest.json').read_text())
    assert e_record['selection']['experimental_weight']==.5 and e_sha(e_control_file)==e_record['sha256']

    def e_layout(t):
        words=re.findall(r"[A-Za-z]+(?:['’][A-Za-z]+)?",t.lower())
        lines=[l for l in t.splitlines() if l.strip()];n=max(1,len(lines));length=max(1,len(t))
        normlines=[' '.join(l.lower().split()) for l in lines]
        grams=list(zip(words,words[1:],words[2:]));counts=Counter(grams)
        repeat=lambda w:1-len(set(zip(w,w[1:],w[2:])))/max(1,len(w)-2) if len(w)>=3 else 0.
        half=len(words)//2
        return dict(words=len(t.split()),characters=len(t),blank_blocks=len([z for z in re.split(r'\n\s*\n',t) if z.strip()]),
            newlines=t.count('\n'),nonempty_lines=len(lines),hardwrap_ratio=sum(60<=len(l)<=85 for l in lines)/n,
            line_length_std=float(np.std([len(l) for l in lines])) if lines else 0.,
            bullet_ratio=sum(bool(re.match(r'^\s*[-*•]\s+',l)) for l in lines)/n,
            numbered_ratio=sum(bool(re.match(r'^\s*\d+[.)]\s+',l)) for l in lines)/n,
            html_breaks=len(re.findall(r'<br\s*/?>',t,re.I)),bold_markers=t.count('**'),
            repeated_line_ratio=1-len(set(normlines))/n if lines else 0.,
            trigram_repeat=repeat(words),max_trigram_ratio=max(counts.values(),default=0)/max(1,len(grams)),
            repetition_tail_change=repeat(words[half:])-repeat(words[:half]),
            exclamation_ratio=t.count('!')/length,question_ratio=t.count('?')/length,
            curly_quote_ratio=sum(t.count(c) for c in '’‘“”')/length,
            digit_ratio=sum(c.isdigit() for c in t)/length,
            transition_count=len(re.findall(r'\b(overall|in conclusion|furthermore|moreover|in summary)\b',t,re.I)))
    ids=e_all_text.id
    raw=pd.read_csv(E_RAW/'raw_features.csv').set_index('id').loc[ids]
    norm=pd.read_csv(E_BASE/'frozen_lm_features.csv').set_index('id').loc[ids]
    py=pd.read_csv(E_PY/'features.csv').set_index('id').loc[ids]
    layout=pd.DataFrame([e_layout(t) for t in e_all_text.text],index=ids)
    e_stats=pd.concat([raw.add_prefix('raw__'),norm.add_prefix('norm__'),(raw-norm).add_prefix('delta__'),layout.add_prefix('layout__'),py.add_prefix('pythia__')],axis=1)
    expected=json.loads((E_ALL/'config.json').read_text())
    assert e_stats.columns.tolist()==expected['feature_names']
    assert hashlib.sha256(pd.util.hash_pandas_object(e_stats,index=True).values.tobytes()).hexdigest()==expected['feature_sha']
    with np.load(E_EMB,allow_pickle=False) as z:
        emb_ids=z['ids'].copy();emb=z['vectors'].copy()
    assert emb.shape==(13000,768) and len(set(emb_ids))==13000 and np.isfinite(emb).all()
    e_vectors=pd.DataFrame(emb,index=emb_ids).loc[ids]
    length=np.linalg.norm(e_vectors.to_numpy(),axis=1,keepdims=True)
    assert (length>0).all()
    e_vectors=pd.DataFrame((e_vectors.to_numpy()/length).astype(np.float32),index=ids)
    e_oof=pd.read_csv(E_ALL/'oof_predictions.csv').set_index('id')
    e_folds=pd.read_csv(E_BASE/'fold_assignments.csv').set_index('id').loc[e_oof.index]
    e_y=e_oof.label.to_numpy();e_baseline=.5*e_oof.control.to_numpy()+.5*e_oof.joint.to_numpy()
    assert len(e_y)==8000 and np.array_equal(e_y,e_folds.label) and set(e_folds.fold)=={1,2,3,4,5}
    assert np.array_equal(e_train.set_index('id').loc[e_oof.index,'label'],e_y)
    E_SPECS=[dict(pca_components=k,depth=d) for k in [16,32] for d in [4,6]]
    E_WEIGHTS=[0.,.05,.1,.2,.3,.5,.7,1.]
    e_config=dict(version='cached_embedding_joint_v1',sklearn=sklearn.__version__,catboost=catboost.__version__,embedding_sha=e_sha(E_EMB),stats_sha=expected['feature_sha'],train_sha=e_sha(E_DATA/'train.csv'),fold_sha=e_sha(E_BASE/'fold_assignments.csv'),oof_sha=e_sha(E_ALL/'oof_predictions.csv'),control_sha=e_sha(e_control_file),control_public_auc=.98677,normalization='row_l2_no_fitting',pca_fit='training_rows_only',pca_max_components=32,pca_solver='randomized',seed=42,specs=E_SPECS,weights=E_WEIGHTS,iterations=600,learning_rate=.04,l2_leaf_reg=10.)
    E_DIR=E_OUT/'embedding_statistics_joint'/hashlib.sha256(json.dumps(e_config,sort_keys=True).encode()).hexdigest()[:16]
    E_DIR.mkdir(parents=True,exist_ok=True);(E_DIR/'config.json').write_text(json.dumps(e_config,indent=2));(E_DIR/'control_098677.csv').write_bytes(e_control_file.read_bytes())
    def e_model(spec):
        return CatBoostClassifier(depth=spec['depth'],l2_leaf_reg=10.,iterations=600,learning_rate=.04,loss_function='Logloss',random_seed=42,thread_count=2,verbose=False,allow_writing_files=False)
    print('Statistical features:',e_stats.shape,'| Cached embeddings:',e_vectors.shape)
    print('Experiment:',E_DIR)
else:
    print('Skipped: optional stage is disabled.')


### 9.18 Optional Fold-Specific PCA and Joint-Tree Search

Four configurations: 16/32 used components × depth 4/6. The 32-component PCA is
fitted on each fold's training rows; validation-fold rows never fit it.


In [ ]:
if RUN_EMBEDDING_RESEARCH:
    # B. Fold-specific PCA, four CatBoost configurations, and OOF fusion
    # Each 32-D PCA is fitted once per training fold; 16-D views use its first 16 PCs.
    e_rows=[];e_predictions={k:np.full(len(e_y),np.nan) for k in range(len(E_SPECS))}
    e_x=e_stats.loc[e_oof.index].to_numpy();e_v=e_vectors.loc[e_oof.index].to_numpy()
    for fold in range(1,6):
        fit=np.flatnonzero(e_folds.fold.to_numpy()!=fold);hold=np.flatnonzero(e_folds.fold.to_numpy()==fold)
        pc_cache=E_DIR/f'pca32_fold{fold}.npz'
        if pc_cache.is_file():
            with np.load(pc_cache) as z:
                assert np.array_equal(z['fit'],fit) and np.array_equal(z['hold'],hold)
                pc_fit=z['fit_projection'];pc_hold=z['hold_projection']
        else:
            start=time.perf_counter()
            with threadpool_limits(limits=2):
                pca=PCA(n_components=32,svd_solver='randomized',random_state=42)
                pca.fit(e_v[fit]);pc_fit=pca.transform(e_v[fit]);pc_hold=pca.transform(e_v[hold])
            joblib.dump(pca,E_DIR/f'pca32_fold{fold}.joblib')
            tmp=pc_cache.with_suffix('.tmp')
            with tmp.open('wb') as h:np.savez_compressed(h,fit=fit,hold=hold,fit_projection=pc_fit,hold_projection=pc_hold)
            tmp.replace(pc_cache)
            print(f'PCA fold {fold}/5 fitted on {len(fit)} rows: {time.perf_counter()-start:.1f}s',flush=True)
        for k,spec in enumerate(E_SPECS):
            cache=E_DIR/f'config{k}_fold{fold}.npz';components=spec['pca_components']
            if cache.is_file():
                with np.load(cache) as z:
                    assert np.array_equal(z['hold'],hold)
                    p=z['p'];seconds=float(z['seconds'])
            else:
                xf=np.hstack([e_x[fit],pc_fit[:,:components]]);xh=np.hstack([e_x[hold],pc_hold[:,:components]])
                assert np.isfinite(xf).all() and np.isfinite(xh).all()
                start=time.perf_counter();model=e_model(spec).fit(xf,e_y[fit]);p=model.predict_proba(xh)[:,1]
                seconds=time.perf_counter()-start;tmp=cache.with_suffix('.tmp')
                with tmp.open('wb') as h:np.savez_compressed(h,hold=hold,p=p,seconds=seconds)
                tmp.replace(cache)
            e_predictions[k][hold]=p;auc=roc_auc_score(e_y[hold],p)
            e_rows.append(dict(config=k,pca_components=components,depth=spec['depth'],fold=fold,auc=auc,seconds=seconds))
            print(f'Fold {fold}/5, config {k+1}/4 {spec}: AUC={auc:.6f}, {seconds:.1f}s',flush=True)
    for p in e_predictions.values():assert np.isfinite(p).all()
    e_cv=pd.DataFrame(e_rows);e_cv.to_csv(E_DIR/'cv_folds.csv',index=False)
    e_summary=e_cv.groupby(['config','pca_components','depth'],as_index=False).agg(mean_auc=('auc','mean'),worst_auc=('auc','min'),seconds=('seconds','sum')).sort_values(['mean_auc','config'],ascending=[False,True])
    e_best_k=int(e_summary.iloc[0]['config']);e_candidate=e_predictions[e_best_k]
    masks=[e_folds.fold.to_numpy()==f for f in range(1,6)]
    bf=np.array([roc_auc_score(e_y[m],e_baseline[m]) for m in masks])
    rows=[]
    for w in E_WEIGHTS:
        p=(1-w)*e_baseline+w*e_candidate;af=np.array([roc_auc_score(e_y[m],p[m]) for m in masks])
        rows.append(dict(weight=w,mean_auc=af.mean(),worst_auc=af.min(),mean_gain=(af-bf).mean(),worst_gain=af.min()-bf.min(),improved_folds=int((af>bf).sum())))
    e_blend=pd.DataFrame(rows)
    e_blend['eligible']=(e_blend.weight>0)&(e_blend.mean_gain>=.0005)&(e_blend.worst_gain>=0)&(e_blend.improved_folds>=3)
    ranked=e_blend[e_blend.weight.gt(0)].sort_values(['mean_auc','weight'],ascending=[False,True]);eligible=ranked[ranked.eligible]
    best=(eligible if len(eligible) else ranked).iloc[0];e_weight=float(best.weight)
    e_positive=bool(best.mean_gain>0 and best.worst_gain>=0 and best.improved_folds>=3)
    e_selection=dict(spec=E_SPECS[e_best_k],experimental_weight=e_weight,positive_cv_signal=e_positive,passes_cv_gate=bool(len(eligible)),public_auc=None)
    e_summary.to_csv(E_DIR/'cv_summary.csv',index=False);e_blend.to_csv(E_DIR/'blend_cv_summary.csv',index=False)
    (E_DIR/'selection.json').write_text(json.dumps(e_selection,indent=2))
    pd.DataFrame(dict(id=e_oof.index,label=e_y,control=e_baseline,embedding_joint=e_candidate)).to_csv(E_DIR/'oof_predictions.csv',index=False)
    display(e_summary);display(e_blend);print('Locked candidate:',e_selection)
    print('Repeated OOF selection is exploratory, not an independent estimate of leaderboard performance.')
else:
    print('Skipped: optional stage is disabled.')


### 9.19 Optional Fixed Validation with the 8,000-Row PCA and Tree


In [ ]:
if RUN_EMBEDDING_RESEARCH and e_positive:
    val=pd.read_csv(E_ALL/'validation_predictions.csv').set_index('id')
    assert val.index.is_unique and set(val.index).isdisjoint(e_oof.index)
    assert np.array_equal(val.label,e_train.set_index('id').loc[val.index,'label'])
    spec=E_SPECS[e_best_k];components=spec['pca_components']
    with threadpool_limits(limits=2):
        dev_pca=PCA(n_components=32,svd_solver='randomized',random_state=42).fit(e_vectors.loc[e_oof.index].to_numpy())
        train_pc=dev_pca.transform(e_vectors.loc[e_oof.index].to_numpy())[:,:components]
        val_pc=dev_pca.transform(e_vectors.loc[val.index].to_numpy())[:,:components]
    xf=np.hstack([e_stats.loc[e_oof.index].to_numpy(),train_pc]);xv=np.hstack([e_stats.loc[val.index].to_numpy(),val_pc])
    dev=e_model(spec).fit(xf,e_y);p=dev.predict_proba(xv)[:,1]
    joblib.dump(dev_pca,E_DIR/'pca_8000.joblib');dev.save_model(str(E_DIR/'tree_8000.cbm'))
    baseline=.5*val.control.to_numpy()+.5*val.joint.to_numpy()
    new=(1-e_weight)*baseline+e_weight*p
    metrics=pd.DataFrame([dict(model='control_098677',auc=roc_auc_score(val.label,baseline),log_loss=log_loss(val.label,baseline)),dict(model='experimental',auc=roc_auc_score(val.label,new),log_loss=log_loss(val.label,new))])
    metrics.to_csv(E_DIR/'validation_metrics.csv',index=False);display(metrics)
    pd.DataFrame(dict(id=val.index,label=val.label,control=baseline,embedding_joint=p)).to_csv(E_DIR/'validation_predictions.csv',index=False)
else:
    print('Skipped: optional stage is disabled.')


### 9.20 Restore the Scored Embedding/Statistics Comparison

These are saved development diagnostics, not a fresh independent test.
The scored candidate uses 50% new tree and 50% previous ensemble.


In [ ]:
embedding_cv=pd.read_csv(EMBEDDING_JOINT_DIR/'cv_summary.csv')
embedding_blend=pd.read_csv(EMBEDDING_JOINT_DIR/'blend_cv_summary.csv')
embedding_validation=pd.read_csv(EMBEDDING_JOINT_DIR/'validation_metrics.csv')
display(embedding_cv);display(embedding_blend);display(embedding_validation)
import matplotlib.pyplot as plt
fig,ax=plt.subplots(figsize=(7,4))
ax.plot(embedding_blend.weight.to_numpy(),embedding_blend.mean_auc.to_numpy(),marker='o')
ax.axvline(.5,color='tab:green',linestyle='--',label='Scored best: 50% embedding/statistics tree')
ax.set(xlabel='Embedding/statistics tree weight',ylabel='Mean five-fold AUROC',title='Saved fusion comparison')
ax.legend();ax.grid(alpha=.2);fig.tight_layout();plt.show()


### 9.21 Optional Current-Best Error Diagnostics

Uses saved 8,000-row-model predictions on the fixed 2,000-row validation set.
Reports actual threshold errors, AUROC pair-ordering errors, observable subgroups
and branch disagreement. No hidden test labels or 10,000-row model evaluations.


In [ ]:
if RUN_DIAGNOSTICS:
    # A. Restore and verify development/validation predictions — no training
    from pathlib import Path
    import json,hashlib,re
    import numpy as np
    import pandas as pd
    from sklearn.metrics import roc_auc_score,log_loss,confusion_matrix
    from IPython.display import display
    DIAG_ROOT = Path(PROJECT_DIR)
    DIAG_DATA=next(DIAG_ROOT/d for d in ['data','Data'] if (DIAG_ROOT/d/'train.csv').is_file())
    DIAG_OUT=DIAG_ROOT/'outputs'
    DIAG_BASE=DIAG_OUT/'frozen_lm_lr/8b1f60e51c7f454a'
    DIAG_TREE=DIAG_OUT/'structure_catboost/404ce0b7b71319b1'
    DIAG_PYTHIA=DIAG_OUT/'second_lm/c496bdce6ff4eb8a/classifiers/1963be0721ebc96f'
    DIAG_ALL=DIAG_OUT/'all_features_catboost/b28c916d42a794e2'
    DIAG_EMB=DIAG_OUT/'embedding_statistics_joint/172ad2e61031c035'
    def diag_sha(p):return hashlib.sha256(Path(p).read_bytes()).hexdigest()
    diag_config=json.loads((DIAG_BASE/'config.json').read_text())
    assert diag_sha(DIAG_DATA/'train.csv')==diag_config['train_sha256']
    diag_train=pd.read_csv(DIAG_DATA/'train.csv').set_index('id')
    assert diag_train.index.is_unique and len(diag_train)==10000
    diag_all_val=pd.read_csv(DIAG_ALL/'validation_predictions.csv').set_index('id')
    diag_p_val=pd.read_csv(DIAG_PYTHIA/'validation_predictions.csv').set_index('id').loc[diag_all_val.index]
    diag_t_val=pd.read_csv(DIAG_TREE/'validation_predictions.csv').set_index('id').loc[diag_all_val.index]
    diag_emb_val=pd.read_csv(DIAG_EMB/'validation_predictions.csv').set_index('id').loc[diag_all_val.index]
    diag_oof=pd.read_csv(DIAG_EMB/'oof_predictions.csv').set_index('id')
    assert len(diag_all_val)==2000 and len(diag_oof)==8000
    assert diag_all_val.index.is_unique and diag_oof.index.is_unique
    assert set(diag_all_val.index).isdisjoint(diag_oof.index)
    assert set(diag_all_val.index)|set(diag_oof.index)==set(diag_train.index)
    for df in [diag_all_val,diag_p_val,diag_t_val,diag_emb_val,diag_oof]:
        assert np.array_equal(df.label,diag_train.loc[df.index,'label'])
    np.testing.assert_allclose(diag_p_val.control,.4*diag_t_val.control+.6*diag_t_val.tree,atol=1e-12,rtol=0)
    np.testing.assert_allclose(diag_all_val.control,.8*diag_p_val.control+.2*diag_p_val.pythia,atol=1e-12,rtol=0)
    diag_selection=json.loads((DIAG_ALL/'selection.json').read_text())
    assert diag_selection['experimental_weight']==.5
    scored_file=DIAG_EMB/'submission_embedding_statistics_experimental.csv'
    scored_record=json.loads((DIAG_EMB/'submission_manifest.json').read_text())
    assert scored_record['selection']['experimental_weight']==.5
    assert diag_sha(scored_file)==scored_record['sha256']
    diag=diag_train.loc[diag_all_val.index,['text','label']].copy()
    diag['old_096875']=diag_t_val.control
    diag['structure_tree']=diag_t_val.tree
    diag['dual_lm_tree']=diag_p_val.pythia
    diag['all_feature_tree']=diag_all_val.joint
    diag['control_098431']=diag_all_val.control
    diag['control_098677']=.5*diag_all_val.control+.5*diag_all_val.joint
    np.testing.assert_allclose(diag_emb_val.control,diag.control_098677,atol=1e-12,rtol=0)
    diag['embedding_statistics_tree']=diag_emb_val.embedding_joint
    diag['best_098816']=.5*diag_emb_val.control+.5*diag_emb_val.embedding_joint
    assert np.isfinite(diag.iloc[:,2:].to_numpy()).all() and diag.iloc[:,2:].ge(0).all().all() and diag.iloc[:,2:].le(1).all().all()
    DIAG_THRESHOLD=.5
    meta=dict(version='best098816_error_diagnostics_v1',scored_csv_sha=diag_sha(scored_file),train_sha=diag_sha(DIAG_DATA/'train.csv'),validation_sha=diag_sha(DIAG_EMB/'validation_predictions.csv'),oof_sha=diag_sha(DIAG_EMB/'oof_predictions.csv'),threshold=DIAG_THRESHOLD,public_auc=.98816,public_score_source='user-reported',validation_repeatedly_reused=True,final_10000_models_evaluated_on_validation=False)
    DIAG_DIR=DIAG_OUT/'error_analysis_098816'/hashlib.sha256(json.dumps(meta,sort_keys=True).encode()).hexdigest()[:16]
    DIAG_DIR.mkdir(parents=True,exist_ok=True);(DIAG_DIR/'config.json').write_text(json.dumps(meta,indent=2))
    rows=[]
    for col in ['old_096875','structure_tree','dual_lm_tree','all_feature_tree','control_098431','control_098677','embedding_statistics_tree','best_098816']:
        p=diag[col].to_numpy();tn,fp,fn,tp=confusion_matrix(diag.label,p>=DIAG_THRESHOLD,labels=[0,1]).ravel()
        rows.append(dict(model=col,validation_auc=roc_auc_score(diag.label,p),log_loss=log_loss(diag.label,p),FP=int(fp),FN=int(fn),threshold=DIAG_THRESHOLD))
    diag_metrics=pd.DataFrame(rows);diag_metrics.to_csv(DIAG_DIR/'validation_models.csv',index=False);display(diag_metrics)
    oof_prob=.5*diag_oof.control+.5*diag_oof.embedding_joint
    print('Development OOF AUROC (not in-sample training AUROC):',roc_auc_score(diag_oof.label,oof_prob))
    print('Validation AUROC:',roc_auc_score(diag.label,diag.best_098816))
    print('Diagnostic directory:',DIAG_DIR)
else:
    print('Skipped: optional stage is disabled.')


In [ ]:
if RUN_DIAGNOSTICS:
    # B. AUROC ranking mistakes and observable subgroups
    # An AUROC error is a human passage ranked above an AI passage; threshold errors differ.
    diag['prediction']=(diag.best_098816>=DIAG_THRESHOLD).astype(int)
    diag['error_type']=np.select([diag.label.eq(0)&diag.prediction.eq(1),diag.label.eq(1)&diag.prediction.eq(0)],['FP','FN'],default='Correct')
    h=np.sort(diag.loc[diag.label.eq(0),'best_098816'].to_numpy())
    a=np.sort(diag.loc[diag.label.eq(1),'best_098816'].to_numpy())
    diag['rank_error_pairs']=0.
    for label,other in [(0,a),(1,h)]:
        mask=diag.label.eq(label);p=diag.loc[mask,'best_098816'].to_numpy()
        left=np.searchsorted(other,p,side='left');right=np.searchsorted(other,p,side='right')
        burden=left+.5*(right-left) if label==0 else len(other)-right+.5*(right-left)
        diag.loc[mask,'rank_error_pairs']=burden
    human_burden=diag.loc[diag.label.eq(0),'rank_error_pairs'].sum()
    ai_burden=diag.loc[diag.label.eq(1),'rank_error_pairs'].sum()
    np.testing.assert_allclose(human_burden,ai_burden,atol=1e-9,rtol=0)
    np.testing.assert_allclose(1-human_burden/(len(h)*len(a)),roc_auc_score(diag.label,diag.best_098816),atol=1e-12,rtol=0)

    def diag_observables(t):
        lines=[line for line in t.splitlines() if line.strip()]
        return dict(words=len(t.split()),blank_blocks=len([z for z in re.split(r'\n\s*\n',t) if z.strip()]),
            hardwrap_ratio=sum(60<=len(line)<=85 for line in lines)/max(1,len(lines)),
            html_break=bool(re.search(r'<br\s*/?>',t,re.I)),
            has_list=bool(re.search(r'^\s*(?:[-*•]\s+|\d+[.)]\s+)',t,re.M)))
    observed=pd.DataFrame([diag_observables(t) for t in diag.text],index=diag.index)
    for col in observed:diag[col]=observed[col]
    diag['length_group']=pd.cut(diag.words,bins=[-np.inf,150,250,np.inf],labels=['<=150 words','151-250 words','>250 words'])
    diag['blank_group']=np.where(diag.blank_blocks>1,'multiple blank-line blocks','one blank-line block')
    diag['wrap_group']=np.where(diag.hardwrap_ratio>.5,'mostly fixed-width lines','other line layout')
    diag['html_group']=np.where(diag.html_break,'HTML breaks','no HTML breaks')
    diag['list_group']=np.where(diag.has_list,'list present','no list')
    rows=[]
    for group in ['length_group','blank_group','wrap_group','html_group','list_group']:
        for value,part in diag.groupby(group,observed=True):
            two=part.label.nunique()==2
            rows.append(dict(group=group,subgroup=str(value),rows=len(part),human=int(part.label.eq(0).sum()),ai=int(part.label.eq(1).sum()),auc=roc_auc_score(part.label,part.best_098816) if two else np.nan,FP=int(part.error_type.eq('FP').sum()),FN=int(part.error_type.eq('FN').sum()),human_rank_burden=part.loc[part.label.eq(0),'rank_error_pairs'].sum(),ai_rank_burden=part.loc[part.label.eq(1),'rank_error_pairs'].sum()))
    diag_groups=pd.DataFrame(rows);diag_groups.to_csv(DIAG_DIR/'subgroup_metrics.csv',index=False)
    display(diag_groups)
    # Rank burden is against the whole validation set, not just within each subgroup.
    print('Human-AI pairs:',len(h)*len(a),'| Misordered pair equivalents (ties count 0.5):',human_burden)
    print('Single-class subgroup AUROC is undefined (NaN). Small subgroups are uncertain.')
    diag.drop(columns='text').to_csv(DIAG_DIR/'validation_diagnostics.csv')
else:
    print('Skipped: optional stage is disabled.')


In [ ]:
if RUN_DIAGNOSTICS:
    # C. Review actual cases and branch disagreement; no invented examples or explanations
    branch_cols=['structure_tree','dual_lm_tree','all_feature_tree','embedding_statistics_tree']
    diag['branch_range']=diag[branch_cols].max(axis=1)-diag[branch_cols].min(axis=1)
    diag['branch_std']=diag[branch_cols].std(axis=1,ddof=0)
    diag['excerpt']=diag.text.map(lambda t:t[:500]+(' ...' if len(t)>500 else ''))
    review_cols=['label','error_type','best_098816','rank_error_pairs','words','blank_blocks','hardwrap_ratio','html_break','has_list','structure_tree','dual_lm_tree','all_feature_tree','embedding_statistics_tree','branch_range','excerpt']
    review_ids={}
    for error in ['FP','FN']:
        actual=diag[diag.error_type.eq(error)].sort_values('best_098816',ascending=(error=='FN')).head(5)
        actual[review_cols].to_csv(DIAG_DIR/f'{error.lower()}_actual_cases.csv')
        review_ids[error]=actual.index.tolist()
        print(f'{error}: showing {len(actual)} actual cases from {int(diag.error_type.eq(error).sum())} available; threshold={DIAG_THRESHOLD}')
        display(actual[review_cols])

    rank_cases=[]
    for label in [0,1]:
        actual=diag[(diag.label==label)&(diag.rank_error_pairs>0)].sort_values(['rank_error_pairs','branch_range'],ascending=False).head(10)
        rank_cases.append(actual)
        print('Highest AUROC-error burden, true label:',label)
        display(actual[review_cols])
    pd.concat(rank_cases)[review_cols].to_csv(DIAG_DIR/'top_rank_error_cases.csv')
    disagreement=diag.sort_values('branch_range',ascending=False).head(20)
    disagreement[review_cols].to_csv(DIAG_DIR/'top_branch_disagreements.csv')
    print('Top 10 disagreements (disagreement does not imply an error):')
    display(disagreement[review_cols].head(10))
    # Human explanations must be entered after reading the actual passages.
    manual=pd.concat(rank_cases).loc[lambda d:~d.index.duplicated(),review_cols].copy()
    manual['observed_pattern']=''
    manual['failure_hypothesis']=''
    manual['proposed_test']=''
    manual_path=DIAG_DIR/'manual_review.csv'
    if not manual_path.exists():manual.to_csv(manual_path)

    import matplotlib.pyplot as plt
    fig,axes=plt.subplots(1,3,figsize=(15,4))
    axes[0].scatter(diag.structure_tree,diag.dual_lm_tree,c=diag.label,cmap='coolwarm',alpha=.4,s=12)
    axes[0].plot([0,1],[0,1],'--',color='gray');axes[0].set(xlabel='Structural tree P(AI)',ylabel='Dual-LM tree P(AI)',title='Branch disagreement (red=AI, blue=human)')
    for label,color in [(0,'tab:blue'),(1,'tab:red')]:
        values=diag.loc[diag.label.eq(label),'best_098816']
        axes[1].hist(values,bins=np.linspace(0,1,31),histtype='step',label=f'True label {label}',color=color)
    axes[1].axvline(DIAG_THRESHOLD,color='gray',linestyle='--');axes[1].set(xlabel='Final ensemble P(AI)',ylabel='Rows',title='Actual validation score distribution');axes[1].legend()
    length=diag_groups[diag_groups.group.eq('length_group')]
    axes[2].bar(length.subgroup.to_list(),length.auc.to_numpy(),color='tab:green')
    axes[2].set_ylim(0,1);axes[2].set(ylabel='AUROC',title='Length subgroups (full 0–1 scale)');axes[2].tick_params(axis='x',rotation=20)
    fig.tight_layout();fig.savefig(DIAG_DIR/'diagnostic_overview.png',dpi=160,bbox_inches='tight');plt.show()
    report=dict(config=meta,metrics=diag_metrics.to_dict('records'),threshold_error_ids=review_ids,rank_error_pair_equivalents=float(human_burden),human_ai_pairs=int(len(h)*len(a)),manual_review=str(manual_path),notes=['All examples are observed validation passages','Rank errors differ from threshold errors','Formatting associations are not proof of authorship or source','No new training, feature extraction, test predictions or submissions'])
    (DIAG_DIR/'summary.json').write_text(json.dumps(report,indent=2))
    print('Diagnostic outputs:',DIAG_DIR)
    print('No new model or submission created. Preserve the 0.98816 incumbent.')
else:
    print('Skipped: optional stage is disabled.')


## 10. Final Models, Exact Best Submission, and Reproducibility

Normal use exports the exact scored 0.98816 file after validating its bytes, IDs and
probabilities. Optional refits create separate experimental files. Saved PCA and
classifier versions are recorded for prediction reproducibility.


### 10.1 Optional Earlier Final Models and Candidate Exports


In [ ]:
if RUN_TREE_RESEARCH:
    # D. Final 10,000-row refit and separate candidate exports
    # Never evaluate this final model on validation: those labels enter its training.
    tree_template=pd.read_csv(DATA_DIR/'sample_submission.csv')
    assert sha256_file(LEGACY_BEST_SUBMISSION_PATH)==COMPONENT_BASELINE['sha256']
    tree_control_test=pd.read_csv(LEGACY_BEST_SUBMISSION_PATH)
    assert np.array_equal(tree_control_test.id,tree_template.id)
    final_path=TREE_DIR/'tree_10000.cbm'
    if final_path.is_file():
        tree_final=tree_model(TREE_CONFIGS[tree_best_k]);tree_final.load_model(str(final_path))
    else:
        tree_final=tree_model(TREE_CONFIGS[tree_best_k]).fit(tree_features.loc[tree_train.id],tree_train.label.to_numpy())
        tree_final.save_model(str(final_path))
    p_tree=tree_final.predict_proba(tree_features.loc[tree_template.id])[:,1]
    exports=[]
    for name,w in [('experimental',tree_experimental_weight),('selected',tree_selected_weight)]:
        path=TREE_DIR/f'submission_tree_{name}.csv'
        if w==0: path.write_bytes(LEGACY_BEST_SUBMISSION_PATH.read_bytes())
        else:
            out=tree_template[['id']].copy();out['label']=(1-w)*tree_control_test.label.to_numpy()+w*p_tree;out.to_csv(path,index=False)
        out=pd.read_csv(path)
        assert out.columns.tolist()==['id','label'] and len(out)==3000 and out.id.is_unique
        assert np.array_equal(out.id,tree_template.id) and np.isfinite(out.label).all() and out.label.between(0,1).all()
        exports.append(dict(variant=name,path=str(path),tree_weight=w,sha256=sha256_file(path),public_auc=None))
    (TREE_DIR/'submission_manifest.json').write_text(json.dumps(exports,indent=2))
    assert sha256_file(LEGACY_BEST_SUBMISSION_PATH)==COMPONENT_BASELINE['sha256']
    display(pd.DataFrame(exports));print('Tree experiment:',TREE_DIR)
    print('CV gate passed; candidate public score unknown.' if tree_selected_weight else 'CV gate failed: selected duplicates 0.96875. Experimental is separate.')
    if TREE_DOWNLOAD and IN_COLAB:
        from google.colab import files
        files.download(str(TREE_DIR/'submission_tree_experimental.csv'))
else:
    print('Skipped: optional experiment is disabled.')


In [ ]:
if RUN_PYTHIA_RESEARCH:
    # D. Validation is assessed after locking parameters; final 10k model predicts test only.
    if not p_positive:
        print('No consistent positive CV signal. Preserve 0.98270; no final refit/submission is generated.')
    else:
        vp=pd.read_csv(source/'validation_predictions.csv').set_index('id')
        indexed=p_train.set_index('id');features=views[p_best_spec['view']]
        assert set(vp.index).isdisjoint(oof.index) and np.array_equal(indexed.loc[oof.index,'label'],y)
        assert np.array_equal(indexed.loc[vp.index,'label'],vp.label)
        with threadpool_limits(limits=2):
            dev=p_classifier(p_best_spec).fit(features.loc[oof.index].to_numpy(),y)
            pv=dev.predict_proba(features.loc[vp.index].to_numpy())[:,1]
        joblib.dump(dev,P_RUN/'classifier_8000.joblib',compress=3)
        bv=.4*vp.control.to_numpy()+.6*vp.tree.to_numpy();new=(1-p_weight)*bv+p_weight*pv
        metrics=pd.DataFrame([dict(model='control_098270',auc=roc_auc_score(vp.label,bv),log_loss=log_loss(vp.label,bv)),dict(model='experimental',auc=roc_auc_score(vp.label,new),log_loss=log_loss(vp.label,new))])
        metrics.to_csv(P_RUN/'validation_metrics.csv',index=False);display(metrics)
        pd.DataFrame(dict(id=vp.index,label=vp.label,control=bv,pythia=pv)).to_csv(P_RUN/'validation_predictions.csv',index=False)
        template=pd.read_csv(DATA_DIR/'sample_submission.csv')
        with threadpool_limits(limits=2):
            final=p_classifier(p_best_spec).fit(features.loc[p_train.id].to_numpy(),p_train.label.to_numpy())
            pt=final.predict_proba(features.loc[template.id].to_numpy())[:,1]
        joblib.dump(final,P_RUN/'classifier_10000.joblib',compress=3)
        control=pd.read_csv(control_csv);assert np.array_equal(control.id,template.id) and sha256_file(control_csv)==rec['sha256']
        out=template[['id']].copy();out['label']=(1-p_weight)*control.label.to_numpy()+p_weight*pt
        assert len(out)==3000 and out.id.is_unique and np.isfinite(out.label).all() and out.label.between(0,1).all()
        path=P_RUN/'submission_pythia_experimental.csv';out.to_csv(path,index=False)
        (P_RUN/'submission_manifest.json').write_text(json.dumps(dict(path=str(path),sha256=sha256_file(path),selection=p_selection,public_auc=None),indent=2))
        print('Experimental candidate:',path)
        print('Public score unknown; incumbent 0.98270 file unchanged.')
        if P_DOWNLOAD and IN_COLAB:
            from google.colab import files
            files.download(str(path))
    print('Feature folder:',P_DIR)
    print('Classifier folder:',P_RUN)
else:
    print('Skipped: optional experiment is disabled.')


### 10.2 Optional All-Feature Final Refit on 10,000 Labels


In [ ]:
if RUN_ALL_FEATURES_RESEARCH and positive:

    final=joint_tree(specs[best_k]).fit(features.loc[train.id],train.label.to_numpy());final.save_model(str(folder/'model_10000.cbm'))
    template=pd.read_csv(data/'sample_submission.csv');ptest=final.predict_proba(features.loc[template.id])[:,1]
    control=pd.read_csv(control_csv);assert np.array_equal(control.id,template.id) and sha(control_csv)==control_record['sha256']
    output=template[['id']].copy();output['label']=(1-w)*control.label.to_numpy()+w*ptest
    assert len(output)==3000 and output.id.is_unique and np.isfinite(output.label).all() and output.label.between(0,1).all()
    path=folder/'submission_all_features_experimental.csv';output.to_csv(path,index=False)
    (folder/'submission_manifest.json').write_text(json.dumps(dict(path=str(path),sha256=sha(path),selection=selection,public_auc=None),indent=2))
    print('Experimental candidate:',path)
    print('Public score unknown. Original 0.98431 remains preserved.')
else:
    print('Skipped: optional experiment is disabled.')


### 10.3 Optional Final PCA and Embedding/Statistics Tree Refit


In [ ]:
if RUN_EMBEDDING_RESEARCH and e_positive:
    template=pd.read_csv(E_DATA/'sample_submission.csv')
    with threadpool_limits(limits=2):
        final_pca=PCA(n_components=32,svd_solver='randomized',random_state=42).fit(e_vectors.loc[e_train.id].to_numpy())
        train_pc=final_pca.transform(e_vectors.loc[e_train.id].to_numpy())[:,:components]
        test_pc=final_pca.transform(e_vectors.loc[template.id].to_numpy())[:,:components]
    final=e_model(spec).fit(np.hstack([e_stats.loc[e_train.id].to_numpy(),train_pc]),e_train.label.to_numpy())
    ptest=final.predict_proba(np.hstack([e_stats.loc[template.id].to_numpy(),test_pc]))[:,1]
    joblib.dump(final_pca,E_DIR/'pca_10000.joblib');final.save_model(str(E_DIR/'tree_10000.cbm'))
    (E_DIR/'inference_manifest.json').write_text(json.dumps(dict(normalization='row_l2',pca_file='pca_10000.joblib',pca_components_used=components,statistical_feature_names=e_stats.columns.tolist(),model_file='tree_10000.cbm',pretrained_embedding_cache=str(E_EMB),selection=e_selection),indent=2))
    control=pd.read_csv(e_control_file);assert e_sha(e_control_file)==e_record['sha256'] and np.array_equal(control.id,template.id)
    output=template[['id']].copy();output['label']=(1-e_weight)*control.label.to_numpy()+e_weight*ptest
    assert len(output)==3000 and output.id.is_unique and np.isfinite(output.label).all() and output.label.between(0,1).all()
    path=E_DIR/'submission_embedding_statistics_experimental.csv';output.to_csv(path,index=False)
    (E_DIR/'submission_manifest.json').write_text(json.dumps(dict(path=str(path),sha256=e_sha(path),selection=e_selection,public_auc=None),indent=2))
    print('Experimental candidate:',path)
    print('Public score unknown; exact 0.98677 incumbent is preserved.')
else:
    print('Skipped: optional stage is disabled.')


### 10.4 Verify and Export the Exact 0.98816 Submission


In [ ]:
BEST_DIR.mkdir(parents=True,exist_ok=True)
BEST_SUBMISSION_PATH=BEST_DIR/'submission_best_098816.csv'
payload=BEST_SOURCE_PATH.read_bytes()
assert hashlib.sha256(payload).hexdigest()==CONFIRMED_BEST['sha256']
template=pd.read_csv(DATA_DIR/'sample_submission.csv');submission=pd.read_csv(BEST_SOURCE_PATH)
assert submission.columns.tolist()==['id','label'] and len(submission)==3000 and submission.id.is_unique
assert np.array_equal(submission.id,template.id)
assert np.isfinite(submission.label).all() and submission.label.between(0,1).all() and submission.label.nunique()>2
if BEST_SUBMISSION_PATH.exists():
    assert BEST_SUBMISSION_PATH.read_bytes()==payload,'Existing best file differs; review before replacing.'
else:
    BEST_SUBMISSION_PATH.write_bytes(payload)
assert sha256_file(BEST_SUBMISSION_PATH)==CONFIRMED_BEST['sha256']
(BEST_DIR/'manifest.json').write_text(json.dumps({**CONFIRMED_BEST,'best_submission':str(BEST_SUBMISSION_PATH),'rows':len(submission)},indent=2))
print('BEST — confirmed public AUROC: 0.98816')
print('Submit this exact file:',BEST_SUBMISSION_PATH)
display(submission.head())
if DOWNLOAD_BEST:
    if IN_COLAB:
        from google.colab import files
        files.download(str(BEST_SUBMISSION_PATH))
    else:print('Local file:',BEST_SUBMISSION_PATH.resolve())


### 10.5 Model Locations and User-Reported Score History


In [ ]:
model_locations=dict(sparse_and_normalized=str(BASELINE_DIR/'final_models'),
    embedding=str(EMBEDDING_DIR/'embedding_lr_10000.joblib'),
    raw_statistics=str(RAW_BASELINE_DIR/'raw_lr_10000.joblib'),
    joint_statistics=str(CONFIRMED_JOINT_DIR/'joint_lr_10000.joblib'),
    structural_tree=str(TREE_SOURCE_DIR/'tree_10000.cbm'),
    pythia_statistics_tree=str(PYTHIA_MODEL_DIR/'classifier_10000.joblib'),
    all_feature_tree=str(ALL_FEATURES_DIR/'model_10000.cbm'),
    pythia_features=str(PYTHIA_FEATURE_DIR/'features.csv'),
    pythia_extraction_helpers=str(PYTHIA_FEATURE_DIR/'feature_helpers.py'),
    embedding_pca=str(EMBEDDING_JOINT_DIR/'pca_10000.joblib'),
    embedding_statistics_tree=str(EMBEDDING_JOINT_DIR/'tree_10000.cbm'),
    embedding_inference_manifest=str(EMBEDDING_JOINT_DIR/'inference_manifest.json'))
missing=[p for p in model_locations.values() if not Path(p).exists()]
if missing:print('Unavailable local model/cache paths (CSV export can still work):',missing)
(BEST_DIR/'model_locations.json').write_text(json.dumps(model_locations,indent=2))
history=pd.DataFrame([
    dict(version='Normalized statistics fusion',public_auc=.94175),
    dict(version='Add GPT-2 embeddings',public_auc=.94189),
    dict(version='Add raw statistics',public_auc=.96719),
    dict(version='Add joint statistics',public_auc=.96875),
    dict(version='30% structural tree',public_auc=.97988),
    dict(version='60% structural tree',public_auc=.98270),
    dict(version='Add 20% dual-LM tree',public_auc=.98431),
    dict(version='Add 50% all-feature tree',public_auc=.98677),
    dict(version='Add 50% embedding/statistics tree — BEST',public_auc=.98816)])
history['score_source']='user-reported Kaggle public leaderboard'
history.to_csv(BEST_DIR/'score_history.csv',index=False)
display(history);display(pd.DataFrame(model_locations.items(),columns=['component','saved_path']))


### 10.6 Environment, Pretrained Model Disclosure, and Contributions


In [ ]:
from importlib.metadata import version, PackageNotFoundError

package_versions = {}
for package in ['numpy', 'pandas', 'scikit-learn', 'torch', 'transformers', 'tokenizers', 'joblib']:
    try:
        package_versions[package] = version(package)
    except PackageNotFoundError:
        package_versions[package] = None
record = dict(python=sys.version, packages=package_versions, confirmed_best=CONFIRMED_BEST,
    pretrained_models=[dict(model="gpt2",config=str(BASELINE_DIR/"config.json"),frozen=True),
        dict(model="EleutherAI/pythia-410m",config=str(PYTHIA_FEATURE_DIR/"config.json"),license="Apache-2.0",model_card="https://huggingface.co/EleutherAI/pythia-410m",frozen=True)],
    external_inference_api=False, external_labeled_data=False,
    limitations=['Repeated CV and validation reuse', 'Public/private distribution differences',
                 'No guarantee of future leaderboard improvement'])
(BEST_DIR/'environment_and_provenance.json').write_text(json.dumps(record, indent=2))
TEAM_CONTRIBUTIONS = {}  # Fill only actual contributions before submitting the report.
SAVE_TEAM_DOCUMENTATION = False
if SAVE_TEAM_DOCUMENTATION:
    (BEST_DIR/'team_documentation.json').write_text(json.dumps(dict(
        contributions=TEAM_CONTRIBUTIONS,
        ai_assistance='Codex assisted with code organization and experimental analysis; the team reviews results.'), indent=2))
print('Best submission:', BEST_SUBMISSION_PATH)
print('Models, score history, and provenance:', BEST_DIR)
